<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_03_feature_engineering/stage_03_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **stage_03_feature_engineering**

## **Introducción**

Esta notebook corresponde al stage_03 - Technical Indicators del pipeline neural_profit y tiene como objetivo generar, evaluar y consolidar indicadores técnicos intradía para el índice MNQ, a partir de datos minuto a minuto.

Partiendo del dataset intradía ya etiquetado con objetivos de retorno, se calculan indicadores técnicos de forma independiente por jornada, evitando la mezcla de información entre días. Esto garantiza consistencia temporal y previene leakage en etapas posteriores de modelado.

El proceso incluye la evaluación cuantitativa de los indicadores mediante Information Coefficient (IC), utilizando correlación de Spearman entre cada indicador y los targets de retorno definidos para distintos horizontes. Este análisis permite medir no solo la relación promedio con el target, sino también su estabilidad a lo largo del tiempo.

Como resultado final, se generan datasets consolidados y listos para modelado, que incluyen:

- Variables OHLCV
- Targets de retorno a distintos horizontes
- Indicadores técnicos seleccionados y validados

Estos artefactos serán utilizados en las siguientes etapas del pipeline para selección de features, entrenamiento y evaluación de modelos predictivos.

## **0. Configuración del Entorno**


## 0.1. Clonado de repositorio / Acceso a Drive

In [138]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
#!git clone https://github.com/GUNAPILLCO/neural_profit.git

In [139]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 0.2. Instalación e importación de librerías


In [140]:
import sys
import re
#Instalación de librería pandas_market_calendars
#!{sys.executable} -m pip install -q pandas_market_calendars
#print("Librería instalada: pandas_market_calendars")


from functools import reduce
# Utilidades generales
from datetime import datetime, timedelta
import os
import glob
import requests
import warnings
warnings.filterwarnings('ignore')

# Manejo y procesamiento de datos

import pandas as pd
import numpy as np
from tabulate import tabulate
import matplotlib.pyplot as plt
# Calendario de mercados
#import pandas_market_calendars as mcal



from scipy.stats import spearmanr

from scipy.stats import spearmanr
from __future__ import annotations  # permite type hints modernos (Python < 3.11)
import json                         # para guardar el summary como JSON
import os                           # para leer variables de entorno
from pathlib import Path            # manejo robusto de rutas
import pandas as pd

In [141]:
!{sys.executable} -m pip install -q ta
print("Librería instalada: technical-analysis")

Librería instalada: technical-analysis


In [142]:
import ta
from ta.momentum import StochasticOscillator, ROCIndicator
from ta.volatility import BollingerBands, AverageTrueRange

## 0.4. Definición de rutas



In [143]:
#/content/drive/MyDrive/neural_profit/data/processed/mnq_intraday_labeled.parquet
DRIVE_DIR =   Path(os.environ.get("DRIVE_DIR", "/content/drive/MyDrive/neural_profit/"))

In [144]:
# ---------------------------------------------------------------------
# Configuración de rutas (DVC-friendly)
# ---------------------------------------------------------------------
# Estas rutas son RELATIVAS al repositorio.
# DVC necesita paths locales y determinísticos.
# Si mañana quiere apuntar a Drive, se hace vía DVC remote o symlink,
# NO cambiando la lógica del stage.

# PARA EL DVC:
IN_PARQUET = Path(os.environ.get("IN_PARQUET", "data/targets/mnq_intraday_targets.parquet"))
#IN_ARTIFACT = Path(os.environ.get("IN_ARTIFACT", "reports/stage_03b_target_definition_summary.json"))
OUT_PARQUET = Path(os.environ.get("OUT_PARQUET", "data/features/mnq_features_target.parquet"))
#OUT_SUMMARY = Path(os.environ.get("OUT_SUMMARY", "reports/stage_04_feature_engineering_summary.json"))

In [145]:
# PARA EL NOTEBOOK:
IN_PARQUET = DRIVE_DIR / IN_PARQUET
#IN_ARTIFACT = DRIVE_DIR / IN_ARTIFACT
OUT_PARQUET = DRIVE_DIR / OUT_PARQUET
#OUT_SUMMARY = DRIVE_DIR / OUT_SUMMARY

## 0.5. Códigos auxiliares para carga de datos y visualización


In [146]:
def load_mnq_parquet():
    os.path.exists(IN_PARQUET)
    print("Archivo encontrado en disco. Cargando dataset local...")
    mnq_parquet = pd.read_parquet(IN_PARQUET)
    return mnq_parquet

In [147]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Optional, Tuple

import pandas as pd


def mnq_dataset_info(
    df: pd.DataFrame,
    *,
    name: str = "mnq_raw",
    tz_assume_if_naive: Optional[str] = None,  # ej: "UTC" o "America/New_York"
    day_def: str = "calendar",  # "calendar" (fecha calendario) o "trading" (días con datos)
) -> Dict[str, Any]:
    """
    Resume un dataset OHLCV con DatetimeIndex (ideal para mnq_raw).

    - Si el índice es tz-naive:
        - Si tz_assume_if_naive != None, lo localiza a esa tz.
        - Si no, reporta "tz-naive" (no se puede afirmar horario UTC).
    - Devuelve dict con métricas principales (y lo imprime bonito si se desea).
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError(f"{name}: se requiere DatetimeIndex, recibido: {type(df.index)}")

    idx = df.index

    # --- timezone / UTC info ---
    tzinfo = idx.tz
    if tzinfo is None:
        tz_status = "tz-naive (sin zona horaria)"
        if tz_assume_if_naive:
            idx = idx.tz_localize(tz_assume_if_naive)
            tzinfo = idx.tz
            tz_status = f"localizado como {tzinfo}"
    else:
        tz_status = f"{tzinfo}"

    # --- rango temporal ---
    ts_min = idx.min()
    ts_max = idx.max()

    first_day = ts_min.date()
    last_day = ts_max.date()

    # --- días ---
    if day_def == "calendar":
        total_days = (pd.Timestamp(last_day) - pd.Timestamp(first_day)).days + 1
    elif day_def == "trading":
        total_days = idx.normalize().nunique()
    else:
        raise ValueError("day_def debe ser 'calendar' o 'trading'")

    # --- columnas ---
    columns = list(df.columns)

    # --- checks útiles ---
    n_rows = len(df)
    n_cols = df.shape[1]
    n_missing = int(df.isna().sum().sum())
    missing_by_col = df.isna().sum().to_dict()
    dup_index = int(idx.duplicated().sum())
    is_monotonic = bool(idx.is_monotonic_increasing)

    # Frecuencia estimada (puede fallar si hay huecos grandes)
    freq = pd.infer_freq(idx[: min(50000, len(idx))])  # muestra grande pero acotada

    # Cobertura por día (min/max de hora del día, en tz del índice)
    # (Útil para ver si es 24/7 o horario de sesión)
    tod = pd.Series(idx.time)
    # Convertimos time a minutos del día para resumen robusto
    tod_minutes = pd.Series([t.hour * 60 + t.minute for t in tod])
    typical_minute_min = int(tod_minutes.min())
    typical_minute_max = int(tod_minutes.max())

    # Rango promedio de filas por día (sólo días con datos)
    rows_per_day = df.groupby(idx.normalize()).size()
    rows_per_day_stats = {
        "days_with_data": int(rows_per_day.shape[0]),
        "rows_per_day_min": int(rows_per_day.min()),
        "rows_per_day_p50": float(rows_per_day.median()),
        "rows_per_day_max": int(rows_per_day.max()),
    }

    # Si hay tz, también mostramos rango en UTC
    if tzinfo is not None:
        ts_min_utc = ts_min.tz_convert("UTC")
        ts_max_utc = ts_max.tz_convert("UTC")
        utc_range = (str(ts_min_utc), str(ts_max_utc))
        utc_note = "El índice está tz-aware; el horario UTC es inequívoco."
    else:
        utc_range = None
        utc_note = "El índice es tz-naive; no se puede asegurar si está en UTC sin suposiciones."

    info: Dict[str, Any] = {
        "name": name,
        "shape": (n_rows, n_cols),
        "columns": columns,
        "index_type": type(df.index).__name__,
        "index_tz": tz_status,
        "utc_note": utc_note,
        "datetime_min": str(ts_min),
        "datetime_max": str(ts_max),
        "first_day": str(first_day),
        "last_day": str(last_day),
        "total_days": int(total_days),
        "day_definition": day_def,
        "utc_range_if_applicable": utc_range,
        "is_index_monotonic_increasing": is_monotonic,
        "duplicated_timestamps_in_index": dup_index,
        "inferred_freq_sample": freq,
        "missing_total_cells": n_missing,
        "missing_by_col": missing_by_col,
        "rows_per_day_stats": rows_per_day_stats,
        "time_of_day_minutes_range": {
            "min_minute_of_day": typical_minute_min,
            "max_minute_of_day": typical_minute_max,
        },
    }
    return info


def print_mnq_dataset_info(info: Dict[str, Any]) -> None:
    """Imprime el dict de mnq_dataset_info de forma ordenada."""
    print(f"Dataset: {info['name']}")
    print(f"Shape: {info['shape']}")
    print(f"Columns: {info['columns']}")
    print(f"Index: {info['index_type']} | TZ: {info['index_tz']}")
    print(f"Datetime min/max: {info['datetime_min']}  ->  {info['datetime_max']}")
    print(f"First/Last day: {info['first_day']}  ->  {info['last_day']}")
    print(f"Total days ({info['day_definition']}): {info['total_days']}")
    #print(f"Inferred freq (sample): {info['inferred_freq_sample']}")
    #print(f"Index monotonic increasing: {info['is_index_monotonic_increasing']}")
    #print(f"Duplicated timestamps in index: {info['duplicated_timestamps_in_index']}")
    #print(f"Missing total cells: {info['missing_total_cells']}")
    #print(f"Missing by col: {info['missing_by_col']}")
    #print(f"Rows/day stats: {info['rows_per_day_stats']}")
    print(f"Time-of-day range (minutes): {info['time_of_day_minutes_range']}")
    print(f"UTC note: {info['utc_note']}")
    if info["utc_range_if_applicable"] is not None:
        print(f"UTC range: {info['utc_range_if_applicable'][0]}  ->  {info['utc_range_if_applicable'][1]}")

In [148]:
mnq_intraday_targets = load_mnq_parquet()
info = mnq_dataset_info(mnq_intraday_targets, name="mnq_intraday", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info)

Archivo encontrado en disco. Cargando dataset local...
Dataset: mnq_intraday
Shape: (744013, 21)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 06:30:00-05:00  ->  2025-06-13 16:00:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total days (trading): 1303
Time-of-day range (minutes): {'min_minute_of_day': 390, 'max_minute_of_day': 960}
UTC note: El índice está tz-aware; el horario UTC es inequívoco.
UTC range: 2019-12-23 11:30:00+00:00  ->  2025-06-13 20:00:00+00:00


## 0.6. Auxiliares

In [149]:
def assign_indicator_family(indicator: str) -> str:
    """
    Asigna una familia económica a cada indicador técnico
    en función de su nombre.
    """
    name = indicator.lower()

    if "ema" in name or name.startswith("price_"):
        return "trend_price"

    if name.startswith("bb_"):
        return "volatility_extension"

    if name.startswith("roc") or name.startswith("momentum"):
        return "momentum"

    if name.startswith("rsi"):
        return "momentum_oscillator"

    if name.startswith("stoch"):
        return "momentum_oscillator"

    if name.startswith("atr"):
        return "volatility"

    if name.startswith("volume_ratio"):
        return "volume"

    if name == "macd":
        return "trend_momentum"

    return "other"

In [150]:
def select_top_by_family(
    ic_df: pd.DataFrame,
    top_n: int = 1
) -> pd.DataFrame:
    """
    Selecciona los mejores indicadores por familia
    según abs_IC_delta.
    """
    df = ic_df.copy()

    # Asignar familia
    df["family"] = df["indicator"].apply(assign_indicator_family)

    # Ordenar por fuerza de señal
    df = df.sort_values("abs_IC_delta", ascending=False)

    # Tomar top N por familia
    df_top = (
        df.groupby("family", as_index=False)
          .head(top_n)
          .reset_index(drop=True)
    )

    return df_top

#**1. Relación entre indicadores técnicos, information coefficient y los targets definidos del stage_03**

### **1.1. Indicadores técnicos**

Los indicadores técnicos se construyen a partir de la serie de precios intradía, y en particular sobre la variable de cierre (`close`). Estas transformaciones matemáticas buscan capturar propiedades dinámicas del mercado tales como tendencia, momentum, reversión, volatilidad y estructura temporal del movimiento de precios.

En el marco actual del proyecto, la variable objetivo principal (target) se define como el retorno simple intradía calculado sobre el precio de cierre, para horizontes temporales futuros discretos de 60 y 90 minutos (`ret_60` y `ret_90`). Este retorno representa la variación relativa del precio entre el instante actual y el instante futuro correspondiente al horizonte considerado.

De manera complementaria, se definen como targets secundarios de exploración los deltas en puntos hacia los mismos horizontes temporales (`delta_60` y `delta_90`). Estos targets no constituyen el objetivo principal del modelado, sino que se utilizan con fines comparativos y de análisis económico, permitiendo contrastar el comportamiento de los modelos en una escala absoluta de precio.

En consecuencia, el problema de modelado se plantea principalmente como una tarea de predicción directa de retornos simples, sin discretización en umbrales ni definición de clases económicas. El análisis y la evaluación de los modelos se realizan prioritariamente en el espacio continuo del retorno, preservando tanto su signo como su magnitud.

Esto implica que, aunque los indicadores técnicos se calculen directamente sobre el precio, su evaluación no se orienta a explicar la evolución instantánea del `close`, sino a medir su capacidad para anticipar retornos simples futuros en un horizonte temporal determinado.

El vínculo entre indicadores y targets se establece, por tanto, en términos de poder predictivo sobre la magnitud y el signo del retorno intradía futuro. El análisis posterior se centra en identificar qué indicadores y configuraciones temporales contienen información relevante para explicar y predecir retornos futuros de manera consistente, en el marco de un problema de regresión continua, utilizando los deltas en puntos como referencia secundaria de interpretación económica.


### **1.2. Information Coefficient (IC) versus targets `ret_h` y `delta_h`**

En el dataset `mnq_intraday_targets`, cada fila representa una decisión potencial en un instante $ t $ del intradía. Para ese instante, el *target* continuo `ret_h` indica el **retorno simple intradía futuro**, calculado sobre el precio de cierre, entre $ t $ y $ t+h $, para horizontes fijos (por ejemplo, $ h = 60 $ o $ 90 $ minutos). Ídem para `delta_h`

La **ventana de gestión** no redefine el *target* ni introduce una nueva variable temporal, sino que delimita el conjunto de instantes $ t $ en los cuales el sistema está habilitado a evaluar señales y generar decisiones. En este trabajo, dicha ventana se define empíricamente como el período donde se observa mayor persistencia estadística de los factores, y constituye el intervalo operativo en el que el modelo “observa” el mercado y produce estimaciones de retorno futuro.

Por su parte, la **ventana de ejecución** (o expansión) no es una entidad explícita del modelo ni del cálculo del *Information Coefficient* (IC). Esta ventana surge de manera implícita, ya que corresponde al intervalo temporal en el que se materializa el resultado futuro de las decisiones tomadas durante la ventana de gestión. Para cada instante $ t $ perteneciente a la ventana de gestión, el *target* `ret_h` refleja el comportamiento del precio en $ t+h $, que naturalmente se ubica en una franja horaria posterior.

Con el objetivo de evaluar la **robustez temporal** de los indicadores técnicos y evitar conclusiones dependientes de un único tramo horario, el análisis de IC se realiza bajo tres contextos complementarios:

a) **Jornada completa**: se consideran todos los instantes intradía válidos, con el fin de identificar factores estructurales con capacidad predictiva estable a lo largo del día.

b) **Ventana de gestación (08:00-09:00)**: se restringe el análisis al período previo al inicio del movimiento operativo principal, donde suelen formarse las condiciones iniciales del desplazamiento posterior.

c) **Ventana de ejecución o expansión (09:00-10:00)**: se analiza el período inmediatamente posterior, donde los movimientos tienden a desarrollarse con mayor intensidad y direccionalidad.

Este enfoque permite distinguir entre indicadores con valor explicativo global y aquellos cuya capacidad predictiva depende del contexto horario.

Bajo este esquema, el *Information Coefficient* (IC) se calcula correlacionando, para cada instante $ t $ perteneciente al conjunto temporal considerado:

- $ X_t $: el valor del indicador técnico, calculado exclusivamente con información disponible hasta $ t $.
- $ Y_{t,h} $: el *target* continuo `ret_h`, asociado a ese mismo instante $ t $ y a un horizonte fijo $ h $.

De este modo, el IC mide directamente la capacidad del indicador, evaluado en el momento de decisión, para anticipar la **magnitud y el signo del retorno intradía futuro**. La relación se establece fila a fila, sin agregar bloques temporales ni comparar ventanas entre sí, respetando estrictamente la causalidad temporal.

En términos operativos, un IC distinto de cero indica que ciertos estados del mercado, caracterizados por los indicadores técnicos en el instante de evaluación, están sistemáticamente asociados a retornos futuros de mayor o menor magnitud y/o dirección. Esto justifica su uso como variables explicativas en el entrenamiento del modelo predictivo.


### **1.3. Alineación entre el punto 8 (stage_03b) y el cálculo del IC (stage_04)**

La construcción de las ventanas operativas desarrollada en el punto 8 del stage_03b establece una separación conceptual fundamental entre:

- una ventana de gestación (predicción), donde se origina la información anticipatoria, y

- una ventana de expansión (ejecución), donde los movimientos alcanzan magnitud económica explotable.

Esta separación no entra en conflicto con la definición de los targets ni con el cálculo del Information Coefficient (IC); por el contrario, ambos enfoques son complementarios y coherentes, siempre que se entienda correctamente el rol temporal de cada elemento.



#### **1.3.1. Nivel estadístico (dataset y targets)**

En `mnq_intraday_labeled`, el *target* (`ret_h`) está definido **fila a fila**, para cada instante $ t $, como el **retorno simple intradía futuro** observado entre $ t $ y $ t+h $, para un horizonte fijo $ h $.

Esto implica que:

- el *target* está **anclado temporalmente al instante $ t $**,
- el horizonte $ h $ determina **cuándo se materializa el resultado futuro**,
- no existen *targets* definidos “por ventana”, sino **por cada decisión potencial asociada a un minuto específico**.

Cuando el análisis se restringe a la **ventana de gestación** (por ejemplo, 08:20-08:40), lo que se realiza es una **selección del subconjunto de instantes $ t $** que, de acuerdo con el análisis empírico previo, concentran mayor información anticipatoria relevante. Esta restricción no modifica la definición del *target*, sino únicamente el conjunto temporal sobre el cual se evalúa la relación entre indicadores y retornos futuros.

En este contexto, el *Information Coefficient* (IC) mide:

- la **relación estadística** entre el estado del mercado en el instante $ t $, capturado por los indicadores técnicos calculados con información disponible hasta ese momento, y
- el **retorno simple futuro `ret_h`** asociado a ese mismo instante $ t $, cuyo efecto se materializa naturalmente en una franja horaria posterior, correspondiente a la ventana de ejecución o expansión.

De este modo, el IC evalúa de forma directa la capacidad de los indicadores técnicos, observados en el momento de decisión, para anticipar la magnitud y el signo del movimiento futuro del precio, respetando estrictamente la causalidad temporal.


#### **1.3.2. Nivel operativo (modelo y ejecución)**

Desde el punto de vista operativo, el esquema temporal se interpreta de la siguiente manera:

- **Ventana de gestación (08:20-08:40)**  
  - Se calculan los indicadores técnicos utilizando únicamente información disponible hasta cada instante \( t \).  
  - El modelo evalúa si, desde esos instantes, es esperable un **retorno simple intradía** significativo a horizontes de 60 o 90 minutos.  
  - En esta franja reside la **capacidad predictiva**, ya que se identifican contextos de mercado con potencial direccional futuro.

- **Ventana de expansión (09:10-09:40)**  
  - Corresponde al período donde, de forma empírica, los movimientos intradía tienden a manifestarse con mayor frecuencia e intensidad.  
  - Las predicciones generadas durante la ventana de gestación habilitan (o no) la toma de decisiones operativas en esta franja.  
  - El punto de entrada puede ubicarse en cualquier minuto dentro de esta ventana, sujeto a reglas operativas adicionales y a la lógica del sistema de ejecución.

- **Horizonte de resultado**  
  - El resultado de la operación se evalúa en función del **retorno simple futuro** observado a un horizonte fijo (60 o 90 minutos) contado desde el instante de entrada.  
  - El cierre de la operación se rige por reglas operativas externas al modelo predictivo (por ejemplo, gestión de riesgo, toma de ganancias o stop), manteniendo siempre la coherencia temporal con el horizonte definido.

Bajo este esquema, el modelo no predice eventos discretos ni umbrales de puntos, sino la **magnitud y el signo del retorno futuro**, mientras que las ventanas temporales organizan el proceso de decisión y ejecución sin alterar la definición del *target*.

#### **1.3.3. Rol del IC dentro de este esquema**

El *Information Coefficient* (IC) no evalúa la ejecución operativa, sino la **calidad de la información predictiva** contenida en los indicadores técnicos bajo distintos contextos temporales.

Su función es responder a la pregunta:

¿Los indicadores técnicos calculados en el instante \( t \) contienen información útil para anticipar el retorno simple futuro `ret_h` en \( t+h \)?

Por lo tanto:

- El IC Se calcula en **tres contextos complementarios**:
  - **Jornada completa** (todos los instantes intradía válidos),
  - **Ventana de gestación** (por ejemplo, 08:00–09:00),
  - **Ventana de ejecución/expansión** (por ejemplo, 09:00–10:00).
- En todos los casos, los *targets* (`ret_h`) incorporan de forma natural el desfase temporal \( h \), ya que están definidos como retorno futuro entre \( t \) y \( t+h \).
- La coherencia temporal y la ausencia de *data leakage* quedan garantizadas por construcción: $ X_t $ se calcula con información disponible hasta \( t \), y $ Y_{t,h} $ se observa en el futuro, en \( t+h \).


### **1.4. Conclusión sintética**

La lógica operativa define **cuándo se observa información** (gestación), **cuándo se ejecuta** (expansión) y **cómo se mide el resultado** (horizonte \( h \) desde el instante de entrada).

El cálculo del IC, en el `stage_04`, cuantifica qué tan informativos son los indicadores técnicos respecto del retorno futuro `ret_h` bajo tres escenarios: **global (todo el día)** y **dependientes del contexto horario** (gestación y expansión).

Ambos enfoques describen el mismo fenómeno desde niveles distintos (operativo vs estadístico) y se mantienen alineados dentro del diseño del pipeline.

# **2. Separación de dataset IS vs OOS**

En esta etapa se realiza la separación del dataset en dos subconjuntos temporales: **in-sample (IS)** y **out-of-sample (OOS)**.  
Esta división se efectúa de manera estrictamente cronológica, respetando el orden temporal de los datos y evitando cualquier tipo de filtración de información futura.

El conjunto **in-sample (IS)** se utiliza para:
- el cálculo y selección de indicadores técnicos,
- el análisis del Information Coefficient (IC),
- la evaluación de correlaciones y redundancias entre features.

El conjunto **out-of-sample (OOS)** se reserva exclusivamente para:
- validar la estabilidad temporal de las relaciones observadas,
- comprobar que la capacidad predictiva de los indicadores no es producto del sobreajuste,
- verificar que las señales seleccionadas mantienen poder explicativo en datos no vistos.

Esta separación es un paso crítico para garantizar la validez estadística del proceso de feature engineering.  
Un indicador solo se considera apto para su uso en el modelo predictivo si demuestra un comportamiento consistente entre IS y OOS, tanto en magnitud como en signo del IC.

De este modo, la selección final de features se basa en criterios de **robustez temporal**, y no únicamente en el desempeño observado dentro del período de entrenamiento.


**Criterio propuesto (ajustable)**

- IS: desde 2019-12-23 hasta 2022-12-31
- OOS: desde 2023-01-01 hasta 2025-06-13

Este split es solo para selección y validación de features, no es el split final de modelado.

In [151]:
# Trabajamos sobre el dataset existente
df = mnq_intraday_targets.copy()

# Aseguramos tipo datetime
df["date"] = pd.to_datetime(df["date"])

# Definimos fecha de corte IS / OOS
cut_date = pd.to_datetime("2022-12-31")

# Creamos columna de split para Feature Engineering
df["split_fe"] = np.where(
    df["date"] <= cut_date,
    "IS",   # In-Sample (selección de features)
    "OOS",  # Out-Of-Sample (validación de features)
)

# Sobrescribimos el dataset con la nueva columna
mnq_intraday_targets = df



In [152]:
mnq_intraday_targets.head()

,date,open,high,low,close,volume,minute_of_day,is_premarket,is_opening,is_regular,...,is_mon,is_tue,is_wed,is_thu,is_fri,delta_60,ret_60,delta_90,ret_90,split_fe
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-23 06:30:00-05:00,2019-12-23,8727.75,8728.00,8727.75,8727.75,7,390,0,0,0,...,1,0,0,0,0,9.00,0.001031,6.00,0.000687,IS
2019-12-23 06:31:00-05:00,2019-12-23,8727.50,8727.75,8726.50,8726.50,89,391,0,0,0,...,1,0,0,0,0,9.25,0.001060,7.50,0.000859,IS
2019-12-23 06:32:00-05:00,2019-12-23,8726.50,8726.50,8725.25,8725.25,34,392,0,0,0,...,1,0,0,0,0,9.75,0.001117,8.00,0.000917,IS
2019-12-23 06:33:00-05:00,2019-12-23,8725.50,8726.25,8724.75,8726.00,53,393,0,0,0,...,1,0,0,0,0,8.50,0.000974,8.00,0.000917,IS
2019-12-23 06:34:00-05:00,2019-12-23,8726.00,8726.00,8726.00,8726.00,3,394,0,0,0,...,1,0,0,0,0,8.00,0.000917,7.75,0.000888,IS


# **3. Indicadores Técnicos**

Los indicadores técnicos calculados en cada jornada tienen como objetivo capturar dinámicas intradía relevantes del precio y el volumen, tales como momentum, sobrecompra/sobreventa, presión institucional o posibles reversiones. Cada uno aporta información complementaria sobre el comportamiento del mercado a corto plazo. En particular:

## **3.1. Indicadores técnicos individuales**

#### 1. **RSI (Relative Strength Index)**

Mide la fuerza relativa del precio en los últimos períodos (3, 5, 7, 14), oscilando entre 0 y 100.

  - Valores altos indican posibles condiciones de sobrecompra, mientras que valores bajos sugieren sobreventa.
  
  - Calculado sobre los precios de cierre intradía, el RSI es útil para identificar puntos de reversión potenciales en el corto plazo.

In [153]:
def calcular_rsi(df=mnq_intraday_targets, target='close' ):
  rsi_columns = ['rsi_14', 'rsi_7', 'rsi_5', 'rsi_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['rsi_14'] = ta.momentum.RSIIndicator(grupo[target], window=14).rsi()
        grupo['rsi_7'] = ta.momentum.RSIIndicator(grupo[target], window=7).rsi()
        grupo['rsi_5'] = ta.momentum.RSIIndicator(grupo[target], window=5).rsi()
        grupo['rsi_3'] = ta.momentum.RSIIndicator(grupo[target], window=3).rsi()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, rsi_columns

#### 2. **Momentum**

Mide la aceleración reciente del precio mediante la variación porcentual entre el precio actual y el de hace N minutos.

  - Un valor positivo indica una subida reciente, lo que podría sugerir una continuación alcista.

  - Un valor negativo señala presión bajista reciente, potencialmente anticipando una continuación a la baja.

In [154]:
def calcular_momentum(df=mnq_intraday_targets, target='close' ):
  momentum_columns = ['mom_10', 'mom_5','mom_3']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['mom_10'] = grupo[target].pct_change(10)
        grupo['mom_5'] = grupo[target].pct_change(5)
        grupo['mom_3'] = grupo[target].pct_change(3)
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)
  return df, momentum_columns

#### 3. **Relación de volumen actual vs. su promedio reciente**

Compara el volumen actual con su media móvil en distintas ventanas de tiempo: 15, 20 y 30 minutos.

  - Un valor mayor a 1 indica un volumen superior al promedio de la ventana correspondiente, lo que puede reflejar interés creciente o actividad institucional.

  - Un valor menor a 1 sugiere baja actividad o consolidación del precio.

Esta métrica permite detectar aumentos de volumen ("spikes") sin depender del volumen en crudo, y las diferentes ventanas permiten capturar variaciones en la dinámica de corto plazo con distinta sensibilidad.


In [155]:
def calcular_volumen_ratio(df=mnq_intraday_targets, target='close'):
  volume_ratio_columns = ['volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90']

  def aplicar_por_dia (grupo):
        grupo = grupo.copy()
        grupo['volume_ratio_15'] = grupo['volume'] / grupo['volume'].rolling(15).mean()
        grupo['volume_ratio_20'] = grupo['volume'] / grupo['volume'].rolling(20).mean()
        grupo['volume_ratio_30'] = grupo['volume'] / grupo['volume'].rolling(30).mean()
        grupo['volume_ratio_60'] = grupo['volume'] / grupo['volume'].rolling(60).mean()
        grupo['volume_ratio_90'] = grupo['volume'] / grupo['volume'].rolling(90).mean()
        return grupo

  df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

  return df, volume_ratio_columns

#### 4. **MACD diferencial (señal de cruce)**

Representa la diferencia entre la línea MACD y su línea de señal (una media exponencial de sí misma).

  - Un valor positivo y creciente indica momentum alcista.

  - Un valor negativo sugiere presión bajista.
  
Es ampliamente utilizado para detectar giros de tendencia y cambios en la dinámica del mercado.


In [156]:
def calcular_macd(df=mnq_intraday_targets, target='close'):
    macd_columns = ['macd']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['macd'] = ta.trend.MACD(grupo[target]).macd_diff()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, macd_columns

#### 5. **Distancia del precio actual a su EMA (15, 20 y 30 minutos)**

Mide el desvío porcentual del precio respecto a su media exponencial en diferentes ventanas, y actúa como indicador de sobreextensión o retorno a la media.

  - Si el precio está muy por encima de la EMA, puede anticipar una reversión bajista o una posible aceleración alcista.

  - Si está por debajo, podría indicar agotamiento o presión vendedora.<br>

Esta métrica se expresa como un porcentaje relativo, lo que facilita la comparación entre distintas ventanas temporales y condiciones de mercado.

Usar varias ventanas (15, 20 y 30 minutos) permite capturar diferentes horizontes de reacción del precio frente a su media móvil.


In [157]:
def calcular_ema(df=mnq_intraday_targets, target='close'):
    ema_columns = ['ema_15', 'ema_20', 'ema_30',  'ema_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['ema_15'] = grupo[target] / grupo[target].ewm(span=15).mean() - 1
        grupo['ema_20'] = grupo[target] / grupo[target].ewm(span=20).mean() - 1
        grupo['ema_30'] = grupo[target] / grupo[target].ewm(span=30).mean() - 1
        grupo['ema_60'] = grupo[target] / grupo[target].ewm(span=60).mean() - 1
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, ema_columns

#### 6. **%K Estocástico**

Mide la posición relativa del precio actual dentro del rango alto-bajo de los últimos n periodos (generalmente 14).

  - Se utiliza para identificar condiciones extremas de sobrecompra o sobreventa.

  - Un valor cercano a 100 indica que el precio está cerca del máximo reciente (potencial sobrecompra), mientras que un valor cercano a 0 indica proximidad al mínimo reciente (posible sobreventa).

Es útil para detectar momentos en los que el precio puede estar excesivamente extendido y susceptible a una reversión.


In [158]:
def calcular_stochastic(df=mnq_intraday_targets, target='close'):
    stoch_columns = ['stoch_k_14', 'stoch_k_20', 'stoch_k_30']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        stoch_14 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=14, smooth_window=3
        )
        grupo['stoch_k_14'] = stoch_14.stoch()

        stoch_20 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=20, smooth_window=3
        )
        grupo['stoch_k_20'] = stoch_20.stoch()

        stoch_30 = StochasticOscillator(
            high=grupo['high'], low=grupo['low'], close=grupo[target], window=30, smooth_window=3
        )
        grupo['stoch_k_30'] = stoch_30.stoch()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, stoch_columns


#### 7.**%B de Bollinger (Bollinger Band Percent)**

Indica la posición del precio actual en relación con las bandas de Bollinger, que están construidas alrededor de una media móvil usando desviaciones estándar.

  - Un valor de %B > 1 sugiere que el precio está por encima de la banda superior, lo que podría implicar exceso de optimismo o momentum fuerte.

  - Un valor < 0 indica que está por debajo de la banda inferior, posible señal de pánico o sobreventa extrema.

Este indicador es eficaz para identificar zonas de congestión, breakout o reversiones basadas en la volatilidad reciente.


In [159]:
from ta.volatility import BollingerBands

def calcular_bollinger(df=mnq_intraday_targets, target='close'):
    '''bollinger_columns = [
        'bb_percent_15_15', 'bb_percent_20_15', 'bb_percent_30_15',
        'bb_percent_15_20', 'bb_percent_20_20', 'bb_percent_30_20',
        'bb_percent_15_25', 'bb_percent_20_25', 'bb_percent_30_25',
    ]'''

    bollinger_columns = [
        'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15',
        'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20',
        'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 1.5
        grupo['bb_15_15'] = BollingerBands(grupo[target], window=15, window_dev=1.5).bollinger_pband()
        grupo['bb_20_15'] = BollingerBands(grupo[target], window=20, window_dev=1.5).bollinger_pband()
        grupo['bb_30_15'] = BollingerBands(grupo[target], window=30, window_dev=1.5).bollinger_pband()
        grupo['bb_60_15'] = BollingerBands(grupo[target], window=60, window_dev=1.5).bollinger_pband()

        # std: 2
        grupo['bb_15_20'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30_20'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60_20'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()

        # std: 2.5
        grupo['bb_15_25'] = BollingerBands(grupo[target], window=15, window_dev=2.5).bollinger_pband()
        grupo['bb_20_25'] = BollingerBands(grupo[target], window=20, window_dev=2.5).bollinger_pband()
        grupo['bb_30_25'] = BollingerBands(grupo[target], window=30, window_dev=2.5).bollinger_pband()
        grupo['bb_60_25'] = BollingerBands(grupo[target], window=60, window_dev=2.5).bollinger_pband()



        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns


In [160]:
def calcular_bollinger_resume(df=mnq_intraday_targets, target='close'):

    bollinger_columns = [
        'bb_15', 'bb_20', 'bb_30', 'bb_60',
    ]

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()

        # std: 2
        grupo['bb_15'] = BollingerBands(grupo[target], window=15, window_dev=2).bollinger_pband()
        grupo['bb_20'] = BollingerBands(grupo[target], window=20, window_dev=2).bollinger_pband()
        grupo['bb_30'] = BollingerBands(grupo[target], window=30, window_dev=2).bollinger_pband()
        grupo['bb_60'] = BollingerBands(grupo[target], window=60, window_dev=2).bollinger_pband()
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, bollinger_columns

#### 8. **ATR normalizado (Average True Range / precio)**

Representa la volatilidad absoluta reciente ajustada al nivel del precio.

  - El ATR mide el rango promedio de oscilación de un activo en los últimos n periodos, capturando tanto movimientos bruscos como gaps.

  - Al normalizarlo dividiéndolo por el precio, se obtiene una medida relativa, comparable entre distintos niveles de mercado.

Este indicador es útil para detectar momentos de alta o baja volatilidad intradía, que pueden influir en la confiabilidad de otras señales técnicas.


In [161]:
def calcular_atr(df=mnq_intraday_targets, target='close', windows=[5, 10, 14, 20, 30]):
    atr_columns = []

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        for w in windows:
            col_name = f'atr_norm_{w}'
            atr = AverageTrueRange(high=grupo['high'], low=grupo['low'], close=grupo[target], window=w)
            grupo[col_name] = atr.average_true_range() / grupo[target]
            atr_columns.append(col_name)
        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, list(set(atr_columns))

####  9. **ROC (Rate of Change)**

Calcula la tasa de cambio porcentual del precio con respecto a su valor n minutos atrás.

- Es un indicador de momentum que capta aceleraciones o desaceleraciones recientes del precio.

- Valores positivos indican presión alcista; negativos, presión bajista.

A diferencia del momentum tradicional, el ROC expresa el cambio de forma normalizada y en porcentaje, lo que facilita su interpretación comparativa entre distintos activos o marcos temporales.


In [162]:
def calcular_roc(df=mnq_intraday_targets, target='close'):
    roc_columns = ['roc_5', 'roc_10', 'roc_20','roc_30','roc_60']

    def aplicar_por_dia(grupo):
        grupo = grupo.copy()
        grupo['roc_5'] = ROCIndicator(close=grupo[target], window=5).roc()
        grupo['roc_10'] = ROCIndicator(close=grupo[target], window=10).roc()
        grupo['roc_20'] = ROCIndicator(close=grupo[target], window=20).roc()
        grupo['roc_30'] = ROCIndicator(close=grupo[target], window=30).roc()
        grupo['roc_60'] = ROCIndicator(close=grupo[target], window=60).roc()

        return grupo

    df = df.groupby('date', group_keys=False).apply(aplicar_por_dia)

    return df, roc_columns

## **3.2. Cálculo de indicadores técnicos**

Calculamos los indicadores técnicos

In [163]:
import os
import glob
import pandas as pd

# ============================================================
# Cargar parquet existente o calcular indicadores y guardar
# ============================================================

PROCESSED_DIR = "/content/drive/MyDrive/neural_profit/data/targets"
PARQUET_NAME = "mnq_intraday_with_indicators.parquet"
PARQUET_PATH = os.path.join(PROCESSED_DIR, PARQUET_NAME)

os.makedirs(PROCESSED_DIR, exist_ok=True)

# 1) Si existe el parquet (o alguno compatible), cargarlo
if os.path.exists(PARQUET_PATH):
    mnq_intraday_with_indicators = pd.read_parquet(PARQUET_PATH)
    print(f"[OK] Cargado: {PARQUET_PATH}")
    indicator_columns = mnq_intraday_with_indicators.columns.tolist()


    # Asegurar que el índice esté en formato datetime
    mnq_intraday_with_indicators.index = pd.to_datetime(mnq_intraday_with_indicators.index)


    columns_to_remove = [
    'date', 'minute_of_day', 'open', 'high', 'low', 'close', 'volume',
    'ret_60', 'ret_90', 'split_fe'
      ]

    indicator_columns = [
        col for col in indicator_columns if col not in columns_to_remove
    ]

else:
    # Fallback: si no existe el nombre exacto, intenta encontrar algún parquet similar
    candidates = sorted(glob.glob(os.path.join(PROCESSED_DIR, "*with_indicators*.parquet")))
    if candidates:
        mnq_intraday_with_indicators = pd.read_parquet(candidates[-1])
        print(f"[OK] Cargado (fallback): {candidates[-1]}")
    else:
        # 2) Calcular indicadores
        mnq_intraday_with_indicators = mnq_intraday_targets.copy()
        print(f"[OK] Calculando indicadores técnicos")

        mnq_intraday_with_indicators, rsi_columns = calcular_rsi(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, momentum_columns = calcular_momentum(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, volume_ratio_columns = calcular_volumen_ratio(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, macd_columns = calcular_macd(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, ema_columns = calcular_ema(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, stoch_columns = calcular_stochastic(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, bollinger_columns = calcular_bollinger(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, atr_columns = calcular_atr(mnq_intraday_with_indicators)
        mnq_intraday_with_indicators, roc_columns = calcular_roc(mnq_intraday_with_indicators)

        indicator_columns = (
          rsi_columns
          + momentum_columns
          + volume_ratio_columns
          + macd_columns
          + ema_columns
          + stoch_columns
          + bollinger_columns
          + atr_columns
          + roc_columns
          )


        # 3) Guardar parquet final
        mnq_intraday_with_indicators.to_parquet(PARQUET_PATH, index=True)
        print(f"[OK] Calculado y guardado: {PARQUET_PATH}")


[OK] Cargado: /content/drive/MyDrive/neural_profit/data/targets/mnq_intraday_with_indicators.parquet


Filtramos todos los NaNs del dataset

In [164]:
# Eliminación de filas con NaN
mnq_intraday_with_indicators = mnq_intraday_with_indicators.dropna()

In [165]:
info_mnq_indicators = mnq_dataset_info(mnq_intraday_with_indicators, name="mnq_intraday_with_indicators", tz_assume_if_naive=None, day_def="trading")
print_mnq_dataset_info(info_mnq_indicators)

Dataset: mnq_intraday_with_indicators
Shape: (510776, 64)
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'minute_of_day', 'is_premarket', 'is_opening', 'is_regular', 'is_closing', 'is_closed', 'is_mon', 'is_tue', 'is_wed', 'is_thu', 'is_fri', 'delta_60', 'ret_60', 'delta_90', 'ret_90', 'split_fe', 'rsi_14', 'rsi_7', 'rsi_5', 'rsi_3', 'mom_10', 'mom_5', 'mom_3', 'volume_ratio_15', 'volume_ratio_20', 'volume_ratio_30', 'volume_ratio_60', 'volume_ratio_90', 'macd', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30', 'bb_15_15', 'bb_20_15', 'bb_30_15', 'bb_60_15', 'bb_15_20', 'bb_20_20', 'bb_30_20', 'bb_60_20', 'bb_15_25', 'bb_20_25', 'bb_30_25', 'bb_60_25', 'atr_norm_5', 'atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'roc_5', 'roc_10', 'roc_20', 'roc_30', 'roc_60']
Index: DatetimeIndex | TZ: America/New_York
Datetime min/max: 2019-12-23 07:59:00-05:00  ->  2025-06-13 14:30:00-04:00
First/Last day: 2019-12-23  ->  2025-06-13
Total day

In [166]:
assert not mnq_intraday_with_indicators.isna().any().any(), \
    "El dataset contiene NaN"

In [167]:
# Verificación posterior
start_time_full_day = info_mnq_indicators["datetime_min"][11:16]
final_time_full_day = info_mnq_indicators["datetime_max"][11:16]

#**4. Calculo de Information Coefficient (IC)**

## **4.1. Target `ret_h`**

En el dataset `mnq_intraday_labeled` se emplea actualmente **un único tipo de variable objetivo**, correspondiente a **targets continuos de retorno**, definidos para distintos horizontes temporales. En particular, se consideran los targets `ret_h` para **h = 60** y **h = 90 minutos**.

Dado que ambos horizontes comparten la misma definición y naturaleza estadística, el cálculo y la interpretación del **Information Coefficient (IC)** se realizan bajo un **marco metodológico único, coherente y homogéneo**.

La variable `ret_h` cuantifica el **retorno futuro del precio**, expresado en términos porcentuales, entre el instante \( t \) y el instante \( t + h \). Se trata de un target **continuo**, que encapsula simultáneamente la **dirección** y la **magnitud relativa** del movimiento futuro del mercado.

Este tipo de variable objetivo constituye el caso **más directo y conceptualmente apropiado** para el análisis mediante IC, ya que permite evaluar si un indicador técnico:

- anticipa correctamente la **dirección** de los movimientos futuros, y  
- discrimina la **intensidad relativa** de dichos movimientos.

Para este análisis se adopta como métrica el **coeficiente de correlación de Spearman**, debido a que:

- no presupone relaciones lineales,  
- presenta robustez frente a valores atípicos, y  
- captura relaciones **monotónicas**, más acordes con el comportamiento de series financieras intradía.

La interpretación del IC es inmediata:

- un **IC positivo** indica que valores más elevados del indicador se asocian, en promedio, con retornos futuros mayores,  
- un **IC negativo** indica una relación inversa entre el indicador y el retorno futuro.

En virtud de lo anterior, `ret_h` (para **h = 60** y **h = 90**) se adopta como **target principal y exclusivo** para el análisis de Information Coefficient en esta etapa del pipeline.


## **4.2. Criterio metodológico — Evaluación de IC por jornada completa y ventanas horarias**

Con el objetivo de alinear la selección de indicadores técnicos con el enfoque del libro, se adopta el siguiente criterio:

Se calculan **múltiples tablas de Information Coefficient (IC)** sobre distintos recortes temporales del día, manteniendo siempre la separación **IS / OOS**:

- **Jornada completa** (mercado intradía completo)
- **Ventana de gestación 08:00–09:00**
- **Ventana  de ejecución 09:00–10:00**

**Justificación**

Este enfoque es metodológicamente sólido porque:

- Permite identificar **factores estructurales**, es decir, indicadores que mantienen IC OOS positivo a lo largo de toda la jornada.
- Permite detectar **factores dependientes del horario**, cuya capacidad predictiva se concentra en ventanas específicas.
- Evita sesgos, siempre que la **selección inicial se base en la jornada completa** y el análisis por ventanas se utilice como refinamiento posterior.

De este modo, el análisis no optimiza prematuramente por horario, sino que primero prioriza **robustez global**.

---

**Uso de los resultados**

A partir de las distintas `ic_tables`, se pueden realizar los siguientes pasos:

1. Identificar la **intersección** de indicadores con IC OOS positivo en todas las ventanas  
   → *core de factores robustos*.

2. Analizar las **diferencias entre ventanas**  
   → detección de features condicionadas por tramo horario.

3. Decidir estrategias posteriores:
   - activación o ponderación de features según el horario, o
   - uso de un único modelo con contexto temporal explícito, o
   - modelos específicos por ventana (etapa posterior).

---

**Síntesis**

> Evaluar IC en la jornada completa y en ventanas horarias permite pasar de una  
> **selección global de factores** a un **refinamiento operativo**,  
> sin romper la lógica metodológica del libro ni introducir sobreajuste.


## **4.3. Implementación de cálculo de IC**

### **4.3.1. Marcas temporales de régimen**

In [168]:
import pandas as pd

def find_regime_minute_ranges(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    regime_flags = ("is_premarket", "is_opening", "is_regular", "is_closing", "is_closed"),
) -> pd.DataFrame:
    """
    Devuelve inicio y fin (min/max minute_of_day) para cada régimen,
    usando los flags binarios del dataset.
    """
    rows = []

    for flag in regime_flags:
        if flag not in df.columns:
            raise KeyError(f"Falta la columna '{flag}' en el DataFrame.")

        m = df.loc[df[flag].astype(int) == 1, minute_col].dropna()

        if m.empty:
            rows.append({
                "regime_flag": flag,
                "start_minute_of_day": pd.NA,
                "end_minute_of_day": pd.NA,
                "n_rows": 0,
            })
        else:
            rows.append({
                "regime_flag": flag,
                "start_minute_of_day": int(m.min()),
                "end_minute_of_day": int(m.max()),
                "n_rows": int(m.shape[0]),
            })

    out = pd.DataFrame(rows)

    # orden útil
    return out.sort_values("start_minute_of_day", na_position="last").reset_index(drop=True)

def minute_to_hhmm(minute_of_day: int) -> str:
    h = minute_of_day // 60
    m = minute_of_day % 60
    return f"{h:02d}:{m:02d}"

In [169]:
regime_ranges = find_regime_minute_ranges(mnq_intraday_with_indicators)

regime_ranges["start_hhmm"] = regime_ranges["start_minute_of_day"].apply(
    lambda x: minute_to_hhmm(int(x)) if pd.notna(x) else pd.NA
)
regime_ranges["end_hhmm"] = regime_ranges["end_minute_of_day"].apply(
    lambda x: minute_to_hhmm(int(x)) if pd.notna(x) else pd.NA
)
regime_ranges

,regime_flag,start_minute_of_day,end_minute_of_day,n_rows,start_hhmm,end_hhmm
0,is_closed,479,509,40393,07:59,08:29
1,is_premarket,510,569,78180,08:30,09:29
2,is_opening,570,659,117270,09:30,10:59
3,is_regular,660,870,274933,11:00,14:30
4,is_closing,<NA>,<NA>,0,<NA>,<NA>


In [170]:
# ============================================================
# Ventanas temporales para IC (full day + ventanas por hora)
# ============================================================

# Jornada completa (reutiliza los límites generales del dataset intradía)
start_time_full_day
final_time_full_day

# Ventana de gestación
start_time_gestation_window = (regime_ranges.set_index("regime_flag").loc["is_premarket", "start_hhmm"])
final_time_gestation_window = (regime_ranges.set_index("regime_flag").loc["is_premarket", "end_hhmm"])

# Ventana de ejecución/expansión
start_time_execution_window = (regime_ranges.set_index("regime_flag").loc["is_opening", "start_hhmm"])
final_time_execution_window = (regime_ranges.set_index("regime_flag").loc["is_opening", "end_hhmm"])



### **4.3.2. Funciones para calcular IC**

In [171]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# =========================
# Utilidades base
# =========================

def spearman_ic(x: pd.Series, y: pd.Series) -> float:
    """IC Spearman entre x e y, ignorando NaNs."""
    mask = x.notna() & y.notna()
    if mask.sum() < 3:
        return np.nan
    return spearmanr(x[mask], y[mask]).correlation


def filter_time_window(df: pd.DataFrame, start: str = "07:59", end: str = "14:30") -> pd.DataFrame:
    """Filtra por ventana horaria intradía. Requiere DatetimeIndex."""
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El índice del DataFrame debe ser DatetimeIndex para usar between_time().")
    return df.between_time(start, end)


def daily_ic(df: pd.DataFrame, indicator_col: str, target_col: str, date_col: str = "date") -> pd.Series:
    """IC Spearman por día (promediable). Requiere columna date_col."""
    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")
    return df.groupby(date_col).apply(lambda g: spearman_ic(g[indicator_col], g[target_col]))


# =========================
# Tabla IC (indicadores × horizontes) con IS/OOS
# =========================

def compute_ic_table_is_oos(
    df: pd.DataFrame,
    indicator_columns: list[str],
    *,
    target_prefix: str,          # "ret" o "delta" (o "lret")
    horizons: tuple[int, ...] = (60, 90),
    window_start: str,
    window_end: str,
    use_daily_ic: bool = True,
    split_col: str = "split_fe",         # "IS" / "OOS"
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Calcula IC Spearman entre cada indicador y el target (target_prefix_h),
    separando IS y OOS según split_col.

    Targets esperados (según target_prefix):
      - ret_60, ret_90      si target_prefix="ret"
      - delta_60, delta_90  si target_prefix="delta"

    Devuelve:
      - indicator, horizon, target_prefix, target_col
      - IC_IS, IC_OOS, target_oos_minus_is
      - n_pairs_IS, n_pairs_OOS
      - abs_IC_OOS (ranking recomendado)
      - note
    """
    if target_prefix not in {"ret", "delta"}:
        raise ValueError("target_prefix debe ser 'ret', 'delta'")

    # 1) Filtrado horario intradía
    dfw = filter_time_window(df, window_start, window_end).copy()

    # 2) Validaciones de columnas estructurales
    required = {split_col, date_col}
    missing_req = [c for c in required if c not in dfw.columns]
    if missing_req:
        raise ValueError(f"Faltan columnas requeridas: {missing_req}")

    # 3) Subsets IS / OOS
    df_is = dfw[dfw[split_col] == "IS"]
    df_oos = dfw[dfw[split_col] == "OOS"]

    rows = []

    for ind in indicator_columns:
        for h in horizons:
            target_col = f"{target_prefix}_{h}"

            # Validación de columnas
            missing = [c for c in (ind, target_col) if c not in dfw.columns]
            if missing:
                rows.append({
                    "indicator": ind,
                    "horizon": h,
                    "target_prefix": target_prefix,
                    "target_col": target_col,
                    "IC_IS": np.nan,
                    "IC_OOS": np.nan,
                    "target_oos_minus_is": np.nan,
                    "n_pairs_IS": 0,
                    "n_pairs_OOS": 0,
                    "note": f"missing: {missing}",
                })
                continue

            # --- IS ---
            if use_daily_ic:
                ic_is_series = daily_ic(df_is, ind, target_col, date_col=date_col)
                ic_is = float(ic_is_series.mean())
            else:
                ic_is = float(spearman_ic(df_is[ind], df_is[target_col]))
            n_pairs_is = int((df_is[ind].notna() & df_is[target_col].notna()).sum())

            # --- OOS ---
            if use_daily_ic:
                ic_oos_series = daily_ic(df_oos, ind, target_col, date_col=date_col)
                ic_oos = float(ic_oos_series.mean())
            else:
                ic_oos = float(spearman_ic(df_oos[ind], df_oos[target_col]))
            n_pairs_oos = int((df_oos[ind].notna() & df_oos[target_col].notna()).sum())

            rows.append({
                "indicator": ind,
                "horizon": h,
                "target_prefix": target_prefix,
                "target_col": target_col,
                "IC_IS": ic_is,
                "IC_OOS": ic_oos,
                "target_oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_is) and pd.notna(ic_oos)) else np.nan,
                "n_pairs_IS": n_pairs_is,
                "n_pairs_OOS": n_pairs_oos,
                "note": "",
            })

    out = pd.DataFrame(rows)

    # Ranking recomendado: por |IC_OOS| (lo que generaliza)
    out["abs_IC_OOS"] = out["IC_OOS"].abs()
    out = (
        out.sort_values(["target_prefix", "horizon", "abs_IC_OOS"], ascending=[True, True, False])
           .reset_index(drop=True)
    )

    return out

### **4.3.3. Función para calculo de IC tables**

In [172]:
import os
import json
import pandas as pd

# ============================================================
# Cache de IC tables (Google Drive)
# ============================================================

CACHE_DIR_RET = "/content/drive/MyDrive/neural_profit/data/targets/ic_tables_return"
os.makedirs(CACHE_DIR_RET, exist_ok=True)

CACHE_DIR_DELTA = "/content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta"
os.makedirs(CACHE_DIR_DELTA, exist_ok=True)

# Si en algún momento agrega lret:
#CACHE_DIR_LRET = "/content/drive/MyDrive/neural_profit/data/processed/ic_tables_lret"
#os.makedirs(CACHE_DIR_LRET, exist_ok=True)


def _select_cache_dir(target_prefix: str) -> str:
    """
    Devuelve el directorio de cache según el target.
    """
    if target_prefix == "ret":
        return CACHE_DIR_RET
    if target_prefix == "delta":
        return CACHE_DIR_DELTA
    #if target_prefix == "lret":
    #    return CACHE_DIR_LRET
    raise ValueError("target_prefix debe ser 'ret', 'delta'") # o 'lret'")


def _ic_cache_paths(cache_dir: str, name: str) -> dict:
    """
    Devuelve paths de cache para una ic_table:
      - parquet: datos
      - json: metadatos (parámetros relevantes)
    """
    return {
        "data": os.path.join(cache_dir, f"{name}.parquet"),
        "meta": os.path.join(cache_dir, f"{name}.meta.json"),
    }


def load_or_compute_ic_table(
    *,
    name: str,
    df: pd.DataFrame,
    indicator_columns: list[str],
    target_prefix: str,              # "ret" o "delta" (o "lret")
    horizons: tuple[int, ...],
    window_start: str,
    window_end: str,
    use_daily_ic: bool,
    split_col: str,
    date_col: str,
    force_recompute: bool = False,
) -> pd.DataFrame:
    """
    Carga una ic_table desde cache si existe; si no existe, la calcula y la guarda.

    Parámetros clave:
      - name: identificador del artefacto en cache (parquet/json)
      - df: DataFrame con indicadores + targets + columnas auxiliares
      - indicator_columns: lista de features a evaluar
      - target_prefix: define el target ("ret" o "delta" o "lret")
      - horizons: horizontes a evaluar (p.ej. (60, 90))
      - window_start/window_end: ventana intradía
      - use_daily_ic: IC por día (promediable) o IC global
      - split_col: columna de split (IS/OOS)
      - date_col: columna de fecha (para daily IC)
      - force_recompute: si True, ignora cache y recalcula
    """
    cache_dir = _select_cache_dir(target_prefix)
    paths = _ic_cache_paths(cache_dir, name)

    # 1) Si existe cache y no forzamos recálculo -> cargar
    if (not force_recompute) and os.path.exists(paths["data"]):
        ic_table = pd.read_parquet(paths["data"])
        print(f"[cache] Loaded: {paths['data']}")
        return ic_table

    # 2) Calcular ic_table para el target elegido
    ic_table = compute_ic_table_is_oos(
        df=df,
        indicator_columns=indicator_columns,
        target_prefix=target_prefix,
        horizons=horizons,
        window_start=window_start,
        window_end=window_end,
        use_daily_ic=use_daily_ic,
        split_col=split_col,
        date_col=date_col,
    )

    # 3) Guardar datos
    ic_table.to_parquet(paths["data"], index=False)

    # 4) Guardar metadatos mínimos
    meta = {
        "name": name,
        "target_prefix": target_prefix,
        "horizons": list(horizons),
        "window_start": window_start,
        "window_end": window_end,
        "use_daily_ic": use_daily_ic,
        "split_col": split_col,
        "date_col": date_col,
        "n_indicators": len(indicator_columns),
        "cache_dir": cache_dir,
    }
    with open(paths["meta"], "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)

    print(f"[cache] Computed & saved: {paths['data']}")
    return ic_table


### **4.3.4. Aplicación de cálculos**

In [173]:
# ============================================================
# IC tables IS vs OOS por ventana (con cache en Drive)
# ============================================================

# ============================================================
# FULL DAY
# ============================================================

ic_table_full_day_ret = load_or_compute_ic_table(
    name="ic_table_full_day_ret",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_full_day_delta = load_or_compute_ic_table(
    name="ic_table_full_day_delta",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)


[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_full_day_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_full_day_delta.parquet


In [174]:
# ============================================================
# GESTATION
# ============================================================

ic_table_gestation_ret = load_or_compute_ic_table(
    name="ic_table_gestation_ret",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_gestation_delta = load_or_compute_ic_table(
    name="ic_table_gestation_delta",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_gestation_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_gestation_delta.parquet


In [175]:
# ============================================================
# EXECUTION
# ============================================================

ic_table_execution_ret = load_or_compute_ic_table(
    name="ic_table_execution_ret",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_execution_delta = load_or_compute_ic_table(
    name="ic_table_execution_delta",
    df=mnq_intraday_with_indicators,
    indicator_columns=indicator_columns,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_execution_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_execution_delta.parquet


### **4.3.3. Resultado**

In [176]:
ic_table_full_day_ret.head(10)

,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,ret,ret_60,-0.193578,-0.189173,0.004405,281064,229712,,0.189173
1,roc_60,60,ret,ret_60,-0.187156,-0.177858,0.009298,281064,229712,,0.177858
2,bb_60_15,60,ret,ret_60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
3,bb_60_20,60,ret,ret_60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
4,bb_60_25,60,ret,ret_60,-0.152318,-0.150149,0.002168,281064,229712,,0.150149
5,ema_30,60,ret,ret_60,-0.141073,-0.137669,0.003404,281064,229712,,0.137669
6,roc_30,60,ret,ret_60,-0.135632,-0.135081,0.000551,281064,229712,,0.135081
7,rsi_14,60,ret,ret_60,-0.135146,-0.134811,0.000335,281064,229712,,0.134811
8,stoch_k_30,60,ret,ret_60,-0.122581,-0.127539,-0.004957,281064,229712,,0.127539
9,bb_30_15,60,ret,ret_60,-0.113618,-0.117318,-0.003699,281064,229712,,0.117318


In [177]:
ic_table_gestation_ret.head(10)

,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,ret,ret_60,-0.247058,-0.293809,-0.046751,43020,35160,,0.293809
1,roc_60,60,ret,ret_60,-0.221354,-0.287290,-0.065936,43020,35160,,0.287290
2,ema_30,60,ret,ret_60,-0.219531,-0.250970,-0.031438,43020,35160,,0.250970
3,rsi_14,60,ret,ret_60,-0.215600,-0.246458,-0.030858,43020,35160,,0.246458
4,roc_30,60,ret,ret_60,-0.189454,-0.241835,-0.052381,43020,35160,,0.241835
5,stoch_k_30,60,ret,ret_60,-0.194650,-0.241451,-0.046801,43020,35160,,0.241451
6,bb_60_15,60,ret,ret_60,-0.205973,-0.240867,-0.034894,43020,35160,,0.240867
7,bb_60_20,60,ret,ret_60,-0.205973,-0.240867,-0.034894,43020,35160,,0.240867
8,bb_60_25,60,ret,ret_60,-0.205973,-0.240867,-0.034894,43020,35160,,0.240867
9,bb_30_15,60,ret,ret_60,-0.187165,-0.227805,-0.040639,43020,35160,,0.227805


## **4.4. Separación por horizonte**

In [178]:
# ============================================================
# Separación por horizonte – jornada completa
# ============================================================
ic_60_full_day_ret = ic_table_full_day_ret[ic_table_full_day_ret["horizon"] == 60].copy()
ic_90_full_day_ret = ic_table_full_day_ret[ic_table_full_day_ret["horizon"] == 90].copy()

ic_60_full_day_delta = ic_table_full_day_delta[ic_table_full_day_delta["horizon"] == 60].copy()
ic_90_full_day_delta = ic_table_full_day_delta[ic_table_full_day_delta["horizon"] == 90].copy()

# ============================================================
# Separación por horizonte – ventana de gestación
# ============================================================
ic_60_gestation_ret = ic_table_gestation_ret[ic_table_gestation_ret["horizon"] == 60].copy()
ic_90_gestation_ret = ic_table_gestation_ret[ic_table_gestation_ret["horizon"] == 90].copy()

ic_60_gestation_delta = ic_table_gestation_delta[ic_table_gestation_delta["horizon"] == 60].copy()
ic_90_gestation_delta = ic_table_gestation_delta[ic_table_gestation_delta["horizon"] == 90].copy()


# ============================================================
# Separación por horizonte – ventana de ejecución / expansión
# ============================================================
ic_60_execution_ret = ic_table_execution_ret[ic_table_execution_ret["horizon"] == 60].copy()
ic_90_execution_ret = ic_table_execution_ret[ic_table_execution_ret["horizon"] == 90].copy()

ic_60_execution_delta = ic_table_execution_delta[ic_table_execution_delta["horizon"] == 60].copy()
ic_90_execution_delta = ic_table_execution_delta[ic_table_execution_delta["horizon"] == 90].copy()



## **4.5. Primer filtro: fuerza miníma de señal**

Para el análisis intradía se adopta el siguiente criterio empírico de interpretación del Information Coefficient (IC):

- $|IC| < 0.02 $ → ruido
- $ 0.02 ≤ |IC| < 0.05 $ → débil
- $ 0.05 ≤ |IC| < 0.10 $ → moderado
- $ |IC| ≥ 0.10 $ → fuerte

Dado que el objetivo de este stage es identificar señales con capacidad predictiva real, se descartan aquellas cuyo |IC| se encuentra por debajo del umbral de relevancia. En consecuencia, se conservan únicamente los indicadores que presentan una señal fuerte, definida como:

$$|IC_Δ|≥0.10$$

Este primer filtro elimina indicadores dominados por ruido y reduce el espacio de features a un conjunto con relación estadísticamente significativa respecto a la magnitud del movimiento futuro.

In [179]:
# ============================================================
# Umbral mínimo de relevancia estadística (|IC_OOS|)
# ============================================================

IC_TH = 0.10  # umbral de señal fuerte


# ============================================================
# Utilidad genérica
# ============================================================

def filter_relevant_ic(df: pd.DataFrame, ic_th: float = IC_TH) -> pd.DataFrame:
    """
    Filtra indicadores con señal estadísticamente relevante
    según |IC_OOS| >= ic_th.
    """
    return df[df["abs_IC_OOS"] >= ic_th].copy()


# ============================================================
# Diccionario con todas las tablas IC
# ============================================================

ic_tables = {
    # Full day
    "full_day_ret_60": ic_60_full_day_ret,
    "full_day_ret_90": ic_90_full_day_ret,
    "full_day_delta_60": ic_60_full_day_delta,
    "full_day_delta_90": ic_90_full_day_delta,

    # Gestation
    "gestation_ret_60": ic_60_gestation_ret,
    "gestation_ret_90": ic_90_gestation_ret,
    "gestation_delta_60": ic_60_gestation_delta,
    "gestation_delta_90": ic_90_gestation_delta,

    # Execution / expansion
    "execution_ret_60": ic_60_execution_ret,
    "execution_ret_90": ic_90_execution_ret,
    "execution_delta_60": ic_60_execution_delta,
    "execution_delta_90": ic_90_execution_delta,
}


# ============================================================
# Filtrado automático de todas las tablas
# ============================================================

ic_relevant = {
    name: filter_relevant_ic(df)
    for name, df in ic_tables.items()
}


# ============================================================
# Ejemplo de acceso
# ============================================================
# ic_relevant["full_day_delta_60"]
# ic_relevant["gestation_ret_90"]



In [180]:
# ============================================================
# Helper: extraer lista ordenada y única de indicadores
# ============================================================

from typing import List
import pandas as pd


def extract_indicator_list(df: pd.DataFrame) -> List[str]:
    """
    Extrae una lista única, ordenada alfabéticamente, de la columna 'indicator'.
    """
    return (
        df["indicator"]
        .drop_duplicates()
        .sort_values()
        .tolist()
    )


# ============================================================
# Construcción automática de listas de indicadores relevantes
# ============================================================

ti_ic_relevant_lists = {
    name: extract_indicator_list(df)
    for name, df in ic_relevant.items()
}


# ============================================================
# Ejemplos de acceso
# ============================================================
# ti_ic_relevant_lists["full_day_delta_60"]
# ti_ic_relevant_lists["gestation_ret_90"]
# ti_ic_relevant_lists["execution_delta_60"]



### **4.5.1. Análisis de Indicadores Técnicos Relevantes**

#### **Targets: Delta y Return | Horizontes: 60 / 90 | Ventanas: Full Day, Gestation, Execution**


In [181]:
# ============================================================
# Impresión ordenada de listas de indicadores relevantes
# ============================================================

def print_relevant_indicators(indicator_dict: dict[str, list[str]]) -> None:
    """
    Imprime de forma ordenada las listas de indicadores relevantes.
    """
    for name in sorted(indicator_dict.keys()):
        print(f"{name}: {indicator_dict[name]}")


# ============================================================
# Ejecución
# ============================================================

print_relevant_indicators(ti_ic_relevant_lists)

execution_delta_60: ['atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'atr_norm_5', 'bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'macd', 'mom_10', 'mom_3', 'mom_5', 'roc_10', 'roc_20', 'roc_30', 'roc_5', 'roc_60', 'rsi_14', 'rsi_3', 'rsi_5', 'rsi_7', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30']
execution_delta_90: ['atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'atr_norm_5', 'bb_15_15', 'bb_15_20', 'bb_15_25', 'bb_20_15', 'bb_20_20', 'bb_20_25', 'bb_30_15', 'bb_30_20', 'bb_30_25', 'bb_60_15', 'bb_60_20', 'bb_60_25', 'ema_15', 'ema_20', 'ema_30', 'ema_60', 'macd', 'mom_10', 'mom_3', 'mom_5', 'roc_10', 'roc_20', 'roc_30', 'roc_5', 'roc_60', 'rsi_14', 'rsi_3', 'rsi_5', 'rsi_7', 'stoch_k_14', 'stoch_k_20', 'stoch_k_30']
execution_ret_60: ['atr_norm_10', 'atr_norm_14', 'atr_norm_20', 'atr_norm_30', 'atr_norm_5', 'bb_15_15', 'bb_15_20

**1. Observación estructural clave**

  Para cada combinación de **ventana temporal** y **horizonte**, los conjuntos de indicadores técnicos relevantes (según |IC_OOS| ≥ umbral) para los targets:

  - `delta`
  - `return`

  son **idénticos o prácticamente idénticos**.

  Ejemplos representativos:
  - `execution_delta_60` ≡ `execution_ret_60`
  - `gestation_delta_60` ≡ `gestation_ret_60`
  - `full_day_delta_60` ≡ `full_day_ret_60`
  - El mismo patrón se observa en **H = 90**

  **Conclusión directa:**  
  La selección de indicadores basada en IC **no discrimina entre delta y return**.  
  La diferencia entre ambos targets **no reside en qué indicadores son relevantes**, sino en **cómo esos indicadores se traducen en capacidad predictiva al entrenar modelos**.

---

**2. Las diferencias reales aparecen por ventana, no por target**

- **Ventana de Execution**

  Es la ventana **más rica en señal estadística**. Aparecen de forma consistente:

  - Volatilidad:
    - `atr_norm_*`
  - Tendencia y velocidad:
    - `ema_*`
    - `macd`
    - `mom_*`
    - `roc_*`
  - Osciladores:
    - `rsi_*`
    - `stoch_k_*`
  - Bandas:
    - `bb_*`

  **Interpretación:**  
  Corresponde a una fase de mercado donde **dominan la volatilidad y la velocidad del precio**, coherente con una etapa activa de descubrimiento de precios.

---

- **Ventana de Gestation**

  Conjunto amplio, pero **menos orientado a volatilidad explícita**. Predominan:

  - `bb_*`
  - `ema_*`
  - `macd`
  - `mom_*`, `roc_*` (en escalas suaves)
  - `rsi_*`, `stoch_k_*`

  **Interpretación:**  
  Compatible con una fase de **acumulación o preparación del movimiento**, donde el precio aún no entra en expansión plena.

---

- **Ventana Full Day**

  - **H = 60**:
    - Conjunto más reducido
    - Prácticamente no aparecen ATR ni momentum corto
  - **H = 90**:
    - El set vuelve a ampliarse
    - Reaparecen indicadores de volatilidad y momentum

  **Interpretación:**  
  A escala diaria la señal se **diluye**, y solo emerge de forma clara cuando el horizonte es suficientemente largo.

---

**3. Estabilidad temporal (hallazgo positivo)**

  Para cada ventana:

  - Los indicadores relevantes en **H = 60** y **H = 90** son **muy similares**
  - No se observa rotación caótica del conjunto

  **Implicación:**  
  La relevancia detectada es **estructural y no espuria**.  
  Los indicadores capturan **propiedades persistentes del mercado**.

---

**4. Implicación para el debate _Return vs Delta_**

- El **IC indica**:
  > Los mismos indicadores están correlacionados con ambos targets.

- Los **modelos predictivos muestran**:
  > El delta presenta mayor estabilidad y capacidad de aprendizaje.

No hay contradicción metodológica:

- El IC evalúa **relación estadística marginal**
- El entrenamiento de modelos requiere:
  - continuidad temporal
  - menor ruido
  - un target compatible con la dinámica aprendible

**Lectura clave:**  
Returns y deltas **comparten los mismos drivers**, pero:
- el return tiende a destruir señal al normalizar
- el delta preserva magnitud y estructura, facilitando el aprendizaje

---

**5. Conclusiones técnicas**

1. **La ingeniería de features es consistente y correcta**  
   Los mismos indicadores emergen de forma estable en todas las configuraciones.

2. **La selección de variables no es el cuello de botella**  
   La diferencia de desempeño proviene de la **formulación del target**, no de los features.

3. **No se descarta ningún target**  
   Delta y return deben mantenerse como objetivos válidos, evaluados bajo el mismo set de indicadores.

4. **La ventana de execution es la más informativa**, independientemente del target.

---

**6. Decisión metodológica adoptada**

- **Targets**:  
  - Se mantienen **delta** y **return**
- **Horizontes**:  
  - 60 y 90 minutos
- **Ventanas**:  
  - Full Day, Gestation y Execution
- **Indicadores técnicos**:  
  - Se utilizará el **conjunto común validado por IC**, sin ampliaciones ad hoc

Este enfoque permite un pipeline **coherente, defendible y comparable**, donde las diferencias de desempeño entre targets pueden atribuirse al target en sí y no a la selección de variables.


# **5. Correlación**

## **5.1. Correlación de indicadores técnicos**

Una vez identificado el conjunto de indicadores técnicos relevantes a partir del análisis de Information Coefficient (IC), se procede a estudiar la estructura de dependencia existente entre ellos.

El objetivo de este análisis es cuantificar el grado de colinealidad y redundancia informativa dentro del conjunto seleccionado, así como comprender cómo se relacionan entre sí las distintas familias de indicadores técnicos (tendencia, momentum, magnitud y volatilidad).

Para ello, se calcula la matriz de correlación de Spearman promedio por día, restringida al mismo contexto temporal utilizado en el análisis de IC. En particular, el estudio se realiza de manera diferenciada para la jornada completa y para las ventanas horarias de gestación (08:00–09:00) y de ejecución/expansión (09:00–10:00). Este enfoque permite capturar relaciones monótonas, robustas frente a outliers, y evaluar la estabilidad de dichas relaciones bajo distintos regímenes intradía.

El uso de correlaciones promedio por jornada preserva la consistencia intradía del comportamiento del mercado y evita que episodios aislados dominen la estimación de dependencia entre indicadores.

Este análisis constituye un paso clave previo a la resolución de clusters de redundancia y a la selección final de features, asegurando que el conjunto de variables utilizado en los modelos predictivos sea informativamente complementario y estadísticamente robusto.


### **5.1.1. Implementación para calculo de Correlación**

In [182]:
import numpy as np
import pandas as pd

def daily_spearman_corr_matrix(
    df: pd.DataFrame,
    indicator_columns: list[str],
    window_start: str,
    window_end: str,
    date_col: str = "date",
) -> pd.DataFrame:
    """
    Matriz Spearman PROMEDIO por día, restringida a la ventana horaria.

    Requisitos:
      - df.index: DatetimeIndex
      - df contiene columna date_col
      - indicator_columns existen en df
    """
    # 0) Validaciones rápidas
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("df.index debe ser DatetimeIndex para usar between_time().")

    if not indicator_columns:
        raise ValueError("indicator_columns está vacío. No hay indicadores para correlacionar.")

    if date_col not in df.columns:
        raise ValueError(f"Falta la columna '{date_col}' en df.")

    # 1) Filtrar ventana horaria
    df_win = df.between_time(window_start, window_end).copy()

    # 2) Validar columnas
    missing = [c for c in indicator_columns if c not in df_win.columns]
    if missing:
        raise ValueError(f"Faltan columnas en df: {missing}")

    # 3) Correlación Spearman por día (matriz)
    daily_corr = []
    for _, g in df_win.groupby(date_col):
        X = g[indicator_columns]
        corr = X.corr(method="spearman")

        # Guardar solo matrices con algo útil
        if corr.notna().values.any():
            daily_corr.append(corr)

    if not daily_corr:
        raise ValueError("No se pudieron calcular correlaciones (ventana vacía, NaNs o sin datos por día).")

    # 4) Promedio (alineado por índices/columnas)
    corr_mean = sum(daily_corr) / len(daily_corr)
    return corr_mean


def top_abs_correlations(
    corr: pd.DataFrame,
    top_n: int = 15,
    min_abs_rho: float = 0.0,
) -> pd.DataFrame:
    """
    Ranking de pares con mayor |rho| (sin duplicados i-j y sin diagonal).
    """
    c = corr.copy()

    # Enmascarar diagonal y triángulo inferior (evita duplicados)
    mask = np.tril(np.ones(c.shape, dtype=bool))
    c = c.mask(mask)

    pairs = (
        c.stack()
         .rename("rho")
         .reset_index()
         .rename(columns={"level_0": "indicator_1", "level_1": "indicator_2"})
    )

    pairs["abs_rho"] = pairs["rho"].abs()
    pairs = pairs[pairs["abs_rho"] >= min_abs_rho]

    return pairs.sort_values("abs_rho", ascending=False).head(top_n).reset_index(drop=True)



### **5.1.2. Aplicación de cálculo de Correlación**

In [183]:
# ============================================================
# Configuración de ventanas (alineado con full_day / gestation / execution)
# ============================================================

WINDOWS = {
    "full_day":  (start_time_full_day, final_time_full_day),
    "gestation": (start_time_gestation_window, final_time_gestation_window),
    "execution": (start_time_execution_window, final_time_execution_window),
}


# ============================================================
# Definición de combinaciones target / ventana / horizonte
# ============================================================

TARGETS   = ["delta", "ret"]
HORIZONS  = [60, 90]


# ============================================================
# Construcción automática del mapa de indicadores
# Usa: ti_ic_relevant_lists
# Clave: (target, window, horizon)
# ============================================================

INDICATORS = {
    (target, window, horizon): ti_ic_relevant_lists[f"{window}_{target}_{horizon}"]
    for target in TARGETS
    for window in WINDOWS.keys()
    for horizon in HORIZONS
}


# ============================================================
# Cálculo de matrices de correlación Spearman promedio por día
# ============================================================

corr_mean = {}

for (target, window_name, horizon), indicators in INDICATORS.items():
    w_start, w_end = WINDOWS[window_name]

    # Evitar errores si no hay indicadores relevantes
    if not indicators:
        print(
            f"[warn] Lista vacía: target={target}, "
            f"window={window_name}, horizon={horizon}. Se omite."
        )
        continue

    corr_mean[(target, window_name, horizon)] = daily_spearman_corr_matrix(
        df=mnq_intraday_with_indicators,
        indicator_columns=indicators,
        window_start=w_start,
        window_end=w_end,
        date_col="date",
    )


# ============================================================
# Acceso a resultados (naming explícito y consistente)
# ============================================================

corr_mean_delta_60_full_day  = corr_mean[("delta", "full_day", 60)]
corr_mean_delta_90_full_day  = corr_mean[("delta", "full_day", 90)]

corr_mean_delta_60_gestation = corr_mean[("delta", "gestation", 60)]
corr_mean_delta_90_gestation = corr_mean[("delta", "gestation", 90)]

corr_mean_delta_60_execution = corr_mean[("delta", "execution", 60)]
corr_mean_delta_90_execution = corr_mean[("delta", "execution", 90)]


corr_mean_ret_60_full_day    = corr_mean[("ret", "full_day", 60)]
corr_mean_ret_90_full_day    = corr_mean[("ret", "full_day", 90)]

corr_mean_ret_60_gestation   = corr_mean[("ret", "gestation", 60)]
corr_mean_ret_90_gestation   = corr_mean[("ret", "gestation", 90)]

corr_mean_ret_60_execution   = corr_mean[("ret", "execution", 60)]
corr_mean_ret_90_execution   = corr_mean[("ret", "execution", 90)]


In [184]:
# ============================================================
# Visualización ordenada de matrices de correlación promedio
# ============================================================

from IPython.display import display


def display_corr_matrices(corr_dict: dict) -> None:
    """
    Muestra de forma ordenada las matrices de correlación
    por target, horizonte y ventana.
    """
    for target in ["delta", "ret"]:
        for horizon in [60, 90]:
            for window in ["full_day", "gestation", "execution"]:
                key = (target, window, horizon)

                if key not in corr_dict:
                    print(
                        f"\n[warn] No disponible: "
                        f"target={target}, horizon={horizon}, window={window}\n"
                    )
                    continue

                print(f"\nCorrelation mean | target={target} | H={horizon} | window={window}\n")
                display(corr_dict[key])





In [185]:
# ============================================================
# Ejecución
# ============================================================

display_corr_matrices(corr_mean)


Correlation mean | target=delta | H=60 | window=full_day



,bb_30_15,bb_30_20,bb_30_25,bb_60_15,bb_60_20,bb_60_25,ema_15,ema_20,ema_30,ema_60,roc_20,roc_30,roc_60,rsi_14,stoch_k_20,stoch_k_30
bb_30_15,1.000000,1.000000,1.000000,0.873520,0.873520,0.873520,0.911477,0.935407,0.928301,0.818683,0.850805,0.743160,0.489298,0.956985,0.926561,0.950160
bb_30_20,1.000000,1.000000,1.000000,0.873520,0.873520,0.873520,0.911477,0.935407,0.928301,0.818683,0.850805,0.743160,0.489298,0.956985,0.926561,0.950160
bb_30_25,1.000000,1.000000,1.000000,0.873520,0.873520,0.873520,0.911477,0.935407,0.928301,0.818683,0.850805,0.743160,0.489298,0.956985,0.926561,0.950160
bb_60_15,0.873520,0.873520,0.873520,1.000000,1.000000,1.000000,0.784890,0.845587,0.908899,0.923402,0.812576,0.849132,0.719773,0.917776,0.775986,0.869967
bb_60_20,0.873520,0.873520,0.873520,1.000000,1.000000,1.000000,0.784890,0.845587,0.908899,0.923402,0.812576,0.849132,0.719773,0.917776,0.775986,0.869967
bb_60_25,0.873520,0.873520,0.873520,1.000000,1.000000,1.000000,0.784890,0.845587,0.908899,0.923402,0.812576,0.849132,0.719773,0.917776,0.775986,0.869967
ema_15,0.911477,0.911477,0.911477,0.784890,0.784890,0.784890,1.000000,0.986533,0.930109,0.778907,0.771658,0.651913,0.451493,0.924837,0.919756,0.874030
ema_20,0.935407,0.935407,0.935407,0.845587,0.845587,0.845587,0.986533,1.000000,0.974482,0.849788,0.835298,0.736701,0.524176,0.958095,0.922734,0.909710
ema_30,0.928301,0.928301,0.928301,0.908899,0.908899,0.908899,0.930109,0.974482,1.000000,0.934315,0.875616,0.831518,0.639579,0.968689,0.883022,0.921139
ema_60,0.818683,0.818683,0.818683,0.923402,0.923402,0.923402,0.778907,0.849788,0.934315,1.000000,0.814760,0.866130,0.813016,0.892329,0.739792,0.833721



Correlation mean | target=delta | H=60 | window=gestation



,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,bb_20_25,bb_30_15,bb_30_20,bb_30_25,bb_60_15,...,roc_30,roc_5,roc_60,rsi_14,rsi_3,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30
bb_15_15,1.000000,1.000000,1.000000,0.956889,0.956889,0.956889,0.859749,0.859749,0.859749,0.709812,...,0.423440,0.813880,0.348695,0.862582,0.864447,0.941731,0.947120,0.930236,0.873873,0.787426
bb_15_20,1.000000,1.000000,1.000000,0.956889,0.956889,0.956889,0.859749,0.859749,0.859749,0.709812,...,0.423440,0.813880,0.348695,0.862582,0.864447,0.941731,0.947120,0.930236,0.873873,0.787426
bb_15_25,1.000000,1.000000,1.000000,0.956889,0.956889,0.956889,0.859749,0.859749,0.859749,0.709812,...,0.423440,0.813880,0.348695,0.862582,0.864447,0.941731,0.947120,0.930236,0.873873,0.787426
bb_20_15,0.956889,0.956889,0.956889,1.000000,1.000000,1.000000,0.930024,0.930024,0.930024,0.777850,...,0.487157,0.762107,0.379530,0.908635,0.801862,0.908519,0.943312,0.908211,0.919703,0.843163
bb_20_20,0.956889,0.956889,0.956889,1.000000,1.000000,1.000000,0.930024,0.930024,0.930024,0.777850,...,0.487157,0.762107,0.379530,0.908635,0.801862,0.908519,0.943312,0.908211,0.919703,0.843163
bb_20_25,0.956889,0.956889,0.956889,1.000000,1.000000,1.000000,0.930024,0.930024,0.930024,0.777850,...,0.487157,0.762107,0.379530,0.908635,0.801862,0.908519,0.943312,0.908211,0.919703,0.843163
bb_30_15,0.859749,0.859749,0.859749,0.930024,0.930024,0.930024,1.000000,1.000000,1.000000,0.871037,...,0.614114,0.679809,0.418583,0.941572,0.708204,0.835082,0.899470,0.833179,0.892433,0.905489
bb_30_20,0.859749,0.859749,0.859749,0.930024,0.930024,0.930024,1.000000,1.000000,1.000000,0.871037,...,0.614114,0.679809,0.418583,0.941572,0.708204,0.835082,0.899470,0.833179,0.892433,0.905489
bb_30_25,0.859749,0.859749,0.859749,0.930024,0.930024,0.930024,1.000000,1.000000,1.000000,0.871037,...,0.614114,0.679809,0.418583,0.941572,0.708204,0.835082,0.899470,0.833179,0.892433,0.905489
bb_60_15,0.709812,0.709812,0.709812,0.777850,0.777850,0.777850,0.871037,0.871037,0.871037,1.000000,...,0.744836,0.560734,0.510649,0.904888,0.581470,0.706366,0.785094,0.701022,0.773832,0.841329



Correlation mean | target=delta | H=60 | window=execution



...

,atr_norm_10,atr_norm_14,atr_norm_20,atr_norm_30,atr_norm_5,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,...,roc_10,roc_20,roc_30,roc_60,rsi_14,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30
atr_norm_10,1.000000,0.992067,0.966928,0.915121,0.974983,-0.024347,-0.024347,-0.024347,-0.038033,-0.038033,...,-0.020197,-0.056747,-0.077064,-0.102763,-0.073756,-0.020697,-0.035995,-0.022219,-0.040938,-0.064184
atr_norm_14,0.992067,1.000000,0.988862,0.948696,0.947371,-0.011299,-0.011299,-0.011299,-0.022814,-0.022814,...,-0.005472,-0.038536,-0.060615,-0.093576,-0.055760,-0.008013,-0.020770,-0.007533,-0.023318,-0.044887
atr_norm_20,0.966928,0.988862,1.000000,0.980521,0.907586,0.000710,0.000710,0.000710,-0.008124,-0.008124,...,0.007984,-0.019144,-0.040288,-0.078786,-0.036142,0.003687,-0.006041,0.006018,-0.006045,-0.024148
atr_norm_30,0.915121,0.948696,0.980521,1.000000,0.846453,0.012321,0.012321,0.012321,0.006629,0.006629,...,0.020773,0.001738,-0.015997,-0.055512,-0.014237,0.015118,0.008860,0.019162,0.011558,-0.001649
atr_norm_5,0.974983,0.947371,0.907586,0.846453,1.000000,-0.054511,-0.054511,-0.054511,-0.069832,-0.069832,...,-0.053817,-0.085914,-0.097646,-0.111769,-0.105093,-0.050368,-0.068440,-0.055760,-0.075996,-0.096517
bb_15_15,-0.024347,-0.011299,0.000710,0.012321,-0.054511,1.000000,1.000000,1.000000,0.963866,0.963866,...,0.856572,0.625922,0.495718,0.333965,0.853612,0.954202,0.958193,0.954631,0.900817,0.807723
bb_15_20,-0.024347,-0.011299,0.000710,0.012321,-0.054511,1.000000,1.000000,1.000000,0.963866,0.963866,...,0.856572,0.625922,0.495718,0.333965,0.853612,0.954202,0.958193,0.954631,0.900817,0.807723
bb_15_25,-0.024347,-0.011299,0.000710,0.012321,-0.054511,1.000000,1.000000,1.000000,0.963866,0.963866,...,0.856572,0.625922,0.495718,0.333965,0.853612,0.954202,0.958193,0.954631,0.900817,0.807723
bb_20_15,-0.038033,-0.022814,-0.008124,0.006629,-0.069832,0.963866,0.963866,0.963866,1.000000,1.000000,...,0.860558,0.745965,0.585921,0.391741,0.912062,0.922037,0.958074,0.935215,0.950860,0.877672
bb_20_20,-0.038033,-0.022814,-0.008124,0.006629,-0.069832,0.963866,0.963866,0.963866,1.000000,1.000000,...,0.860558,0.745965,0.585921,0.391741,0.912062,0.922037,0.958074,0.935215,0.950860,0.877672



Correlation mean | target=ret | H=90 | window=gestation



,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,bb_20_25,bb_30_15,bb_30_20,bb_30_25,bb_60_15,...,roc_30,roc_5,roc_60,rsi_14,rsi_3,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30
bb_15_15,1.000000,1.000000,1.000000,0.956889,0.956889,0.956889,0.859749,0.859749,0.859749,0.709812,...,0.423440,0.813880,0.348695,0.862582,0.864447,0.941731,0.947120,0.930236,0.873873,0.787426
bb_15_20,1.000000,1.000000,1.000000,0.956889,0.956889,0.956889,0.859749,0.859749,0.859749,0.709812,...,0.423440,0.813880,0.348695,0.862582,0.864447,0.941731,0.947120,0.930236,0.873873,0.787426
bb_15_25,1.000000,1.000000,1.000000,0.956889,0.956889,0.956889,0.859749,0.859749,0.859749,0.709812,...,0.423440,0.813880,0.348695,0.862582,0.864447,0.941731,0.947120,0.930236,0.873873,0.787426
bb_20_15,0.956889,0.956889,0.956889,1.000000,1.000000,1.000000,0.930024,0.930024,0.930024,0.777850,...,0.487157,0.762107,0.379530,0.908635,0.801862,0.908519,0.943312,0.908211,0.919703,0.843163
bb_20_20,0.956889,0.956889,0.956889,1.000000,1.000000,1.000000,0.930024,0.930024,0.930024,0.777850,...,0.487157,0.762107,0.379530,0.908635,0.801862,0.908519,0.943312,0.908211,0.919703,0.843163
bb_20_25,0.956889,0.956889,0.956889,1.000000,1.000000,1.000000,0.930024,0.930024,0.930024,0.777850,...,0.487157,0.762107,0.379530,0.908635,0.801862,0.908519,0.943312,0.908211,0.919703,0.843163
bb_30_15,0.859749,0.859749,0.859749,0.930024,0.930024,0.930024,1.000000,1.000000,1.000000,0.871037,...,0.614114,0.679809,0.418583,0.941572,0.708204,0.835082,0.899470,0.833179,0.892433,0.905489
bb_30_20,0.859749,0.859749,0.859749,0.930024,0.930024,0.930024,1.000000,1.000000,1.000000,0.871037,...,0.614114,0.679809,0.418583,0.941572,0.708204,0.835082,0.899470,0.833179,0.892433,0.905489
bb_30_25,0.859749,0.859749,0.859749,0.930024,0.930024,0.930024,1.000000,1.000000,1.000000,0.871037,...,0.614114,0.679809,0.418583,0.941572,0.708204,0.835082,0.899470,0.833179,0.892433,0.905489
bb_60_15,0.709812,0.709812,0.709812,0.777850,0.777850,0.777850,0.871037,0.871037,0.871037,1.000000,...,0.744836,0.560734,0.510649,0.904888,0.581470,0.706366,0.785094,0.701022,0.773832,0.841329



Correlation mean | target=ret | H=90 | window=execution



,atr_norm_10,atr_norm_14,atr_norm_20,atr_norm_30,atr_norm_5,bb_15_15,bb_15_20,bb_15_25,bb_20_15,bb_20_20,...,roc_30,roc_5,roc_60,rsi_14,rsi_3,rsi_5,rsi_7,stoch_k_14,stoch_k_20,stoch_k_30
atr_norm_10,1.000000,0.936142,0.750691,0.439044,0.860758,-0.042031,-0.042031,-0.042031,-0.063475,-0.063475,...,-0.140618,-0.006208,-0.192095,-0.115897,-0.008495,-0.040204,-0.064913,-0.051235,-0.081290,-0.117203
atr_norm_14,0.936142,1.000000,0.907942,0.642641,0.694459,-0.021437,-0.021437,-0.021437,-0.041822,-0.041822,...,-0.121502,0.010501,-0.182564,-0.091001,0.008206,-0.018485,-0.041017,-0.024771,-0.053228,-0.089948
atr_norm_20,0.750691,0.907942,1.000000,0.860021,0.461615,-0.002109,-0.002109,-0.002109,-0.020557,-0.020557,...,-0.094898,0.022938,-0.160708,-0.065453,0.021823,0.000765,-0.018925,-0.000308,-0.025646,-0.060311
atr_norm_30,0.439044,0.642641,0.860021,1.000000,0.156690,0.017188,0.017188,0.017188,0.002499,0.002499,...,-0.059977,0.032746,-0.110768,-0.034721,0.033999,0.019737,0.004267,0.023027,0.003558,-0.024369
atr_norm_5,0.860758,0.694459,0.461615,0.156690,1.000000,-0.086443,-0.086443,-0.086443,-0.106885,-0.106885,...,-0.153624,-0.045926,-0.188627,-0.155153,-0.046936,-0.085342,-0.111017,-0.103367,-0.130240,-0.158234
bb_15_15,-0.042031,-0.021437,-0.002109,0.017188,-0.086443,1.000000,1.000000,1.000000,0.963165,0.963165,...,0.472150,0.829807,0.347656,0.862131,0.875124,0.945897,0.947916,0.941739,0.891045,0.817084
bb_15_20,-0.042031,-0.021437,-0.002109,0.017188,-0.086443,1.000000,1.000000,1.000000,0.963165,0.963165,...,0.472150,0.829807,0.347656,0.862131,0.875124,0.945897,0.947916,0.941739,0.891045,0.817084
bb_15_25,-0.042031,-0.021437,-0.002109,0.017188,-0.086443,1.000000,1.000000,1.000000,0.963165,0.963165,...,0.472150,0.829807,0.347656,0.862131,0.875124,0.945897,0.947916,0.941739,0.891045,0.817084
bb_20_15,-0.063475,-0.041822,-0.020557,0.002499,-0.106885,0.963165,0.963165,0.963165,1.000000,1.000000,...,0.542095,0.784177,0.389328,0.908718,0.817709,0.917842,0.947306,0.925949,0.932238,0.870237
bb_20_20,-0.063475,-0.041822,-0.020557,0.002499,-0.106885,0.963165,0.963165,0.963165,1.000000,1.000000,...,0.542095,0.784177,0.389328,0.908718,0.817709,0.917842,0.947306,0.925949,0.932238,0.870237


### **5.1.3. Análisis de Matrices de Correlación entre Indicadores**

#### **Comparación de Targets: `delta` vs `ret`**


**Conclusión clave**

Estos resultados **no contradicen** la conclusión previa de que **`ret` es un mejor objetivo que `delta`**.  
Lo que muestran estas tablas es **estructura y dependencia entre features**, **no calidad predictiva del target**.

---

**1. Colinealidad extremadamente alta entre features**

En **todas las ventanas** (`full_day`, `gestation`, `execution`) y **todos los horizontes** (H=60, H=90):

- Los indicadores de una misma familia (BB, EMA, RSI, Stoch) presentan **correlaciones muy cercanas a 1**.
- Cambiar parámetros (15/20/25, 30/60, etc.) **no introduce información sustancialmente nueva**.

**Implicación directa**  
El espacio de features presenta **redundancia estructural**.  
Sin una selección agresiva o regularización fuerte, los modelos:
- no ganan capacidad predictiva,
- y este efecto es especialmente perjudicial con targets más ruidosos como `delta`.

---

**2. Delta vs Ret: mismas dependencias, distinta naturaleza del target**

Las **matrices de correlación feature–feature son prácticamente idénticas** entre `delta` y `ret`:

- Mismos bloques colineales.
- Valores numéricamente casi iguales en todas las combinaciones de ventana y horizonte.

Esto es esperable porque, a escala intradía:

$$
\text{ret}_h \;\approx\; \frac{\text{delta}_h}{\text{close}_t}
$$

Es decir, `ret` es una **transformación monótona y casi lineal** de `delta`.

**Lectura correcta**  
La ventaja de `ret` **no proviene de “mejores features”**, sino de la **normalización implícita del target**:
- `ret` reduce la dependencia del nivel absoluto de precios.
- `delta` amplifica el ruido cuando cambia el régimen de volatilidad.

Por ello, aunque las correlaciones entre features sean iguales, **`ret` sigue siendo un objetivo conceptualmente más estable**.

---

**3. Qué miden (y qué no miden) estas tablas**

Estas matrices muestran:

- **Correlación feature–feature promedio** (Spearman diario).

No muestran:

- IC feature–target.
- Performance predictiva OOS.
- Capacidad de generalización del modelo.

**Por lo tanto**:
- No indican cuál target es “mejor”.
- Solo evidencian **redundancia y colinealidad** del espacio de features, elevada en ambos casos.

---

**4. Comportamiento por ventana intradía**

  - **Full Day / Gestation**

    - Dominan señales de **nivel y momentum suave** (EMA, RSI, Stoch).
    - ROC largos (30–60) pierden fuerza relativa.
    - No se observa una diferenciación clara que justifique `delta` frente a `ret`.

  - **Execution**

    - El **ATR normalizado aparece claramente desacoplado** del resto:
      - Correlaciones bajas o negativas con indicadores direccionales.
      - Representa una dimensión distinta: **volatilidad / amplitud del movimiento**.

  Esto refuerza que:

  - `delta` en execution mezcla **dirección + volatilidad**.
  - `ret` aísla mejor la **componente direccional pura**.

---

**5. Efecto del horizonte (H=60 vs H=90)**

Al aumentar el horizonte:

- Crece la correlación entre indicadores de mayor lookback (EMA_60, ROC_60).
- Aumenta la influencia indirecta de la volatilidad.

Este efecto:
- Penaliza más a `delta`, cuya varianza crece con el horizonte.
- Afecta menos a `ret`, que mantiene una escala comparable.

---

**6. La superioridad de `ret` se demostró en otro nivel**

Previamente ya se había demostrado (correctamente) que `ret`:

- Es más **estacionario**.
- Es **escala-invariante**.
- Generaliza mejor **cross-regime** y **out-of-sample**.
- Es más consistente con métricas tipo **IC**, **rank** y modelos no lineales.

Nada de eso se invalida con estas matrices.

---

**7. Lectura correcta de los resultados**

✔ Confirman que el problema principal **no es el target**, sino:
- Fuerte multicolinealidad.
- Redundancia extrema entre indicadores.

✔ Refuerzan la necesidad de:
- Clustering de features.
- Selección por IC.
- Reducción de dimensionalidad o pruning agresivo.

✖ **No aportan evidencia para volver a `delta` como target principal**.

---

**Conclusión técnica final**

Estas matrices **refuerzan**, y no contradicen, la conclusión metodológica previa:

> **`ret` es un target más adecuado para modelado estadístico y ML**,  
> incluso cuando los drivers técnicos subyacentes son los mismos que para `delta`.

El foco del problema y de la mejora del pipeline debe ponerse en la **reducción del espacio de features**, no en redefinir el target.


##**5.2. Clusters de Correlación**


El análisis de las matrices de correlación entre indicadores técnicos evidencia un **alto nivel de colinealidad estructural**, tanto entre distintas familias de indicadores como entre múltiples parametrizaciones de un mismo indicador (por ejemplo, Bandas de Bollinger, EMA, RSI y Stochastic). Este comportamiento se observa de forma consistente:

- en todas las ventanas intradía consideradas (`full_day`, `gestation`, `execution`),
- en ambos horizontes de predicción (60 y 90 minutos),
- y para ambos targets analizados (`delta` y `ret`).

Dado que estas dependencias **no dependen del target**, sino que son una propiedad intrínseca del espacio de features, resulta necesario aplicar un procedimiento sistemático de **agrupamiento por correlación** con los siguientes objetivos:

- agrupar indicadores que capturan esencialmente la misma información,
- reducir la dimensionalidad sin pérdida significativa de señal,
- seleccionar uno o pocos representantes por grupo (o combinar señales),
- disminuir el riesgo de sobreajuste en modelos predictivos,
- mejorar la interpretabilidad y robustez del conjunto final de features.

---

**Definición conceptual (en términos de grafo)**

El procedimiento de clustering se define de la siguiente manera:

- Cada **indicador técnico** se modela como un **nodo** de un grafo no dirigido.
- Se define una **arista** entre dos indicadores \( i, j \) si la correlación absoluta entre ellos cumple:

$$
|\rho_{ij}| \geq 0.85
$$

- Cada **componente conexa** del grafo resultante se interpreta como un **cluster informativo**.

---

**Interpretación operativa**

Bajo esta formulación, cada cluster representa un conjunto de indicadores que:

- presentan dependencias estadísticas muy fuertes,
- aportan información prácticamente redundante,
- y pueden considerarse, a efectos prácticos, como una misma *familia operativa*.

Por lo tanto, dentro de cada cluster es suficiente:

- seleccionar un único indicador representativo, o
- construir una señal agregada (por ejemplo, promedio o componente principal),

sin incurrir en una pérdida relevante de información predictiva.

---

**Justificación metodológica**

Este enfoque está directamente respaldado por los resultados previos:

- Las matrices de correlación feature–feature son prácticamente idénticas entre `delta` y `ret`.
- La colinealidad se mantiene estable entre ventanas y horizontes.
- La diferencia de desempeño entre targets **no proviene del espacio de features**, sino de la formulación del target.

En consecuencia, el clustering por correlación constituye un **paso metodológico clave** para desacoplar el problema de **selección de variables** del problema de **definición del target**, permitiendo evaluaciones más limpias y comparables entre distintos enfoques de modelado.

---

**Resultado esperado**

El resultado de este procedimiento es un **conjunto reducido y no redundante de indicadores técnicos**, apto para:

- entrenamiento de modelos estadísticos y de aprendizaje automático,
- análisis comparativo entre `delta` y `ret`,
- y evaluación robusta de desempeño out-of-sample.


### **5.2.1. Código para construcción de clusters**

Este código:
- toma la matriz de correlación media (corr_mean_ic_h_relevant)
- detecta clusters automáticamente
- devuelve un DataFrame limpio para inspección

In [186]:
import numpy as np
import pandas as pd
import networkx as nx


def build_correlation_clusters(
    corr: pd.DataFrame,
    threshold: float = 0.85,
    *,
    use_upper_triangle: bool = True,
) -> pd.DataFrame:
    """
    Construye clusters de indicadores basados en alta correlación (|rho| >= threshold).

    Idea:
      - Cada indicador = nodo
      - Conectamos dos nodos con una arista si |rho| >= threshold
      - Cada componente conexa del grafo = cluster de redundancia

    Parameters
    ----------
    corr : pd.DataFrame
        Matriz de correlación (Spearman promedio), index = columns = indicadores.
    threshold : float
        Umbral de |rho| para considerar redundancia.
    use_upper_triangle : bool
        Si True, recorre solo el triángulo superior (más eficiente y sin duplicar aristas).

    Returns
    -------
    clusters_df : pd.DataFrame
        Columnas:
          - cluster_id
          - indicator
          - n_in_cluster
    """

    # -------------------------
    # 0) Validaciones mínimas
    # -------------------------
    if not isinstance(corr, pd.DataFrame) or corr.empty:
        raise ValueError("corr debe ser un DataFrame no vacío.")

    if corr.shape[0] != corr.shape[1]:
        raise ValueError("corr debe ser una matriz cuadrada (NxN).")

    if list(corr.index) != list(corr.columns):
        # No es obligatorio, pero evita sorpresas
        corr = corr.copy()
        corr = corr.loc[corr.index, corr.index]

    indicators = corr.columns.tolist()

    # -------------------------
    # 1) Crear grafo
    # -------------------------
    G = nx.Graph()
    G.add_nodes_from(indicators)

    # -------------------------
    # 2) Agregar aristas (|rho| >= threshold)
    # -------------------------
    if use_upper_triangle:
        # Recorre solo i < j (sin duplicar)
        for a, i in enumerate(indicators):
            for j in indicators[a + 1:]:
                rho = corr.at[i, j]
                if pd.notna(rho) and abs(rho) >= threshold:
                    G.add_edge(i, j, weight=float(rho))
    else:
        # Recorre todo (más lento, pero explícito)
        for i in indicators:
            for j in indicators:
                if i == j:
                    continue
                rho = corr.at[i, j]
                if pd.notna(rho) and abs(rho) >= threshold:
                    G.add_edge(i, j, weight=float(rho))

    # -------------------------
    # 3) Componentes conexas = clusters
    # -------------------------
    components = list(nx.connected_components(G))

    # -------------------------
    # 4) Armar DataFrame de salida
    # -------------------------
    records = []
    for cluster_id, comp in enumerate(components):
        comp = sorted(comp)
        n = len(comp)
        for ind in comp:
            records.append({"cluster_id": cluster_id, "indicator": ind, "n_in_cluster": n})

    clusters_df = (
        pd.DataFrame(records)
          .sort_values(["n_in_cluster", "cluster_id", "indicator"], ascending=[False, True, True])
          .reset_index(drop=True)
    )

    return clusters_df


### **5.2.2. Aplicación**

- `cluster_id`: identifica cada grupo informativo
- `n_in_cluster`:
  - 1 → indicador independiente
  - $>$ 1 → redundancia fuerte

In [187]:
import pandas as pd

# ============================================================
# Cálculo de clusters para (target, window, horizon)
# ============================================================

THRESH = 0.85

TARGETS  = ("delta", "ret")
WINDOWS  = ("full_day", "gestation", "execution")
HORIZONS = (60, 90)

# corr_mean debe ser un dict indexado como: corr_mean[(target, window, horizon)] -> DataFrame corr
# Ej: corr_mean[("delta","full_day",60)]

clusters = {}
for target in TARGETS:
    for window in WINDOWS:
        for horizon in HORIZONS:
            key = (target, window, horizon)
            corr_mat = corr_mean[key]
            clusters[key] = build_correlation_clusters(
                corr=corr_mat,
                threshold=THRESH,
                use_upper_triangle=True,
            )

# ============================================================
# Accesos “tipo variable” (si los quiere)
# ============================================================
clusters_delta_60_full_day  = clusters[("delta", "full_day", 60)]
clusters_delta_90_full_day  = clusters[("delta", "full_day", 90)]
clusters_delta_60_gestation = clusters[("delta", "gestation", 60)]
clusters_delta_90_gestation = clusters[("delta", "gestation", 90)]
clusters_delta_60_execution = clusters[("delta", "execution", 60)]
clusters_delta_90_execution = clusters[("delta", "execution", 90)]

clusters_ret_60_full_day    = clusters[("ret", "full_day", 60)]
clusters_ret_90_full_day    = clusters[("ret", "full_day", 90)]
clusters_ret_60_gestation   = clusters[("ret", "gestation", 60)]
clusters_ret_90_gestation   = clusters[("ret", "gestation", 90)]
clusters_ret_60_execution   = clusters[("ret", "execution", 60)]
clusters_ret_90_execution   = clusters[("ret", "execution", 90)]

# ============================================================
# (Opcional) Comparación delta vs ret: nº clusters y tamaños
# ============================================================

def cluster_size_profile(clusters_df: pd.DataFrame) -> list[int]:
    """Devuelve el perfil de tamaños de cluster (ordenado desc)."""
    sizes = (
        clusters_df[["cluster_id", "n_in_cluster"]]
        .drop_duplicates()
        .sort_values("n_in_cluster", ascending=False)["n_in_cluster"]
        .tolist()
    )
    return sizes

rows = []
for window in WINDOWS:
    for horizon in HORIZONS:
        c_delta = clusters[("delta", window, horizon)]
        c_ret   = clusters[("ret",   window, horizon)]

        rows.append({
            "window": window,
            "horizon": horizon,
            "n_clusters_delta": c_delta["cluster_id"].nunique(),
            "n_clusters_ret":   c_ret["cluster_id"].nunique(),
            "top_sizes_delta":  cluster_size_profile(c_delta)[:8],  # primeras 8
            "top_sizes_ret":    cluster_size_profile(c_ret)[:8],
        })

cluster_compare = pd.DataFrame(rows).sort_values(["window", "horizon"]).reset_index(drop=True)
cluster_compare


,window,horizon,n_clusters_delta,n_clusters_ret,top_sizes_delta,top_sizes_ret
0,execution,60,5,5,"[29, 5, 1, 1, 1]","[29, 5, 1, 1, 1]"
1,execution,90,5,5,"[29, 5, 1, 1, 1]","[29, 5, 1, 1, 1]"
2,full_day,60,2,2,"[15, 1]","[15, 1]"
3,full_day,90,3,3,"[26, 5, 1]","[26, 5, 1]"
4,gestation,60,6,6,"[24, 3, 2, 1, 1, 1]","[24, 3, 2, 1, 1, 1]"
5,gestation,90,6,6,"[24, 3, 2, 1, 1, 1]","[24, 3, 2, 1, 1, 1]"


**Observación sobre clusters de correlación**

Cada fila resume, para una combinación *(ventana, horizonte)*, la **estructura de redundancia** del espacio de features:
- `n_clusters_*`: cantidad de clusters de correlación.
- `top_sizes_*`: tamaños de clusters (ej. `[29, 5, 1, 1, 1]` indica un cluster dominante, uno secundario y señales casi independientes).

---

**Hallazgo central**

> **Delta y ret producen exactamente el mismo clustering**  
> en todas las ventanas y horizontes.

- Mismo número de clusters.
- Mismos tamaños.
- Misma geometría del espacio de features.

**Conclusión:**  
La diferencia entre targets **no se explica por la estructura de correlación**, sino por la naturaleza del target. El espacio de features es idéntico para `delta` y `ret`.

---

**Interpretación por ventana**

- **Gestation (H=60 / H=90)**  
  Clusters: 6 → `[24, 3, 2, 1, 1, 1]`  
  - Mayor diversidad estructural.
  - Mejor candidata para combinar señales.

- **Execution (H=60 / H=90)**  
  Clusters: 5 → `[29, 5, 1, 1, 1]`  
  - Mega-cluster dominante (BB, EMA, RSI, Stoch, ROC).  
  - Cluster secundario (volatilidad / ATR).  
  - Señal extremadamente redundante.

- **Full Day**  
  - H=60: 2 clusters → `[15, 1]` (muy baja diversidad).  
  - H=90: 3 clusters → `[26, 5, 1]` (ligero aumento de complejidad).

---

**Conclusiones técnicas**

- El clustering es **independiente del target** (`delta` vs `ret`).
- La **multicolinealidad es extrema**, especialmente en execution.
- **Reducir features es obligatorio**; usar todos es estadísticamente subóptimo.
- Las ventanas no son equivalentes:
  - Execution: señal fuerte pero muy redundante.
  - Gestation: mayor complementariedad.
  - Full day: menor diversidad.

---

**Implicación directa para el pipeline**

- Mantener **ambos targets** (`delta` y `ret`).
- Seleccionar features como:
  - **1 representante por cluster** en execution y full day.
  - **1–2 representantes por cluster** en gestation.
- Elegir representantes por IC OOS, estabilidad o mínima correlación intra-cluster.

---

**Conclusión final:**  
> El problema no está en el target, sino en la **redundancia del espacio de features**.


### **5.2.3. Display de clusters**

In [188]:
for target in ("delta", "ret"):
    print("\n" + "=" * 70)
    print(f"TARGET: {target.upper()}")
    print("=" * 70)

    print(f"\nclusters_{target}_60_full_day:")

    print(f"\nclusters_{target}_60_gestation:")

    print(f"\nclusters_{target}_60_execution:")

    print(f"\nclusters_{target}_90_full_day:")

    print(f"\nclusters_{target}_90_gestation:")

    print(f"\nclusters_{target}_90_execution:")



TARGET: DELTA

clusters_delta_60_full_day:

clusters_delta_60_gestation:

clusters_delta_60_execution:

clusters_delta_90_full_day:

clusters_delta_90_gestation:

clusters_delta_90_execution:

TARGET: RET

clusters_ret_60_full_day:

clusters_ret_60_gestation:

clusters_ret_60_execution:

clusters_ret_90_full_day:

clusters_ret_90_gestation:

clusters_ret_90_execution:


In [189]:
# ============================================================
# Display de clusters (delta/ret) por ventana y horizonte
# ============================================================

for target in ("delta", "ret"):
    print("\n" + "=" * 70)
    print(f"TARGET: {target.upper()}")
    print("=" * 70)

    print(f"\nclusters_{target}_60_full_day:")
    display(clusters[(target, "full_day", 60)])

    print(f"\nclusters_{target}_60_gestation:")
    display(clusters[(target, "gestation", 60)])

    print(f"\nclusters_{target}_60_execution:")
    display(clusters[(target, "execution", 60)])

    print(f"\nclusters_{target}_90_full_day:")
    display(clusters[(target, "full_day", 90)])

    print(f"\nclusters_{target}_90_gestation:")
    display(clusters[(target, "gestation", 90)])

    print(f"\nclusters_{target}_90_execution:")
    display(clusters[(target, "execution", 90)])



TARGET: DELTA

clusters_delta_60_full_day:


,cluster_id,indicator,n_in_cluster
0,0,bb_30_15,15
1,0,bb_30_20,15
2,0,bb_30_25,15
3,0,bb_60_15,15
4,0,bb_60_20,15
5,0,bb_60_25,15
6,0,ema_15,15
7,0,ema_20,15
8,0,ema_30,15
9,0,ema_60,15



clusters_delta_60_gestation:


,cluster_id,indicator,n_in_cluster
0,0,bb_15_15,24
1,0,bb_15_20,24
2,0,bb_15_25,24
3,0,bb_20_15,24
4,0,bb_20_20,24
5,0,bb_20_25,24
6,0,bb_30_15,24
7,0,bb_30_20,24
8,0,bb_30_25,24
9,0,bb_60_15,24



clusters_delta_60_execution:


...

,cluster_id,indicator,n_in_cluster
0,1,bb_15_15,26
1,1,bb_15_20,26
2,1,bb_15_25,26
3,1,bb_20_15,26
4,1,bb_20_20,26
5,1,bb_20_25,26
6,1,bb_30_15,26
7,1,bb_30_20,26
8,1,bb_30_25,26
9,1,bb_60_15,26



clusters_ret_90_gestation:


,cluster_id,indicator,n_in_cluster
0,0,bb_15_15,24
1,0,bb_15_20,24
2,0,bb_15_25,24
3,0,bb_20_15,24
4,0,bb_20_20,24
5,0,bb_20_25,24
6,0,bb_30_15,24
7,0,bb_30_20,24
8,0,bb_30_25,24
9,0,bb_60_15,24



clusters_ret_90_execution:


,cluster_id,indicator,n_in_cluster
0,1,bb_15_15,29
1,1,bb_15_20,29
2,1,bb_15_25,29
3,1,bb_20_15,29
4,1,bb_20_20,29
5,1,bb_20_25,29
6,1,bb_30_15,29
7,1,bb_30_20,29
8,1,bb_30_25,29
9,1,bb_60_15,29


### **5.2.4. Observaciones sobre los clusters de correlación (detalle por target, ventana y horizonte)**

**Hallazgo principal**
- **Delta y Ret generan exactamente los mismos clusters** en todas las ventanas y horizontes.
- La **estructura de redundancia del espacio de features es idéntica** para ambos targets.
- Por lo tanto, las diferencias de desempeño **no provienen de los indicadores**, sino del **target**.

---

**Full Day**
- **H=60**:  
  - 1 cluster dominante (15 indicadores) + 1 señal aislada (`roc_60`).  
  - **Muy baja diversidad informativa**.
- **H=90**:  
  - 1 cluster grande (26) + cluster de volatilidad (ATR, 5) + 1 aislado.  
  - El aumento de horizonte introduce **ligera complejidad**, pero sigue siendo redundante.

---

**Gestation**
- **H=60 / H=90**:  
  - 1 cluster dominante (24 indicadores).  
  - Clusters secundarios pequeños (3, 2) y varias señales aisladas (ROC largos).  
  - **Mayor diversidad estructural** que full day.
- Es la ventana **más apta para combinar señales**.

---

**Execution**
- **H=60 / H=90**:  
  - 1 mega-cluster dominante (29 indicadores direccionales).  
  - 1 cluster claro de **volatilidad** (`atr_norm_*`, tamaño 5).  
  - ROC largos aparecen como señales casi independientes.
- **Máxima colinealidad** del pipeline.

---

**Conclusiones operativas**
- La **multicolinealidad es extrema**, especialmente en execution.
- Usar todos los indicadores es **estadísticamente innecesario**.
- Selección recomendada:
  - **Execution / Full day**: 1 representante por cluster.
  - **Gestation**: 1–2 representantes por cluster.
- Criterio de elección: IC OOS, estabilidad temporal o mínima correlación intra-cluster.

---

**Conclusión final:**  
> El cuello de botella del pipeline no es el target, sino la **redundancia del espacio de features**.


##**5.3. Procesamiento de clusters**


### **5.3.1. Resolución de cluster dominante**

La selección se realiza **dentro de cada `cluster_id`**, eligiendo el indicador con **mayor potencia predictiva**, de acuerdo con los siguientes criterios jerárquicos:

1. **Máximo valor absoluto de IC out-of-sample**  
   - Se utiliza como métrica principal el **|IC_OOS|**, calculado sobre el target correspondiente (`delta` o `ret`).
2. **Criterio de desempate**  
   - En caso de empate, se selecciona el indicador con mayor **|IC_trade_only|**.
3. **Consistencia temporal** *(opcional)*  
   - De persistir el empate, se prioriza el indicador con mayor estabilidad del IC a lo largo del tiempo.

Este procedimiento garantiza que cada cluster quede representado por el **indicador más informativo**, preservando capacidad predictiva y reduciendo el riesgo de sobreajuste, al tiempo que mantiene la coherencia metodológica entre targets, ventanas y horizontes.

### **5.3.2. Código**

In [190]:
import numpy as np
import pandas as pd

def resolve_clusters_by_ic_oos(
    ic_relevant: pd.DataFrame,
    clusters_df: pd.DataFrame,
    *,
    prefer_trade_only_tiebreak: bool = True,
    prefer_ic_is_tiebreak: bool = True,
) -> pd.DataFrame:
    """
    Selecciona 1 indicador por cluster, priorizando generalización OOS.

    Selección (orden de prioridad):
      1) Mayor abs_IC_OOS
      2) (opcional, si existe) Mayor abs_IC_trade_only
      3) (opcional) Mayor abs(IC_IS)
      4) Orden alfabético (determinístico)

    Entradas esperadas (ic_relevant) mínimo:
      ['indicator','horizon','IC_IS','IC_OOS','abs_IC_OOS','n_pairs_OOS','note']
    Opcional para desempate:
      ['IC_trade_only'] o ['abs_IC_trade_only'] (si existe alguno, se usa)

    Entradas esperadas (clusters_df):
      ['cluster_id','indicator','n_in_cluster']
      (opcional) 'horizon' -> si existe, el merge se hace por ['indicator','horizon'].

    Devuelve columnas:
      horizon, cluster_id, n_in_cluster, indicator, keep, reason,
      IC_IS, IC_OOS, abs_IC_OOS, n_pairs_OOS, note
      (+ IC_trade_only / abs_IC_trade_only si estaban disponibles)
    """
    needed_ic = {"indicator", "horizon", "IC_IS", "IC_OOS", "abs_IC_OOS", "n_pairs_OOS", "note"}
    needed_cl = {"cluster_id", "indicator", "n_in_cluster"}

    if not isinstance(ic_relevant, pd.DataFrame) or ic_relevant.empty:
        raise ValueError("ic_relevant debe ser un DataFrame no vacío.")
    if not isinstance(clusters_df, pd.DataFrame) or clusters_df.empty:
        raise ValueError("clusters_df debe ser un DataFrame no vacío.")

    if not needed_ic.issubset(ic_relevant.columns):
        raise ValueError(f"ic_relevant debe contener: {sorted(needed_ic)}")
    if not needed_cl.issubset(clusters_df.columns):
        raise ValueError(f"clusters_df debe contener: {sorted(needed_cl)}")

    ic = ic_relevant.copy()
    cl = clusters_df.copy()

    # -------------------------
    # 0) Normalizar trade_only (si existe)
    # -------------------------
    has_abs_trade = "abs_IC_trade_only" in ic.columns
    has_trade = "IC_trade_only" in ic.columns

    if has_trade and not has_abs_trade:
        ic["abs_IC_trade_only"] = ic["IC_trade_only"].abs()
        has_abs_trade = True

    # -------------------------
    # 1) Merge robusto (evitar duplicados entre horizontes)
    # -------------------------
    # Si clusters_df tiene 'horizon', hacemos merge por ['indicator','horizon'].
    # Si no, merge por 'indicator' y luego resolvemos por (horizon, cluster_id).
    if "horizon" in cl.columns:
        merge_keys = ["indicator", "horizon"]
        ic_cols = ["indicator", "horizon", "IC_IS", "IC_OOS", "abs_IC_OOS", "n_pairs_OOS", "note"]
        if has_trade:
            ic_cols.append("IC_trade_only")
        if has_abs_trade and "abs_IC_trade_only" not in ic_cols:
            ic_cols.append("abs_IC_trade_only")

        df = cl.merge(ic[ic_cols], on=merge_keys, how="left")

    else:
        # Validación: ic_relevant no debe tener duplicados por (indicator, horizon)
        dup = ic.duplicated(subset=["indicator", "horizon"], keep=False)
        if dup.any():
            bad = ic.loc[dup, ["indicator", "horizon"]].drop_duplicates().head(20)
            raise ValueError(
                "ic_relevant tiene duplicados por (indicator, horizon). "
                f"Ejemplos: {bad.to_dict(orient='records')}"
            )

        ic_cols = ["indicator", "horizon", "IC_IS", "IC_OOS", "abs_IC_OOS", "n_pairs_OOS", "note"]
        if has_trade:
            ic_cols.append("IC_trade_only")
        if has_abs_trade and "abs_IC_trade_only" not in ic_cols:
            ic_cols.append("abs_IC_trade_only")

        df = cl.merge(ic[ic_cols], on="indicator", how="left")

    # -------------------------
    # 2) Validar que todos los indicadores tengan IC_OOS
    # -------------------------
    missing = df[df["abs_IC_OOS"].isna()]["indicator"].unique().tolist()
    if missing:
        raise ValueError(f"Indicadores en clusters sin IC_OOS en ic_relevant: {missing}")

    out_rows = []

    # -------------------------
    # 3) Resolver por (horizon, cluster_id)
    # -------------------------
    for (h, cid), g in df.groupby(["horizon", "cluster_id"], sort=True):
        g = g.copy()

        # 1) criterio principal: max abs_IC_OOS
        max_abs_oos = g["abs_IC_OOS"].max()
        candidates = g[g["abs_IC_OOS"] == max_abs_oos].copy()

        # 2) desempate: abs_IC_trade_only (si existe y se solicita)
        trade_tb_used = False
        if (
            prefer_trade_only_tiebreak
            and has_abs_trade
            and "abs_IC_trade_only" in candidates.columns
            and len(candidates) > 1
        ):
            # si hay NaNs, los tratamos como -inf para que no ganen el tiebreak
            candidates["_abs_trade_tb"] = candidates["abs_IC_trade_only"].fillna(-np.inf)
            max_abs_trade = candidates["_abs_trade_tb"].max()
            candidates = candidates[candidates["_abs_trade_tb"] == max_abs_trade].copy()
            trade_tb_used = True

        # 3) desempate: abs(IC_IS) (si se solicita)
        icis_tb_used = False
        if prefer_ic_is_tiebreak and len(candidates) > 1:
            candidates["_abs_icis_tb"] = candidates["IC_IS"].abs()
            candidates = candidates.sort_values(["_abs_icis_tb", "indicator"], ascending=[False, True])
            best = candidates.iloc[0]
            icis_tb_used = True
        else:
            # 4) determinístico: alfabético
            candidates = candidates.sort_values("indicator", ascending=True)
            best = candidates.iloc[0]

        best_indicator = best["indicator"]

        for _, row in g.iterrows():
            keep = (row["indicator"] == best_indicator)

            if keep:
                reason = "max abs_IC_OOS"
                if trade_tb_used:
                    reason += " (tiebreak: abs_IC_trade_only)"
                if icis_tb_used:
                    reason += " (tiebreak: abs(IC_IS))"
                if len(candidates) > 1 and not (trade_tb_used or icis_tb_used):
                    reason += " (tiebreak: indicator order)"
            else:
                reason = f"redundant (same cluster as {best_indicator})"

            rec = {
                "horizon": int(row["horizon"]),
                "cluster_id": int(cid),
                "n_in_cluster": int(row["n_in_cluster"]),
                "indicator": row["indicator"],
                "keep": bool(keep),
                "reason": reason,
                "IC_IS": float(row["IC_IS"]),
                "IC_OOS": float(row["IC_OOS"]),
                "abs_IC_OOS": float(row["abs_IC_OOS"]),
                "n_pairs_OOS": int(row["n_pairs_OOS"]),
                "note": str(row["note"]) if row["note"] is not None else "",
            }

            if has_trade and "IC_trade_only" in row.index and pd.notna(row["IC_trade_only"]):
                rec["IC_trade_only"] = float(row["IC_trade_only"])
            if has_abs_trade and "abs_IC_trade_only" in row.index and pd.notna(row["abs_IC_trade_only"]):
                rec["abs_IC_trade_only"] = float(row["abs_IC_trade_only"])

            out_rows.append(rec)

    resolved = (
        pd.DataFrame(out_rows)
        .sort_values(["horizon", "cluster_id", "keep", "abs_IC_OOS"], ascending=[True, True, False, False])
        .reset_index(drop=True)
    )

    return resolved


### **5.3.3. Aplicación**

#### Mapa explícito de IC por (target, window, horizon)

In [191]:
# ============================================================
# Clusters por target / ventana / horizonte
# ============================================================

CLUSTERS = {
    ("delta", "full_day", 60): clusters_delta_60_full_day,
    ("delta", "full_day", 90): clusters_delta_90_full_day,
    ("delta", "gestation", 60): clusters_delta_60_gestation,
    ("delta", "gestation", 90): clusters_delta_90_gestation,
    ("delta", "execution", 60): clusters_delta_60_execution,
    ("delta", "execution", 90): clusters_delta_90_execution,

    ("ret", "full_day", 60): clusters_ret_60_full_day,
    ("ret", "full_day", 90): clusters_ret_90_full_day,
    ("ret", "gestation", 60): clusters_ret_60_gestation,
    ("ret", "gestation", 90): clusters_ret_90_gestation,
    ("ret", "execution", 60): clusters_ret_60_execution,
    ("ret", "execution", 90): clusters_ret_90_execution,
}

#### Mapa explícito de clusters por (target, window, horizon)

In [192]:
# ============================================================
# IC tables por target / ventana / horizonte
# ============================================================

IC_TABLES = {
    ("delta", "full_day", 60): ic_60_full_day_delta,
    ("delta", "full_day", 90): ic_90_full_day_delta,
    ("delta", "gestation", 60): ic_60_gestation_delta,
    ("delta", "gestation", 90): ic_90_gestation_delta,
    ("delta", "execution", 60): ic_60_execution_delta,
    ("delta", "execution", 90): ic_90_execution_delta,

    ("ret", "full_day", 60): ic_60_full_day_ret,
    ("ret", "full_day", 90): ic_90_full_day_ret,
    ("ret", "gestation", 60): ic_60_gestation_ret,
    ("ret", "gestation", 90): ic_90_gestation_ret,
    ("ret", "execution", 60): ic_60_execution_ret,
    ("ret", "execution", 90): ic_90_execution_ret,
}

#### `prepare_ic_relevant` (sin cambios conceptuales)

In [193]:
TH_IC = 0.02

def prepare_ic_relevant(
    ic_table: pd.DataFrame,
    *,
    horizon: int,
    th: float,
) -> pd.DataFrame:
    dfh = ic_table.copy()

    if "abs_IC_OOS" not in dfh.columns:
        dfh["abs_IC_OOS"] = dfh["IC_OOS"].abs()

    dfh = dfh[dfh["abs_IC_OOS"] >= th].copy()

    if "n_pairs_OOS" not in dfh.columns:
        dfh["n_pairs_OOS"] = np.nan
    if "note" not in dfh.columns:
        dfh["note"] = ""

    return dfh

#### Orquestador correcto: target-aware

In [194]:
# ============================================================
# Resolver por target / ventana / horizonte
# ============================================================

def resolve_target_window(
    *,
    target: str,
    window: str,
    horizon: int,
    threshold_ic: float = TH_IC,
) -> dict[str, object]:
    """
    Resuelve clusters para un (target, ventana, horizonte).
    """
    key = (target, window, horizon)

    ic_table = IC_TABLES[key]
    clusters_df = CLUSTERS[key]

    ic_rel = prepare_ic_relevant(ic_table, horizon=horizon, th=threshold_ic)

    if ic_rel.empty:
        return {
            "final_indicators": [],
            "resolved": pd.DataFrame(),
            "note": f"Sin IC relevante | target={target}, window={window}, H={horizon}",
        }

    clusters_rel = clusters_df[
        clusters_df["indicator"].isin(ic_rel["indicator"])
    ].copy()

    if clusters_rel.empty:
        return {
            "final_indicators": [],
            "resolved": pd.DataFrame(),
            "note": f"Clusters vacíos | target={target}, window={window}, H={horizon}",
        }

    resolved = resolve_clusters_by_ic_oos(
        ic_relevant=ic_rel,
        clusters_df=clusters_rel,
        prefer_trade_only_tiebreak=True,
        prefer_ic_is_tiebreak=True,
    )

    final_indicators = resolved.loc[resolved["keep"], "indicator"].tolist()

    return {
        "final_indicators": final_indicators,
        "resolved": resolved,
        "note": "",
    }


#### Ejecución completa (todo el grid)

In [195]:
RESULTS = {}

for target in ("delta", "ret"):
    for window in ("full_day", "gestation", "execution"):
        for h in (60, 90):
            RESULTS[(target, window, h)] = resolve_target_window(
                target=target,
                window=window,
                horizon=h,
                threshold_ic=TH_IC,
            )


In [196]:
# ============================================================
# Accesos directos (todas las ventanas, horizontes y targets)
# ============================================================

# ----------------------------
# DELTA
# ----------------------------
delta_full_day_60_final = RESULTS[("delta", "full_day", 60)]["final_indicators"]
delta_full_day_90_final = RESULTS[("delta", "full_day", 90)]["final_indicators"]

delta_gestation_60_final = RESULTS[("delta", "gestation", 60)]["final_indicators"]
delta_gestation_90_final = RESULTS[("delta", "gestation", 90)]["final_indicators"]

delta_execution_60_final = RESULTS[("delta", "execution", 60)]["final_indicators"]
delta_execution_90_final = RESULTS[("delta", "execution", 90)]["final_indicators"]

# Tablas resolved (por si quiere auditar decisiones)
delta_full_day_60_resolved = RESULTS[("delta", "full_day", 60)]["resolved"]
delta_full_day_90_resolved = RESULTS[("delta", "full_day", 90)]["resolved"]

delta_gestation_60_resolved = RESULTS[("delta", "gestation", 60)]["resolved"]
delta_gestation_90_resolved = RESULTS[("delta", "gestation", 90)]["resolved"]

delta_execution_60_resolved = RESULTS[("delta", "execution", 60)]["resolved"]
delta_execution_90_resolved = RESULTS[("delta", "execution", 90)]["resolved"]


# ----------------------------
# RET
# ----------------------------
ret_full_day_60_final = RESULTS[("ret", "full_day", 60)]["final_indicators"]
ret_full_day_90_final = RESULTS[("ret", "full_day", 90)]["final_indicators"]

ret_gestation_60_final = RESULTS[("ret", "gestation", 60)]["final_indicators"]
ret_gestation_90_final = RESULTS[("ret", "gestation", 90)]["final_indicators"]

ret_execution_60_final = RESULTS[("ret", "execution", 60)]["final_indicators"]
ret_execution_90_final = RESULTS[("ret", "execution", 90)]["final_indicators"]

# Tablas resolved (por si quiere auditar decisiones)
ret_full_day_60_resolved = RESULTS[("ret", "full_day", 60)]["resolved"]
ret_full_day_90_resolved = RESULTS[("ret", "full_day", 90)]["resolved"]

ret_gestation_60_resolved = RESULTS[("ret", "gestation", 60)]["resolved"]
ret_gestation_90_resolved = RESULTS[("ret", "gestation", 90)]["resolved"]

ret_execution_60_resolved = RESULTS[("ret", "execution", 60)]["resolved"]
ret_execution_90_resolved = RESULTS[("ret", "execution", 90)]["resolved"]


#### Helper único de display

In [197]:
from IPython.display import display

def show_kept_indicators(
    results: dict,
    *,
    target: str,
    window: str,
    horizon: int,
    sort_by: str = "abs_IC_OOS",
    top_n: int | None = None,
):
    """
    Muestra los indicadores conservados (keep=True) para un target/ventana/horizonte.

    results: dict con key (target, window, horizon)
    """
    key = (target, window, horizon)
    res = results[key]["resolved"]

    if res.empty:
        print(f"[{target.upper()} | {window} | H={horizon}] → sin resultados")
        return

    df = (
        res.query("keep")
           .sort_values(sort_by, ascending=False)
           .reset_index(drop=True)
    )

    if top_n is not None:
        df = df.head(top_n)

    print(f"\n[{target.upper()} | {window} | H={horizon}]  (n={len(df)})")
    display(df)


In [198]:
for target in ("delta", "ret"):
    for window in ("full_day", "gestation", "execution"):
        for h in (60, 90):
            show_kept_indicators(
                RESULTS,
                target=target,
                window=window,
                horizon=h,
            )



[DELTA | full_day | H=60]  (n=2)


,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,60,0,15,ema_60,True,max abs_IC_OOS,-0.193498,-0.189089,0.189089,229712,
1,60,1,1,roc_60,True,max abs_IC_OOS,-0.187085,-0.177769,0.177769,229712,



[DELTA | full_day | H=90]  (n=3)


,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,90,1,26,ema_60,True,max abs_IC_OOS,-0.245322,-0.238691,0.238691,229712,
1,90,2,1,roc_60,True,max abs_IC_OOS,-0.234815,-0.225123,0.225123,229712,
2,90,0,5,atr_norm_20,True,max abs_IC_OOS,0.123236,0.107830,0.107830,229712,



[DELTA | gestation | H=60]  (n=6)


...

,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,90,5,1,roc_60,True,max abs_IC_OOS,-0.266423,-0.284787,0.284787,35160,
1,90,0,24,ema_60,True,max abs_IC_OOS,-0.296160,-0.274527,0.274527,35160,
2,90,4,1,roc_30,True,max abs_IC_OOS,-0.229710,-0.197999,0.197999,35160,
3,90,3,1,roc_20,True,max abs_IC_OOS,-0.216050,-0.185151,0.185151,35160,
4,90,1,3,mom_10,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.189195,-0.166238,0.166238,35160,
5,90,2,2,mom_5,True,max abs_IC_OOS (tiebreak: abs(IC_IS)),-0.155953,-0.143128,0.143128,35160,



[RET | execution | H=60]  (n=5)


,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,60,4,1,roc_60,True,max abs_IC_OOS,-0.565021,-0.579610,0.579610,52740,
1,60,1,29,ema_60,True,max abs_IC_OOS,-0.538351,-0.574991,0.574991,52740,
2,60,3,1,roc_30,True,max abs_IC_OOS,-0.416082,-0.457464,0.457464,52740,
3,60,2,1,roc_20,True,max abs_IC_OOS,-0.355241,-0.385675,0.385675,52740,
4,60,0,5,atr_norm_14,True,max abs_IC_OOS,0.180940,0.200456,0.200456,52740,



[RET | execution | H=90]  (n=5)


,horizon,cluster_id,n_in_cluster,indicator,keep,reason,IC_IS,IC_OOS,abs_IC_OOS,n_pairs_OOS,note
0,90,1,29,ema_60,True,max abs_IC_OOS,-0.581363,-0.614028,0.614028,52740,
1,90,4,1,roc_60,True,max abs_IC_OOS,-0.602022,-0.610562,0.610562,52740,
2,90,3,1,roc_30,True,max abs_IC_OOS,-0.450565,-0.495228,0.495228,52740,
3,90,2,1,roc_20,True,max abs_IC_OOS,-0.392366,-0.419271,0.419271,52740,
4,90,0,5,atr_norm_14,True,max abs_IC_OOS,0.184374,0.214745,0.214745,52740,


### **5.3.4. Resultados de la resolución de clusters (síntesis)**

**1. Consistencia total entre targets (delta vs ret)**
- Para **todas las ventanas (full_day, gestation, execution)** y **ambos horizontes (H=60, H=90)**, los **indicadores seleccionados son idénticos** entre *delta* y *ret*.
- Los valores de **IC_IS, IC_OOS y abs_IC_OOS** son prácticamente iguales.
- Esto confirma que la **estructura informativa y la jerarquía de señales no dependen del target**, sino del régimen temporal.

---

**2. Dominancia clara de familias de indicadores**
En todos los casos, los clusters se resuelven sistemáticamente hacia:

- **EMA (ema_60)**  
  → Indicador dominante en clusters grandes y estables (tendencia).
- **ROC (roc_60, roc_30, roc_20)**  
  → Capturan momentum direccional a distintas escalas.
- **Momentum (mom_10, mom_5)**  
  → Aparecen en gestation como señales complementarias.
- **ATR normalizado (atr_norm_14 / atr_norm_20)**  
  → Único representante consistente de volatilidad (cluster pequeño pero estable).

---

**3. Diferencias por ventana (no por target)**
- **Full day**: selección mínima (2–3 indicadores). Señal más diluida.
- **Gestation**: mayor riqueza informativa (6 indicadores). IC OOS alto y balanceado.
- **Execution**: señales más fuertes en valor absoluto (|IC_OOS| más alto), con clara dominancia de EMA + ROC.

> La **ventana temporal** explica la estructura de señal; el **target no**.

---

**4. Validación del criterio de selección**
- El uso de **max abs(IC_OOS)** como criterio principal funciona correctamente:
  - Selecciona indicadores con **mejor generalización OOS**.
  - Los desempates por **abs(IC_IS)** son pocos y coherentes.
- El pruning por clusters reduce decenas de features redundantes a **5–6 señales robustas por caso**.

---

**Conclusión operativa**
- ✔ El proceso de clustering + resolución es **estable, coherente y target-agnóstico**.
- ✔ Confirma que **ret sigue siendo el target preferido**, sin pérdida de información estructural.
- ✔ El set final de features es **compacto, interpretable y listo para modelado**.


#### **Helper para extraer la lista final desde RESULTS**

In [199]:
def extract_final_indicators(results: dict, *, target: str, window: str, horizon: int) -> list[str]:
    """
    Devuelve la lista de indicadores finales (keep=True)
    para un target / ventana / horizonte.
    """
    df = results[(target, window, horizon)]["resolved"]

    if df.empty:
        return []

    return (
        df.loc[df["keep"], "indicator"]
          .sort_values()
          .tolist()
    )

#### **Construir todas las listas (equivalente a lo que tenía antes)**

In [200]:
FINAL_TI = {}

for target in ("delta", "ret"):
    for window in ("full_day", "gestation", "execution"):
        for h in (60, 90):
            FINAL_TI[(target, window, h)] = extract_final_indicators(
                RESULTS,
                target=target,
                window=window,
                horizon=h,
            )

#### **Print en formato “clásico” (como el que usaba antes)**

In [201]:
for target in ("delta", "ret"):
    print(f"\n===== TARGET: {target.upper()} =====")

    print("TI full day finals H=60:", FINAL_TI[(target, "full_day", 60)])
    print("TI full day finals H=90:", FINAL_TI[(target, "full_day", 90)])

    print("TI gestation finals H=60:", FINAL_TI[(target, "gestation", 60)])
    print("TI gestation finals H=90:", FINAL_TI[(target, "gestation", 90)])

    print("TI execution finals H=60:", FINAL_TI[(target, "execution", 60)])
    print("TI execution finals H=90:", FINAL_TI[(target, "execution", 90)])



===== TARGET: DELTA =====
TI full day finals H=60: ['ema_60', 'roc_60']
TI full day finals H=90: ['atr_norm_20', 'ema_60', 'roc_60']
TI gestation finals H=60: ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
TI gestation finals H=90: ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
TI execution finals H=60: ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']
TI execution finals H=90: ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']

===== TARGET: RET =====
TI full day finals H=60: ['ema_60', 'roc_60']
TI full day finals H=90: ['atr_norm_20', 'ema_60', 'roc_60']
TI gestation finals H=60: ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
TI gestation finals H=90: ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
TI execution finals H=60: ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']
TI execution finals H=90: ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']


#### **Guardas indicadores técnicos en json**

In [202]:
# ============================================================
# Path de salida
# ============================================================
FINAL_TI_PATH = Path(os.environ.get("FINAL_TI_PATH", "data/targets/final_indicators.json"))
FINAL_TI_PATH = DRIVE_DIR / FINAL_TI_PATH

# ============================================================
# Reorganizar a un dict JSON-friendly
# ============================================================
final_indicators_json = {}

for (target, window, horizon), indicators in FINAL_TI.items():
    final_indicators_json.setdefault(target, {})
    final_indicators_json[target].setdefault(window, {})
    final_indicators_json[target][window][str(horizon)] = indicators

# ============================================================
# Guardar JSON
# ============================================================
FINAL_TI_PATH.parent.mkdir(parents=True, exist_ok=True)

with FINAL_TI_PATH.open("w", encoding="utf-8") as f:
    json.dump(final_indicators_json, f, indent=2, ensure_ascii=False)

print(f"[OK] Final indicators guardados en:\n{FINAL_TI_PATH}")

[OK] Final indicators guardados en:
/content/drive/MyDrive/neural_profit/data/targets/final_indicators.json


## **5.4. Correlación final**

Una vez reducido el conjunto de indicadores técnicos mediante el filtrado por fuerza de señal (|IC| ≥ 0.10) y la resolución por clusters, analizaremos la matriz de correlación entre los indicadores resultantes con el objetivo de evaluar posibles redundancias adicionales.

**Criterio final para el análisis de correlación entre indicadores técnicos**

En esta etapa, el criterio de selección deja de ser puramente estadístico y pasa a ser arquitectónico.  
El análisis de Information Coefficient (IC) y los clusters de correlación ya cumplieron su función principal: identificar señal y eliminar redundancia evidente.  
La correlación final no se utiliza para “volver a filtrar” indicadores, sino para ordenar su uso dentro del modelo.

**Principio rector**

> No todos los indicadores finales deben competir entre sí:  
> deben coexistir por rol funcional y por ventana horaria.

**Definición de roles**

Antes de analizar correlaciones, los indicadores se agrupan por el fenómeno de mercado que representan:

  - Estructura / tendencia  
    `ema_60`

  - Magnitud del movimiento  
    `roc_60`, `roc_20`, `roc_30`

  - Aceleración / impulso  
    `mom_5`, `mom_10`

  - Riesgo / volatilidad  
    `atr_norm_20`, `atr_norm_14`

La correlación solo debe penalizar redundancia dentro de un mismo rol, no entre roles distintos.


**Criterio de uso de la correlación**

A. Dentro de un mismo rol  

  La correlación se utiliza para reducir redundancia:

  - |ρ| ≥ 0.85 se considera redundancia fuerte.
  - Se conserva:
    - el indicador con mayor |IC_OOS|, o
    - el más estable entre ventanas, si los IC son similares.

  Ejemplo:
  - mom_5 y mom_10 suelen estar altamente correlacionados.
  - Se conserva uno solo por ventana.

B. Entre roles distintos

  La correlación no se utiliza como criterio de descarte, salvo casos extremos.

  Ejemplos:
  - ema_60 vs roc_60  
  - roc_60 vs momentum_5  

  Aunque puedan mostrar correlación, capturan fenómenos distintos (tendencia, magnitud, aceleración) y pueden ser explotados conjuntamente por modelos no lineales.

**Aplicación directa a los indicadores finales**

- Core estructural (invariable)

  Estos indicadores se conservan siempre:

  - `ema_60`
  - `roc_60`

  Son estables, robustos fuera de muestra, válidos en múltiples horizontes y ventanas.

- Jornada completa

  - H = 60: `ema_60`, `roc_60`
  - H = 90: `atr_norm_20`, `ema_60`, `roc_60`

  El indicador de volatilidad (`atr_norm_20`) cumple un rol distinto y no compite con EMA o ROC.

- Ventanas de gestación y expansión

  En estas ventanas se aplica correlación intra-rol:

  - Momentum:
    - conservar un único momentum corto (por ejemplo, mom_5 o mom_10).
  - ROC:
    - conservar roc_60 como base y, opcionalmente, un ROC de escala más corta.

  Ejemplo razonable por ventana:
  - ema_60
  - roc_60
  - roc_20 o roc_30
  - momentum_5

**Regla operativa final**

> La correlación final no se utiliza para eliminar señales válidas,  
> sino para evitar redundancia dentro del mismo rol funcional.  
> Los indicadores estructurales (ema_60, roc_60) se conservan siempre,  
> mientras que los indicadores dependientes de ventana se seleccionan maximizando diversidad funcional: tendencia, magnitud, aceleración y riesgo.

**Síntesis práctica**

- Core fijo: ema_60, roc_60  
- Por ventana:
  - un ROC corto,
  - un momentum corto,
  - opcionalmente un indicador de volatilidad.

A partir de aquí, la responsabilidad de combinar y ponderar estas señales se delega al modelo.


## **5.5. Tratamiento de familias en ventanas de gestación y expansión**

### **5.5.1. `pick_one_by_family`**

In [203]:
import pandas as pd

def pick_one_by_family(
    *,
    target: str,                  # "delta" o "ret" (solo para etiquetar)
    window_name: str,             # "full_day" / "gestation" / "execution"
    horizon: int,                 # 60 / 90
    finals_list: list[str],       # lista final post-clusters (por target/ventana/h)
    ic_table: pd.DataFrame,       # IC table del target+ventana (incluye target_prefix/target_col)
    corr_mean: pd.DataFrame,      # corr_mean del target+ventana+h
    families: dict[str, list[str]],
    corr_threshold: float = 0.85,
    score_col: str = "abs_IC_OOS",
    enforce_target_prefix: bool = True,  # valida que target_prefix coincida con `target`
) -> dict:
    """
    Resuelve redundancia intra-rol ("families") dentro del set final de indicadores.

    Ajuste importante:
    - ic_table puede traer columnas 'target_prefix' y 'target_col'.
    - Se filtra por (horizon) y, opcionalmente, por (target_prefix == target).

    Lógica:
    - Para cada familia (roc_short, mom_short, atr_norm, ...):
        * Si max |rho| entre candidatos >= corr_threshold => elige 1 por mayor score_col
        * Si no => mantiene todos
    """

    # -------------------------
    # 0) Validaciones mínimas
    # -------------------------
    req = {"indicator", "horizon"}
    if not req.issubset(ic_table.columns):
        raise ValueError(f"ic_table debe contener: {sorted(req)}")
    if not isinstance(corr_mean, pd.DataFrame) or corr_mean.empty:
        raise ValueError("corr_mean debe ser un DataFrame no vacío.")

    # -------------------------
    # 1) Filtrar IC por horizonte (y target_prefix si corresponde)
    # -------------------------
    ic_h = ic_table[ic_table["horizon"] == horizon].copy()

    if enforce_target_prefix and "target_prefix" in ic_h.columns:
        ic_h = ic_h[ic_h["target_prefix"].astype(str) == str(target)].copy()

    if ic_h.empty:
        raise ValueError(
            f"ic_table quedó vacío para horizon={horizon}"
            + (f" y target_prefix='{target}'" if (enforce_target_prefix and "target_prefix" in ic_table.columns) else "")
        )

    # Crear abs_IC_OOS si no está
    if score_col not in ic_h.columns:
        if "IC_OOS" in ic_h.columns:
            ic_h[score_col] = ic_h["IC_OOS"].abs()
        else:
            raise ValueError(f"ic_table debe contener '{score_col}' o 'IC_OOS'.")

    ic_h = ic_h.set_index("indicator")

    # -------------------------
    # 2) Alinear corr a finals_list (evita KeyError)
    # -------------------------
    finals = list(dict.fromkeys(finals_list))  # unique preservando orden
    corr = corr_mean.reindex(index=finals, columns=finals)

    finals_set = set(finals)
    selected = set(finals)
    decision_rows = []

    # -------------------------
    # 3) Resolver familia por familia
    # -------------------------
    for fam_name, fam_members in families.items():
        cand = [x for x in fam_members if x in finals_set]

        if len(cand) == 0:
            continue

        if len(cand) == 1:
            decision_rows.append({
                "target": target, "window": window_name, "horizon": horizon,
                "family": fam_name, "candidates": cand,
                "action": "keep_single", "kept": cand[0],
                "reason": "only one candidate present"
            })
            continue

        # 3a) Redundancia: max |rho| entre pares
        max_abs_rho = 0.0
        for i in range(len(cand)):
            for j in range(i + 1, len(cand)):
                rho = corr.at[cand[i], cand[j]]
                if pd.notna(rho):
                    max_abs_rho = max(max_abs_rho, abs(float(rho)))

        # No redundantes => mantener todos
        if max_abs_rho < corr_threshold:
            decision_rows.append({
                "target": target, "window": window_name, "horizon": horizon,
                "family": fam_name, "candidates": cand,
                "action": "keep_all_not_redundant", "kept": cand,
                "reason": f"max |rho|={max_abs_rho:.3f} < {corr_threshold}"
            })
            continue

        # 3b) Redundantes => elegir por mayor abs_IC_OOS (desempate alfabético)
        scored = []
        for ind in cand:
            score = float(ic_h.at[ind, score_col]) if (ind in ic_h.index and pd.notna(ic_h.at[ind, score_col])) else float("-inf")
            scored.append((ind, score))

        scored.sort(key=lambda t: (-t[1], t[0]))
        kept, kept_score = scored[0]

        for ind, _ in scored[1:]:
            selected.discard(ind)

        decision_rows.append({
            "target": target, "window": window_name, "horizon": horizon,
            "family": fam_name, "candidates": cand,
            "action": "select_one", "kept": kept,
            "reason": f"redundant (max |rho|={max_abs_rho:.3f} >= {corr_threshold}); kept by max {score_col}={kept_score:.6f}"
        })

    decisions = pd.DataFrame(decision_rows)
    return {"selected": sorted(selected), "decisions": decisions}


### **5.5.2. Aplicación `pick_one_by_family`**

In [204]:
# ============================================================
# USO PARA TODOS LOS CASOS (delta/ret × full_day/gestation/execution × 60/90)
# ============================================================

IC_BY_KEY = {
    ("delta", "full_day", 60): ic_60_full_day_delta,
    ("delta", "full_day", 90): ic_90_full_day_delta,
    ("delta", "gestation", 60): ic_60_gestation_delta,
    ("delta", "gestation", 90): ic_90_gestation_delta,
    ("delta", "execution", 60): ic_60_execution_delta,
    ("delta", "execution", 90): ic_90_execution_delta,

    ("ret", "full_day", 60): ic_60_full_day_ret,
    ("ret", "full_day", 90): ic_90_full_day_ret,
    ("ret", "gestation", 60): ic_60_gestation_ret,
    ("ret", "gestation", 90): ic_90_gestation_ret,
    ("ret", "execution", 60): ic_60_execution_ret,
    ("ret", "execution", 90): ic_90_execution_ret,
}

CORR_BY_KEY = {
    ("delta", "full_day", 60): corr_mean_delta_60_full_day,
    ("delta", "full_day", 90): corr_mean_delta_90_full_day,
    ("delta", "gestation", 60): corr_mean_delta_60_gestation,
    ("delta", "gestation", 90): corr_mean_delta_90_gestation,
    ("delta", "execution", 60): corr_mean_delta_60_execution,
    ("delta", "execution", 90): corr_mean_delta_90_execution,

    ("ret", "full_day", 60): corr_mean_ret_60_full_day,
    ("ret", "full_day", 90): corr_mean_ret_90_full_day,
    ("ret", "gestation", 60): corr_mean_ret_60_gestation,
    ("ret", "gestation", 90): corr_mean_ret_90_gestation,
    ("ret", "execution", 60): corr_mean_ret_60_execution,
    ("ret", "execution", 90): corr_mean_ret_90_execution,
}

FAMILIES = {
    "roc_short": ["roc_20", "roc_30"],
    "mom_short": ["mom_5", "mom_10"],
    "atr_norm": ["atr_norm_14", "atr_norm_20"],
}

FINAL_TI_FAMILY = {}
DECISIONS_FAMILY = {}

missing = [k for k in FINAL_TI.keys() if k not in IC_BY_KEY or k not in CORR_BY_KEY]
if missing:
    raise KeyError(f"Faltan claves en IC_BY_KEY o CORR_BY_KEY para: {missing}")

for (target, window, h), finals_list in FINAL_TI.items():
    key = (target, window, int(h))

    out = pick_one_by_family(
        target=target,
        window_name=window,
        horizon=int(h),
        finals_list=finals_list,
        ic_table=IC_BY_KEY[key],
        corr_mean=CORR_BY_KEY[key],
        families=FAMILIES,
        corr_threshold=0.85,
        score_col="abs_IC_OOS",
        enforce_target_prefix=True,
    )

    FINAL_TI_FAMILY[key] = out["selected"]
    DECISIONS_FAMILY[key] = out["decisions"]

### **5.5.3. Observación**

In [205]:
# Ejemplo: ver decisiones de un caso
# display(DECISIONS_FAMILY[("ret","gestation",60)])
for k in sorted(FINAL_TI_FAMILY.keys()):
    print(k, "->", FINAL_TI_FAMILY[k])

('delta', 'execution', 60) -> ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']
('delta', 'execution', 90) -> ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']
('delta', 'full_day', 60) -> ['ema_60', 'roc_60']
('delta', 'full_day', 90) -> ['atr_norm_20', 'ema_60', 'roc_60']
('delta', 'gestation', 60) -> ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
('delta', 'gestation', 90) -> ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
('ret', 'execution', 60) -> ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']
('ret', 'execution', 90) -> ['atr_norm_14', 'ema_60', 'roc_20', 'roc_30', 'roc_60']
('ret', 'full_day', 60) -> ['ema_60', 'roc_60']
('ret', 'full_day', 90) -> ['atr_norm_20', 'ema_60', 'roc_60']
('ret', 'gestation', 60) -> ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
('ret', 'gestation', 90) -> ['ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']


El resultado obtenido es coherente con la lógica actual de la función: solo se elimina un indicador **dentro de una misma familia** cuando se detecta **redundancia fuerte** (máx. |ρ| ≥ 0.85) entre los candidatos que **coexisten en `finals_list`**.

En tus listas finales, las familias presentan los siguientes pares:

- **roc_short**: `roc_20` vs `roc_30`  
- **mom_short**: `mom_5` vs `mom_10`  
- **atr_norm**: `atr_norm_14` vs `atr_norm_20` (pero no coexisten en la misma lista)

Por lo tanto, si en las matrices `corr_mean_*` los pares `roc_20–roc_30` y `mom_5–mom_10` quedan con |ρ| < 0.85 (o bien `NaN`, según cómo se construyó la matriz), **no se aplica ningún filtrado**.

A continuación se deja un bloque para **diagnosticar exactamente por qué no se filtró** (mostrando |ρ| y `abs_IC_OOS` por familia, en cada caso), y una **variante opcional más estricta** por si se desea **forzar un único indicador por familia**, aun cuando |ρ| no alcance el umbral de 0.85.


### **5.5.4.`audit_family_redundancy`**


In [206]:
import pandas as pd

def audit_family_redundancy(
    *,
    FINAL_TI: dict[tuple[str, str, int], list[str]],
    IC_BY_KEY: dict[tuple[str, str, int], pd.DataFrame],
    CORR_BY_KEY: dict[tuple[str, str, int], pd.DataFrame],
    families: dict[str, list[str]],
    corr_threshold: float = 0.85,
    score_col: str = "abs_IC_OOS",
) -> pd.DataFrame:
    """
    Audita, por (target, window, horizon) y por familia:
      - qué candidatos están presentes
      - cuál es el max |rho| entre pares
      - los scores (abs_IC_OOS) de cada candidato
      - si (según threshold) debería filtrar o no
    """
    rows = []

    for (target, window, h), finals in FINAL_TI.items():
        key = (target, window, int(h))
        ic = IC_BY_KEY[key].copy()
        corr = CORR_BY_KEY[key].copy()

        # filtrar IC por horizonte y target_prefix si existe
        ic = ic[ic["horizon"] == int(h)].copy()
        if "target_prefix" in ic.columns:
            ic = ic[ic["target_prefix"].astype(str) == str(target)].copy()

        if score_col not in ic.columns:
            ic[score_col] = ic["IC_OOS"].abs()

        ic = ic.set_index("indicator")

        # corr alineada a finals
        corr = corr.reindex(index=finals, columns=finals)

        finals_set = set(finals)

        for fam, members in families.items():
            cand = [m for m in members if m in finals_set]

            if len(cand) <= 1:
                rows.append({
                    "target": target, "window": window, "horizon": int(h), "family": fam,
                    "candidates": cand,
                    "max_abs_rho": None if len(cand) <= 1 else 0.0,
                    "is_redundant": False,
                    "scores": {c: float(ic.at[c, score_col]) if c in ic.index else None for c in cand},
                })
                continue

            max_abs_rho = 0.0
            for i in range(len(cand)):
                for j in range(i + 1, len(cand)):
                    rho = corr.at[cand[i], cand[j]]
                    if pd.notna(rho):
                        max_abs_rho = max(max_abs_rho, abs(float(rho)))

            rows.append({
                "target": target, "window": window, "horizon": int(h), "family": fam,
                "candidates": cand,
                "max_abs_rho": max_abs_rho,
                "is_redundant": (max_abs_rho >= corr_threshold),
                "scores": {c: float(ic.at[c, score_col]) if c in ic.index else None for c in cand},
            })

    return pd.DataFrame(rows).sort_values(["target", "window", "horizon", "family"]).reset_index(drop=True)


# Ejecutar auditoría
audit = audit_family_redundancy(
    FINAL_TI=FINAL_TI,
    IC_BY_KEY=IC_BY_KEY,
    CORR_BY_KEY=CORR_BY_KEY,
    families=FAMILIES,
    corr_threshold=0.85,
    score_col="abs_IC_OOS",
)




### **5.5.5. Aplicación `audit_family_redundancy`**

In [207]:
display(audit)

,target,window,horizon,family,candidates,max_abs_rho,is_redundant,scores
0,delta,execution,60,atr_norm,[atr_norm_14],NaN,False,{'atr_norm_14': 0.20031895729706078}
1,delta,execution,60,mom_short,[],NaN,False,{}
2,delta,execution,60,roc_short,"[roc_20, roc_30]",0.721427,False,"{'roc_20': 0.3856659699417222, 'roc_30': 0.457..."
3,delta,execution,90,atr_norm,[atr_norm_14],NaN,False,{'atr_norm_14': 0.2146224456446493}
4,delta,execution,90,mom_short,[],NaN,False,{}
5,delta,execution,90,roc_short,"[roc_20, roc_30]",0.721427,False,"{'roc_20': 0.41925472892136145, 'roc_30': 0.49..."
6,delta,full_day,60,atr_norm,[],NaN,False,{}
7,delta,full_day,60,mom_short,[],NaN,False,{}
8,delta,full_day,60,roc_short,[],NaN,False,{}
9,delta,full_day,90,atr_norm,[atr_norm_20],NaN,False,{'atr_norm_20': 0.1078302312542387}


In [208]:
# Atajo: ver solo donde hay >1 candidato y NO se filtró por correlación
display(
    audit[
        audit["candidates"].apply(lambda x: isinstance(x, list) and len(x) > 1)
        & (~audit["is_redundant"])
    ][["target","window","horizon","family","candidates","max_abs_rho","scores"]]
)

,target,window,horizon,family,candidates,max_abs_rho,scores
2,delta,execution,60,roc_short,"[roc_20, roc_30]",0.721427,"{'roc_20': 0.3856659699417222, 'roc_30': 0.457..."
5,delta,execution,90,roc_short,"[roc_20, roc_30]",0.721427,"{'roc_20': 0.41925472892136145, 'roc_30': 0.49..."
13,delta,gestation,60,mom_short,"[mom_5, mom_10]",0.627276,"{'mom_5': 0.1302455328088275, 'mom_10': 0.1728..."
14,delta,gestation,60,roc_short,"[roc_20, roc_30]",0.638368,"{'roc_20': 0.21027343332030146, 'roc_30': 0.24..."
16,delta,gestation,90,mom_short,"[mom_5, mom_10]",0.627276,"{'mom_5': 0.14299069575823695, 'mom_10': 0.166..."
17,delta,gestation,90,roc_short,"[roc_20, roc_30]",0.638368,"{'roc_20': 0.18505148525824008, 'roc_30': 0.19..."
20,ret,execution,60,roc_short,"[roc_20, roc_30]",0.721427,"{'roc_20': 0.3856745981283724, 'roc_30': 0.457..."
23,ret,execution,90,roc_short,"[roc_20, roc_30]",0.721427,"{'roc_20': 0.41927061310795694, 'roc_30': 0.49..."
31,ret,gestation,60,mom_short,"[mom_5, mom_10]",0.627276,"{'mom_5': 0.13028968746692673, 'mom_10': 0.172..."
32,ret,gestation,60,roc_short,"[roc_20, roc_30]",0.638368,"{'roc_20': 0.2104416611952213, 'roc_30': 0.241..."


### **5.5.6. Observaciones y conclusiones del análisis de correlación intra-familia**

**1. Ausencia de redundancia estructural**

En todos los casos analizados (targets **delta** y **ret**, ventanas **full_day**, **gestation** y **execution**, horizontes **H=60** y **H=90**), **no se detectaron redundancias fuertes** dentro de las familias evaluadas:
- Los valores máximos de correlación intra-familia se ubican en el rango **|ρ| ≈ 0.63 - 0.72**, consistentemente por debajo del umbral operativo **|ρ| ≥ 0.85**.
- Por lo tanto, **ninguna familia fue marcada como redundante** bajo el criterio definido.

**2. Familias con múltiples candidatos**

Los únicos casos con más de un candidato activo corresponden a:
- **roc_short**: `[roc_20, roc_30]`
- **mom_short** (solo en ventanas de gestación): `[mom_5, mom_10]`

En ambos casos:
- La correlación es **moderada**, no extrema.
- Los indicadores presentan **valores de abs_IC_OOS distintos y relevantes**, lo que indica aporte diferencial de señal.

**3. Interpretación funcional**

- **ROC corto**: captura magnitud del movimiento en **escalas temporales distintas**; mantener `roc_20` y `roc_30` preserva información multi-escala.
- **Momentum corto**: `mom_5` y `mom_10` reflejan **dinámicas de aceleración complementarias**, especialmente relevantes en ventanas de gestación.
- **ATR normalizado** aparece como indicador único por familia cuando está presente, sin conflicto ni redundancia.

**4. Consistencia entre targets**

El patrón observado es **prácticamente idéntico para delta y ret**, lo que confirma que:
- La estructura de correlaciones depende del **feature space**, no del target.
- El criterio de no forzar selección adicional es robusto y target-agnóstico.

**5. Conclusión operativa**

- **Forzar la selección de un único indicador por familia cuando |ρ| < 0.85 implicaría pérdida de información**.
- El procedimiento aplicado —conservar múltiples indicadores cuando la correlación es moderada— es **coherente con el objetivo de maximizar diversidad funcional** tras un filtrado previo por IC OOS y clusters.
- El resultado final confirma que el pipeline **evita over-pruning** y mantiene señales complementarias potencialmente explotables por modelos no lineales.

**Conclusión final:**  

La etapa de correlación final cumple un rol diagnóstico y organizativo, no de descarte agresivo. Los resultados validan el criterio adoptado y refuerzan la solidez del conjunto final de indicadores.


## **5.6. Decisión de consolidación final**

### **5.6.1. Estrategia general de entrenamiento**

A partir del análisis conjunto de **Information Coefficient (IC)**, **correlación** y **clusters de redundancia**, se concluye que **no es necesario entrenar modelos separados por ventana horaria**.

La evidencia empírica respalda entrenar **un único modelo** que incorpore:

- **Un core de features estructurales**, activo durante toda la jornada.
- **Features adicionales activadas de forma condicional** en ventanas de gestación y expansión.

Este enfoque permite:

- Utilizar el **máximo volumen de datos disponible**.
- Evitar la **fragmentación artificial del dataset**.
- Reducir el **riesgo de sobreajuste**.
- Respetar que ciertas señales son informativas **solo en ventanas específicas**, sin imponer reglas duras.

El modelo aprende implícitamente **cuándo una feature aporta señal y cuándo no**, delegando la ponderación al proceso de entrenamiento.

### **5.6.2. Consolidación del listado final de features**

Bajo este criterio, el conjunto final queda definido por **target, horizonte y ventana**, manteniendo coherencia entre *delta* y *ret*.

**Jornada completa (full_day)**

- **H = 60**  
  `ema_60`, `roc_60`

- **H = 90**  
  `atr_norm_20`, `ema_60`, `roc_60`

**Ventana de gestación**

- **H = 60 / 90**  
  `ema_60`, `roc_60`, `mom_10`, `mom_5`, `roc_20`, `roc_30`

**Ventana de expansión / ejecución**

- **H = 60 / 90**  
  `atr_norm_14`, `ema_60`, `roc_20`, `roc_30`, `roc_60`

En notación compacta, la consolidación adoptada es:

- (`delta` / `ret`, `full_day`, 60) → [`ema_60`, `roc_60`]
- (`delta` / `ret`, `full_day`, 90) → [`ema_60`, `roc_60`, `atr_norm_20`]
- (`delta` / `ret`, `gestation`, 60/90) → [`ema_60`, `roc_60`, `roc_30`,  `roc_20`, `mom_10`, `mom_5`]
- (`delta` / `ret`, `execution`, 60/90) → [`ema_60`, `roc_60`, `roc_30`,  `roc_20`, `atr_norm_14`]

### **5.6.3. Evaluación del criterio adoptado**


El esquema es consistente con todos los resultados obtenidos:

- `ema_60` y `roc_60` actúan como **factores estructurales**, robustos en todas las ventanas.
- `atr_norm_20` y `atr_norm_14` aportan información de **riesgo y volatilidad**, más relevante en horizontes largos o fases activas.
- Los indicadores `momentum_*` y `roc_*` de corto plazo muestran **dependencia horaria clara**, especialmente en gestación y expansión.
- No se observa **redundancia fuerte intra-rol** que justifique eliminar estas señales dentro de las ventanas definidas.

### **5.6.4. Conclusión final**

Es metodológicamente adecuado entrenar **un único modelo** con:

- Un **core estructural** activo durante toda la jornada.
- Features adicionales **condicionales por ventana horaria**.

La consolidación final propuesta es coherente con el análisis empírico realizado y representa un **equilibrio correcto entre robustez global y sensibilidad temporal**, dejando al modelo la tarea de combinar y ponderar las señales disponibles.


### **5.6.5. Código para Consolidación**

In [209]:
# ============================================================
# Consolidación final de indicadores técnicos (desde FINAL_TI)
# ============================================================
# FINAL_TI tiene la forma:
#   (target, window, horizon) -> [lista de indicadores]
#
# Objetivo:
#   - Unificar TODOS los indicadores seleccionados
#   - Sin duplicados
#   - Independiente de target / ventana / horizonte
# ============================================================

tech_indicators_finals = sorted(
    {
        indicator
        for indicators_list in FINAL_TI.values()
        for indicator in indicators_list
    }
)

# Mostrar resultado final
print("tech_indicators_finals:")
print(tech_indicators_finals)

tech_indicators_finals:
['atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']


**Observación importante**

- Esta lista representa el universo final de features que el modelo puede usar.

- La activación por ventana / horizonte ya quedó resuelta aguas arriba (lógica de ventanas, masks temporales, etc.).

- Esto es exactamente lo que necesitás para:
  - stage de feature engineering final,
  - definición de inputs del modelo,
  - documentación del set de features definitivo.

# **6. Análisis de coherencia direccional IS-OOS**

Una vez consolidado el conjunto final de indicadores técnicos:

```
tech_indicators_finals:
  [
  'atr_norm_14',
  'atr_norm_20',
  'ema_60',
  'mom_10',
  'mom_5',
  'roc_20',
  'roc_30',
  'roc_60'
  ]
```

Y priorizada su selección en función del desempeño fuera de muestra (OOS), se introduce un análisis adicional de carácter cualitativo: la coherencia direccional entre los períodos **in-sample (IS)** y **out-of-sample (OOS)**.

El objetivo de este análisis no es seleccionar indicadores, sino **validar su estabilidad conceptual**, verificando si cada indicador mantiene la misma dirección de relación con el target al pasar de IS a OOS. De este modo, se evalúa si la señal aprendida durante el entrenamiento conserva coherencia cuando se expone a datos no vistos.

Este enfoque permite detectar indicadores cuya señal OOS, aun siendo significativa en magnitud, presenta una **inversión de signo** respecto a IS, lo cual constituye una alerta de posible sobreajuste o inestabilidad estructural.

El criterio de coherencia se basa exclusivamente en la comparación del signo de `IC_IS` y `IC_OOS`, utilizando métricas OOS como referencia principal y dejando la diferencia `delta_oos_minus_is` como herramienta de diagnóstico complementaria.


## **6.1. Coherencia direccional IS vs OOS (adaptada a ic_table actuales)**




In [210]:
import numpy as np
import pandas as pd

# ============================================================
# 1) Coherencia direccional IS vs OOS (adaptada a ic_table actuales)
# ============================================================
def check_directional_coherence(
    df_selected: pd.DataFrame,
    *,
    is_col: str = "IC_IS",
    oos_col: str = "IC_OOS",
    abs_oos_col: str = "abs_IC_OOS",
) -> pd.DataFrame:
    """
    Verifica coherencia direccional entre IC_IS e IC_OOS.

    coherent_is_oos = True si:
      - sign(IC_IS) == sign(IC_OOS)
      - ambos distintos de 0
      - ninguno NaN

    Soporta tablas con columnas adicionales como:
      - target_prefix, target_col
      - target_oos_minus_is (si existe, se conserva; si no, se calcula)
    """
    out = df_selected.copy()

    # -------------------------
    # Validaciones mínimas
    # -------------------------
    needed = {"indicator", "horizon", is_col, oos_col}
    missing = [c for c in needed if c not in out.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en df_selected: {missing}")

    # -------------------------
    # abs_IC_OOS (si no existe)
    # -------------------------
    if abs_oos_col not in out.columns:
        out[abs_oos_col] = out[oos_col].abs()

    # -------------------------
    # Signos
    # -------------------------
    out["sign_IC_IS"] = np.sign(out[is_col])
    out["sign_IC_OOS"] = np.sign(out[oos_col])

    # -------------------------
    # Coherencia direccional IS vs OOS
    # -------------------------
    out["coherent_is_oos"] = (
        out[is_col].notna()
        & out[oos_col].notna()
        & (out["sign_IC_IS"] != 0)
        & (out["sign_IC_OOS"] != 0)
        & (out["sign_IC_IS"] == out["sign_IC_OOS"])
    )

    # Flag de flip de signo
    out["sign_flip_is_oos"] = (
        out[is_col].notna()
        & out[oos_col].notna()
        & (out["sign_IC_IS"] != 0)
        & (out["sign_IC_OOS"] != 0)
        & (out["sign_IC_IS"] != out["sign_IC_OOS"])
    )

    # Magnitudes y gaps
    out["abs_IC_IS"] = out[is_col].abs()
    out["abs_IC_OOS"] = out[oos_col].abs()
    out["abs_gap_oos_minus_is"] = out["abs_IC_OOS"] - out["abs_IC_IS"]

    # Diferencia OOS-IS (respeta el nombre nuevo si existe)
    if "target_oos_minus_is" in out.columns:
        pass
    else:
        out["target_oos_minus_is"] = out[oos_col] - out[is_col]

    # Orden: primero incoherentes, luego por fuerza OOS
    out = out.sort_values(["coherent_is_oos", abs_oos_col], ascending=[True, False])

    # Columnas recomendadas (devolver solo las que existan)
    cols = [
        "target_prefix", "target_col",
        "indicator", "horizon",
        is_col, oos_col, "target_oos_minus_is",
        "sign_IC_IS", "sign_IC_OOS",
        "coherent_is_oos", "sign_flip_is_oos",
        "abs_IC_IS", "abs_IC_OOS", "abs_gap_oos_minus_is",
        "n_pairs_IS", "n_pairs_OOS", "note",
    ]
    cols = [c for c in cols if c in out.columns]
    return out[cols]


# ============================================================
# 2) Helper: correr coherencia para una key del dict ic_tables
# ============================================================
def run_coherence_for_key(
    *,
    key: str,
    ic_tables: dict[str, pd.DataFrame],
    indicators: list[str],
) -> pd.DataFrame:
    """
    key: ej 'full_day_delta_60'
    indicators: lista a evaluar (p.ej tech_indicators_finals)
    """
    if key not in ic_tables:
        raise KeyError(f"Key '{key}' no existe en ic_tables. Keys: {sorted(ic_tables.keys())}")

    df = ic_tables[key].copy()

    # Filtrar solo indicadores de interés
    df_sel = df[df["indicator"].isin(indicators)].copy()

    # Si no hay solapamiento, devolver vacío
    if df_sel.empty:
        return pd.DataFrame()

    return check_directional_coherence(df_sel)

## **6.2. Correr coherencia para todas las tablas del diccionario**

In [211]:
coherence_all = {}

for key in sorted(ic_tables.keys()):
    coherence_all[key] = run_coherence_for_key(
        key=key,
        ic_tables=ic_tables,
        indicators=tech_indicators_finals,   # <-- o tu lista final consolidada
    )

## **6.3. Mostrar (solo no-vacíos) de forma prolija**

In [212]:
# ------------------------------------------------------------
# 3) Mostrar (solo no-vacíos) de forma prolija
# ------------------------------------------------------------
for key in sorted(coherence_all.keys()):
    dfc = coherence_all[key]
    if dfc is None or dfc.empty:
        continue

    print(f"\n=== COHERENCE | {key} (n={len(dfc)}) ===")
    display(dfc)


=== COHERENCE | execution_delta_60 (n=8) ===


,target_prefix,target_col,indicator,horizon,IC_IS,IC_OOS,target_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
0,delta,delta_60,roc_60,60,-0.564986,-0.579559,-0.014573,-1.0,-1.0,True,False,0.564986,0.579559,0.014573,64530,52740,
1,delta,delta_60,ema_60,60,-0.538328,-0.574979,-0.036651,-1.0,-1.0,True,False,0.538328,0.574979,0.036651,64530,52740,
3,delta,delta_60,roc_30,60,-0.416087,-0.457481,-0.041393,-1.0,-1.0,True,False,0.416087,0.457481,0.041393,64530,52740,
10,delta,delta_60,roc_20,60,-0.355251,-0.385666,-0.030415,-1.0,-1.0,True,False,0.355251,0.385666,0.030415,64530,52740,
25,delta,delta_60,mom_10,60,-0.280940,-0.302256,-0.021316,-1.0,-1.0,True,False,0.280940,0.302256,0.021316,64530,52740,
28,delta,delta_60,mom_5,60,-0.211075,-0.231262,-0.020188,-1.0,-1.0,True,False,0.211075,0.231262,0.020188,64530,52740,
30,delta,delta_60,atr_norm_14,60,0.180507,0.200319,0.019812,1.0,1.0,True,False,0.180507,0.200319,0.019812,64530,52740,
32,delta,delta_60,atr_norm_20,60,0.168996,0.193373,0.024377,1.0,1.0,True,False,0.168996,0.193373,0.024377,64530,52740,



=== COHERENCE | execution_delta_90 (n=8) ===


,target_prefix,target_col,indicator,horizon,IC_IS,IC_OOS,target_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
42,delta,delta_90,ema_60,90,-0.581418,-0.614023,-0.032605,-1.0,-1.0,True,False,0.581418,0.614023,0.032605,64530,52740,
43,delta,delta_90,roc_60,90,-0.602079,-0.610526,-0.008447,-1.0,-1.0,True,False,0.602079,0.610526,0.008447,64530,52740,
45,delta,delta_90,roc_30,90,-0.450586,-0.495278,-0.044692,-1.0,-1.0,True,False,0.450586,0.495278,0.044692,64530,52740,
52,delta,delta_90,roc_20,90,-0.392389,-0.419255,-0.026866,-1.0,-1.0,True,False,0.392389,0.419255,0.026866,64530,52740,
67,delta,delta_90,mom_10,90,-0.309592,-0.325200,-0.015608,-1.0,-1.0,True,False,0.309592,0.325200,0.015608,64530,52740,
70,delta,delta_90,mom_5,90,-0.234252,-0.247898,-0.013646,-1.0,-1.0,True,False,0.234252,0.247898,0.013646,64530,52740,
72,delta,delta_90,atr_norm_14,90,0.184005,0.214622,0.030618,1.0,1.0,True,False,0.184005,0.214622,0.030618,64530,52740,
74,delta,delta_90,atr_norm_20,90,0.169253,0.203899,0.034646,1.0,1.0,True,False,0.169253,0.203899,0.034646,64530,52740,



=== COHERENCE | execution_ret_60 (n=8) ===


...

,target_prefix,target_col,indicator,horizon,IC_IS,IC_OOS,target_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
42,delta,delta_90,roc_60,90,-0.266323,-0.284789,-0.018466,-1.0,-1.0,True,False,0.266323,0.284789,0.018466,43020,35160,
43,delta,delta_90,ema_60,90,-0.295886,-0.274460,0.021426,-1.0,-1.0,True,False,0.295886,0.274460,-0.021426,43020,35160,
57,delta,delta_90,roc_30,90,-0.229487,-0.197892,0.031595,-1.0,-1.0,True,False,0.229487,0.197892,-0.031595,43020,35160,
61,delta,delta_90,roc_20,90,-0.215870,-0.185051,0.030819,-1.0,-1.0,True,False,0.215870,0.185051,-0.030819,43020,35160,
67,delta,delta_90,mom_10,90,-0.189081,-0.166095,0.022986,-1.0,-1.0,True,False,0.189081,0.166095,-0.022986,43020,35160,
70,delta,delta_90,mom_5,90,-0.155806,-0.142991,0.012815,-1.0,-1.0,True,False,0.155806,0.142991,-0.012815,43020,35160,
75,delta,delta_90,atr_norm_20,90,0.073847,0.024497,-0.049350,1.0,1.0,True,False,0.073847,0.024497,-0.049350,43020,35160,
77,delta,delta_90,atr_norm_14,90,0.064237,0.019033,-0.045204,1.0,1.0,True,False,0.064237,0.019033,-0.045204,43020,35160,



=== COHERENCE | gestation_ret_60 (n=8) ===


,target_prefix,target_col,indicator,horizon,IC_IS,IC_OOS,target_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
0,ret,ret_60,ema_60,60,-0.247058,-0.293809,-0.046751,-1.0,-1.0,True,False,0.247058,0.293809,0.046751,43020,35160,
1,ret,ret_60,roc_60,60,-0.221354,-0.287290,-0.065936,-1.0,-1.0,True,False,0.221354,0.287290,0.065936,43020,35160,
4,ret,ret_60,roc_30,60,-0.189454,-0.241835,-0.052381,-1.0,-1.0,True,False,0.189454,0.241835,0.052381,43020,35160,
13,ret,ret_60,roc_20,60,-0.178751,-0.210442,-0.031690,-1.0,-1.0,True,False,0.178751,0.210442,0.031690,43020,35160,
25,ret,ret_60,mom_10,60,-0.163331,-0.172891,-0.009561,-1.0,-1.0,True,False,0.163331,0.172891,0.009561,43020,35160,
28,ret,ret_60,mom_5,60,-0.130002,-0.130290,-0.000288,-1.0,-1.0,True,False,0.130002,0.130290,0.000288,43020,35160,
33,ret,ret_60,atr_norm_20,60,0.037171,0.018296,-0.018875,1.0,1.0,True,False,0.037171,0.018296,-0.018875,43020,35160,
34,ret,ret_60,atr_norm_14,60,0.029470,0.015177,-0.014293,1.0,1.0,True,False,0.029470,0.015177,-0.014293,43020,35160,



=== COHERENCE | gestation_ret_90 (n=8) ===


,target_prefix,target_col,indicator,horizon,IC_IS,IC_OOS,target_oos_minus_is,sign_IC_IS,sign_IC_OOS,coherent_is_oos,sign_flip_is_oos,abs_IC_IS,abs_IC_OOS,abs_gap_oos_minus_is,n_pairs_IS,n_pairs_OOS,note
42,ret,ret_90,roc_60,90,-0.266423,-0.284787,-0.018364,-1.0,-1.0,True,False,0.266423,0.284787,0.018364,43020,35160,
43,ret,ret_90,ema_60,90,-0.296160,-0.274527,0.021633,-1.0,-1.0,True,False,0.296160,0.274527,-0.021633,43020,35160,
57,ret,ret_90,roc_30,90,-0.229710,-0.197999,0.031711,-1.0,-1.0,True,False,0.229710,0.197999,-0.031711,43020,35160,
61,ret,ret_90,roc_20,90,-0.216050,-0.185151,0.030899,-1.0,-1.0,True,False,0.216050,0.185151,-0.030899,43020,35160,
67,ret,ret_90,mom_10,90,-0.189195,-0.166238,0.022957,-1.0,-1.0,True,False,0.189195,0.166238,-0.022957,43020,35160,
70,ret,ret_90,mom_5,90,-0.155953,-0.143128,0.012825,-1.0,-1.0,True,False,0.155953,0.143128,-0.012825,43020,35160,
75,ret,ret_90,atr_norm_20,90,0.074069,0.024468,-0.049601,1.0,1.0,True,False,0.074069,0.024468,-0.049601,43020,35160,
77,ret,ret_90,atr_norm_14,90,0.064495,0.019086,-0.045409,1.0,1.0,True,False,0.064495,0.019086,-0.045409,43020,35160,


## **6.4. Conclusiones (en base a coherencia IS vs OOS y magnitudes)**


- **Coherencia perfecta:**

  En todas las combinaciones (target delta/ret, ventanas execution/full_day/gestation, horizontes 60/90) los 8 indicadores muestran `coherent_is_oos = True` y `sign_flip_is_oos = False`.

  ⇒ No hay “señales inestables por cambio de signo”: el sentido de la relación se mantiene fuera de muestra.

- **Execution es la ventana más fuerte (por lejos):**

  En execution los IC OOS son muy altos para tendencia/magnitud/impulso:
  `roc_60` y `ema_60` ~0.57–0.61 (en abs), `roc_30` ~0.46–0.50, `roc_20` ~0.39–0.42.

  ⇒ Es la evidencia más sólida de que estas señales generalizan y son muy informativas durante esa ventana.

- **Full day es señal moderada y “estable”:**

  `ema_60` y `roc_60` quedan cerca de ~0.18–0.24 abs OOS (según H), mientras que `roc_20`/`roc_30` y `mom_*` son más bajos.
  Además, varios tienen `abs_gap_oos_minus_is` levemente negativo (OOS algo menor).

  ⇒ Señal real pero más débil, y con la caída típica al pasar a OOS.

- **Gestation refuerza tendencia/magnitud, pero volatilidad aporta poco:**

  En gestation la señal de `ema_60` y `roc_60` sube en OOS (gap positivo claro, por ejemplo +0.04 a +0.066 en abs).
  En cambio `atr_norm_14/20` muestran IC OOS muy bajos (≈0.015–0.025) y gap negativo fuerte.

  ⇒ En esa ventana, volatilidad no es un driver predictivo relevante, mientras que tendencia/magnitud sí.

- **Delta vs Ret:**

  Prácticamente idénticos en estas tablas (mismos signos, valores casi iguales).

  ⇒ La diferencia entre targets no aparece en coherencia direccional; esto es consistente con tu conclusión previa: estas tablas validan estabilidad de señales, no “mejor target”.

Si querés una regla práctica inmediata: mantener core `ema_60` + `roc_60` siempre, y tratar `atr_norm_*` como útil en execution, pero prescindible en gestation (por IC OOS bajo).


In [213]:
tech_indicators_finals

['atr_norm_14',
 'atr_norm_20',
 'ema_60',
 'mom_10',
 'mom_5',
 'roc_20',
 'roc_30',
 'roc_60']

# **7. Análisis de OHLCV**

Las variables OHLCV (`open`, `high`, `low`, `close`, `volume`) representan el estado instantáneo del mercado y constituyen la materia prima a partir de la cual se construyen los indicadores técnicos.

Aunque no están diseñadas explícitamente como señales anticipatorias, su inclusión directa en el modelo puede:

- aportar información contextual relevante,
- introducir redundancia respecto a los indicadores derivados,
- o, en algunos casos, mostrar capacidad predictiva directa.

Por este motivo, se realiza un análisis específico de Information Coefficient (IC) y correlación para las variables OHLCV, con el objetivo de evaluar su contribución real y justificar su inclusión en el modelo.

## **7.1. IC de variables OHLCV vs targets**


Se evalúa la relación entre OHLCV y los targets definidos (`ret_h` y `delta_h`), utilizando el mismo criterio metodológico aplicado a los indicadores técnicos.


In [214]:
ohlcv_cols = ["open", "high", "low", "close", "volume"]

In [215]:
# --- Full Day ---
ic_table_full_day_ohlcv_ret = load_or_compute_ic_table(
    name="ic_table_full_day_ohlcv_ret",
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

# --- Gestation window ---
ic_table_ohlcv_gestation_ret = load_or_compute_ic_table(
    name="ic_table_gestation_ohlcv_ret",
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

# --- Execution window ---
ic_table_ohlcv_execution_ret = load_or_compute_ic_table(
    name="ic_table_execution_ohlcv_ret",
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

print("OK -> ic_table_full_day_ohlcv_ret:", ic_table_full_day_ohlcv_ret.shape)
print("OK -> ic_table_gestation_ohlcv_ret:", ic_table_ohlcv_gestation_ret.shape)
print("OK -> ic_table_execution_ohlcv_ret:", ic_table_ohlcv_execution_ret.shape)

[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_full_day_ohlcv_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_gestation_ohlcv_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_execution_ohlcv_ret.parquet
OK -> ic_table_full_day_ohlcv_ret: (10, 11)
OK -> ic_table_gestation_ohlcv_ret: (10, 11)
OK -> ic_table_execution_ohlcv_ret: (10, 11)


In [216]:
# --- Full Day ---
ic_table_full_day_ohlcv_delta = load_or_compute_ic_table(
    name="ic_table_full_day_ohlcv_delta",
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

# --- Gestation window ---
ic_table_ohlcv_gestation_delta = load_or_compute_ic_table(
    name="ic_table_gestation_ohlcv_delta",
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

# --- Execution window ---
ic_table_ohlcv_execution_delta = load_or_compute_ic_table(
    name="ic_table_execution_ohlcv_delta",
    df=mnq_intraday_with_indicators,
    indicator_columns=ohlcv_cols,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

print("OK -> ic_table_full_day_ohlcv_delta:", ic_table_full_day_ohlcv_delta.shape)
print("OK -> ic_table_gestation_ohlcv_delta:", ic_table_ohlcv_gestation_delta.shape)
print("OK -> ic_table_execution_ohlcv_delta:", ic_table_ohlcv_execution_delta.shape)

[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_full_day_ohlcv_delta.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_gestation_ohlcv_delta.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_execution_ohlcv_delta.parquet
OK -> ic_table_full_day_ohlcv_delta: (10, 11)
OK -> ic_table_gestation_ohlcv_delta: (10, 11)
OK -> ic_table_execution_ohlcv_delta: (10, 11)


## **7.2. Resultados**


In [217]:
from IPython.display import display

# ============================================================
# Mostrar tablas IC OHLCV (ret y delta)
# ============================================================

print("\n=== FULL DAY | OHLCV | RET ===")
display(ic_table_full_day_ohlcv_ret)

print("\n=== GESTATION | OHLCV | RET ===")
display(ic_table_ohlcv_gestation_ret)

print("\n=== EXECUTION | OHLCV | RET ===")
display(ic_table_ohlcv_execution_ret)

print("\n=== FULL DAY | OHLCV | DELTA ===")
display(ic_table_full_day_ohlcv_delta)

print("\n=== GESTATION | OHLCV | DELTA ===")
display(ic_table_ohlcv_gestation_delta)

print("\n=== EXECUTION | OHLCV | DELTA ===")
display(ic_table_ohlcv_execution_delta)


=== FULL DAY | OHLCV | RET ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,ret,ret_60,-0.518605,-0.509525,0.009080,281064,229712,,0.509525
1,high,60,ret,ret_60,-0.514695,-0.506304,0.008391,281064,229712,,0.506304
2,low,60,ret,ret_60,-0.514315,-0.504428,0.009886,281064,229712,,0.504428
3,open,60,ret,ret_60,-0.509250,-0.499916,0.009334,281064,229712,,0.499916
4,volume,60,ret,ret_60,0.071537,0.039715,-0.031822,281064,229712,,0.039715
5,close,90,ret,ret_90,-0.596851,-0.600508,-0.003657,281064,229712,,0.600508
6,high,90,ret,ret_90,-0.592396,-0.596735,-0.004339,281064,229712,,0.596735
7,low,90,ret,ret_90,-0.592414,-0.594956,-0.002542,281064,229712,,0.594956
8,open,90,ret,ret_90,-0.586712,-0.589899,-0.003187,281064,229712,,0.589899
9,volume,90,ret,ret_90,0.071848,0.045548,-0.026300,281064,229712,,0.045548



=== GESTATION | OHLCV | RET ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,ret,ret_60,-0.285432,-0.359326,-0.073894,43020,35160,,0.359326
1,high,60,ret,ret_60,-0.271811,-0.345494,-0.073683,43020,35160,,0.345494
2,low,60,ret,ret_60,-0.267987,-0.338055,-0.070068,43020,35160,,0.338055
3,open,60,ret,ret_60,-0.250625,-0.318483,-0.067858,43020,35160,,0.318483
4,volume,60,ret,ret_60,0.006721,-0.002020,-0.008741,43020,35160,,0.002020
5,close,90,ret,ret_90,-0.366713,-0.363303,0.003409,43020,35160,,0.363303
6,high,90,ret,ret_90,-0.347925,-0.349601,-0.001676,43020,35160,,0.349601
7,low,90,ret,ret_90,-0.347700,-0.339935,0.007765,43020,35160,,0.339935
8,open,90,ret,ret_90,-0.323082,-0.318251,0.004830,43020,35160,,0.318251
9,volume,90,ret,ret_90,0.014915,-0.006055,-0.020969,43020,35160,,0.006055



=== EXECUTION | OHLCV | RET ===


...

,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,delta,delta_60,-0.518397,-0.509275,0.009122,281064,229712,,0.509275
1,high,60,delta,delta_60,-0.514499,-0.506059,0.008440,281064,229712,,0.506059
2,low,60,delta,delta_60,-0.514102,-0.504174,0.009928,281064,229712,,0.504174
3,open,60,delta,delta_60,-0.509049,-0.499667,0.009382,281064,229712,,0.499667
4,volume,60,delta,delta_60,0.071525,0.039813,-0.031712,281064,229712,,0.039813
5,close,90,delta,delta_90,-0.596679,-0.600229,-0.003550,281064,229712,,0.600229
6,high,90,delta,delta_90,-0.592238,-0.596463,-0.004225,281064,229712,,0.596463
7,low,90,delta,delta_90,-0.592238,-0.594674,-0.002436,281064,229712,,0.594674
8,open,90,delta,delta_90,-0.586546,-0.589624,-0.003078,281064,229712,,0.589624
9,volume,90,delta,delta_90,0.071915,0.045720,-0.026194,281064,229712,,0.045720



=== GESTATION | OHLCV | DELTA ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,delta,delta_60,-0.285199,-0.359075,-0.073876,43020,35160,,0.359075
1,high,60,delta,delta_60,-0.271599,-0.345263,-0.073664,43020,35160,,0.345263
2,low,60,delta,delta_60,-0.267781,-0.337798,-0.070017,43020,35160,,0.337798
3,open,60,delta,delta_60,-0.250469,-0.318242,-0.067773,43020,35160,,0.318242
4,volume,60,delta,delta_60,0.006828,-0.002054,-0.008882,43020,35160,,0.002054
5,close,90,delta,delta_90,-0.366369,-0.363139,0.003230,43020,35160,,0.363139
6,high,90,delta,delta_90,-0.347646,-0.349475,-0.001829,43020,35160,,0.349475
7,low,90,delta,delta_90,-0.347427,-0.339775,0.007652,43020,35160,,0.339775
8,open,90,delta,delta_90,-0.322875,-0.318106,0.004769,43020,35160,,0.318106
9,volume,90,delta,delta_90,0.014943,-0.006049,-0.020992,43020,35160,,0.006049



=== EXECUTION | OHLCV | DELTA ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,close,60,delta,delta_60,-0.734693,-0.745173,-0.010480,64530,52740,,0.745173
1,high,60,delta,delta_60,-0.715164,-0.724331,-0.009167,64530,52740,,0.724331
2,low,60,delta,delta_60,-0.713775,-0.721513,-0.007737,64530,52740,,0.721513
3,open,60,delta,delta_60,-0.686451,-0.691853,-0.005402,64530,52740,,0.691853
4,volume,60,delta,delta_60,0.105531,0.080389,-0.025142,64530,52740,,0.080389
5,close,90,delta,delta_90,-0.779650,-0.781557,-0.001907,64530,52740,,0.781557
6,high,90,delta,delta_90,-0.759302,-0.759245,0.000057,64530,52740,,0.759245
7,low,90,delta,delta_90,-0.756053,-0.757333,-0.001280,64530,52740,,0.757333
8,open,90,delta,delta_90,-0.726138,-0.724644,0.001494,64530,52740,,0.724644
9,volume,90,delta,delta_90,0.109954,0.091978,-0.017976,64530,52740,,0.091978


### **7.1.1. Observaciones - IC OHLCV (RET vs DELTA)**

**Estabilidad frente al target**
- Los valores de **IC_IS, IC_OOS y |IC_OOS|** obtenidos con variables OHLCV son prácticamente **idénticos** al utilizar targets **RET** o **DELTA**, para todas las ventanas y horizontes analizados.
- No se observan **cambios de signo** ni alteraciones relevantes en el **ranking relativo** de las variables al cambiar el target.
- El comportamiento de OHLC es, por lo tanto, **robusto a la definición del objetivo**.

**Variables de precio (OHLC)**
- *close, high, low y open* presentan:
  - IC elevados y estables,
  - coherencia direccional consistente entre IS y OOS,
  - degradación fuera de muestra acotada.
- El orden relativo se mantiene estable en todos los casos:
  - **close > high ≈ low > open**.
- La señal asociada al precio es **persistente en todas las ventanas y horizontes**.

**Comportamiento por ventana horaria**
- **Jornada completa (full day)**  
  - |IC_OOS| de magnitud moderada.  
  - Señal presente pero atenuada al promediar distintos regímenes intradía.
- **Ventana de gestación**  
  - Incremento de |IC_OOS| respecto a full day.  
  - Mayor concentración de señal direccional.
- **Ventana de ejecución**  
  - |IC_OOS| muy elevado para todas las variables OHLC.  
  - Máxima consistencia y estabilidad fuera de muestra.

**Efecto del horizonte temporal**
- Para todas las ventanas, **H = 90** presenta valores de IC levemente superiores a **H = 60**.
- La diferencia entre horizontes es más marcada en la **ventana de ejecución**.
- El cambio de horizonte ajusta la **magnitud** de la señal, pero no su **estructura**.

**Variable volumen**
- *volume* muestra:
  - IC bajos o cercanos a cero,
  - pérdida de magnitud fuera de muestra,
  - cambios de signo en algunas ventanas.
- Este comportamiento es consistente tanto para **RET** como para **DELTA**, indicando **baja utilidad predictiva directa**.

**Observación integradora**
- La evidencia empírica muestra que:
  - el comportamiento de OHLC es estable frente al cambio de target,
  - la intensidad de la señal depende fuertemente de la ventana horaria,
  - la escala temporal introduce ajustes de magnitud, pero no cambios estructurales.

### **7.1.2. Interpretación metodológica**

Aunque las variables OHLC presentan valores de IC elevados —especialmente en la ventana de ejecución—, este resultado **no invalida el rol central de los indicadores técnicos**. Por el contrario, confirma que:

- Los **indicadores técnicos** capturan y reexpresan información ya contenida en el precio, de forma más **estructurada, filtrada y operacionalizable**.
- Las variables **OHLC actúan como variables de estado del mercado**, no como señales diseñadas explícitamente para anticipar movimientos.
- El IC elevado de OHLC es **esperable** y:
  - refuerza la consistencia del análisis,
  - confirma el régimen dominante,
  - aporta **contexto estructural** útil para el modelo.

### **7.1.3. Decisión de diseño del modelo**


En base a este análisis, se adopta el siguiente criterio:

- Las variables **OHLC** se mantienen como **features base**, normalizadas o transformadas según corresponda.
- Los **indicadores técnicos seleccionados** constituyen el **núcleo predictivo** del modelo.
- La variable **volume** no se utiliza como feature principal en esta etapa, quedando reservada para análisis contextuales futuros.

Este enfoque permite:
- equilibrar información estructural y capacidad predictiva real,
- preservar interpretabilidad económica,
- evitar tanto **redundancia innecesaria** como **exclusiones arbitrarias**.

En conjunto, la decisión es coherente con la evidencia empírica y con la arquitectura final adoptada para el modelo.

In [218]:
ohlc_cols = ['open', 'high', 'low', 'close']

## **7.2. Correlación entre OHLC**


Antes de profundizar en el análisis predictivo, se realiza un chequeo estructural de redundancia entre las variables de precio básicas (Open, High, Low, Close). Este análisis no tiene como objetivo evaluar capacidad predictiva, ni reemplazar el estudio de Information Coefficient (IS vs OOS), sino verificar el grado de colinealidad intrínseca entre las componentes OHLC, que representan distintas observaciones de una misma variable latente: el precio.

Dado que el pipeline ya define la relevancia de las variables mediante IC IS/OOS y coherencia direccional, el análisis de correlación OHLC se utiliza únicamente como sanity check, con el fin de:

- confirmar la redundancia extrema entre las variables de precio,
- evitar la duplicación innecesaria de información,
- y justificar decisiones de representación posteriores (por ejemplo, uso exclusivo de close, uso de agregados como ohlc4, o mantenimiento explícito de OHLC por motivos de estabilidad numérica o interpretabilidad).

En este marco, la elección de trabajar únicamente con close constituye una decisión de representación y parsimonia, basada en redundancia estadística, no una selección de variables por desempeño predictivo. La correlación OHLC, por lo tanto, complementa el análisis, pero no interviene en el ranking ni en la selección predictiva de features.

In [219]:
corr_ohlc = (
    mnq_intraday_with_indicators[
        ohlc_cols
    ]
    .corr(method="spearman")
    .loc[ohlc_cols]
)

corr_ohlc

,open,high,low,close
open,1.000000,0.999998,0.999998,0.999996
high,0.999998,1.000000,0.999996,0.999998
low,0.999998,0.999996,1.000000,0.999998
close,0.999996,0.999998,0.999998,1.000000


- Redundancia extrema OHLC: el análisis de correlación muestra que open, high, low y close están prácticamente perfectamente correlacionadas (ρ ≈ 0.999). Mantenerlas simultáneamente no agrega información nueva.

- Señal ya validada por IC IS/OOS: `close` presenta IC alto, estable y coherente entre IS y OOS en las tres ventanas (Full Day, Gestation y Execution). No existe evidencia de que open, high o low aporten poder predictivo incremental.

- Principio de parsimonia: eliminar variables redundantes reduce dimensionalidad, colinealidad y complejidad del modelo, mejorando estabilidad y generalización.

- Consistencia conceptual: `close` representa el precio de consenso del mercado al final de cada barra y es la base de la mayoría de indicadores técnicos utilizados (EMA, ROC, Momentum), manteniendo coherencia semántica en todo el pipeline.

- No es una selección “por descarte arbitrario”: la decisión se apoya en evidencia empírica (IC, coherencia direccional, correlación) y no compromete la información relevante.

**Decisión final:**

- Se utiliza close como única variable de precio, descartando open, high y low.
- Esta decisión responde a redundancia estadística, estabilidad IS/OOS y parsimonia, no a limitaciones del modelo

## **7.3. Correlación entre `close` e indicadores técnicos seleccionados**


El objetivo de este paso es evaluar redundancia informativa, es decir, determinar si las variables OHLCV ya están implícitamente capturadas por los indicadores técnicos seleccionados.

In [220]:
corr_ohlc_vs_indicators = (
    mnq_intraday_with_indicators[
        ["close"] + tech_indicators_finals
    ]
    .corr(method="spearman")
    .loc[["close"], tech_indicators_finals]
)

corr_ohlc_vs_indicators

,atr_norm_14,atr_norm_20,ema_60,mom_10,mom_5,roc_20,roc_30,roc_60
close,-0.266143,-0.270824,-0.010662,-0.004462,-0.002915,-0.006856,-0.007442,-0.007388


### **7.3.1. Análisis de Correlación entre features `close` e indicadores técnicos seleccionados**

- La correlación Spearman entre `close` y los indicadores técnicos es, en general, cercana a 0.
  - `ema_60`, `roc_20`, `roc_30`, `roc_60`, `mom_5`, `mom_10`: |ρ| ≈ 0.00–0.01 ⇒ relación monótona prácticamente nula con el nivel de precio.
- Los únicos valores con magnitud moderada corresponden a volatilidad:
  - `atr_norm_14`: ρ ≈ -0.266
  - `atr_norm_20`: ρ ≈ -0.271
  Esto indica que (en tu muestra) cuando el nivel de `close` es mayor, el ATR normalizado tiende a ser menor en términos relativos (o viceversa), pero no implica redundancia completa; describe un vínculo esperado entre nivel de precio y volatilidad “normalizada”.
- Interpretación metodológica:
  - Este análisis no evalúa “capacidad predictiva”, sino **redundancia informativa** (si el indicador es casi una función monótona de `close`).
  - Bajo este criterio, **EMA/ROC/MOM aportan información distinta al nivel del precio**, porque operan sobre cambios/pendientes/impulsos más que sobre el nivel.
- Conclusión operativa:
  - No hay evidencia de que `close` haga redundantes a `ema_60`, `roc_*` o `mom_*`.
  - Sí existe una relación moderada entre `close` y `atr_norm_*`, pero no es lo suficientemente alta como para justificar descarte por redundancia (está lejos de umbrales típicos como |ρ| ≥ 0.85).


### **7.3.2. Conclusión metodológica**


El conjunto final de variables queda compuesto por la variable de precio `close` y un conjunto reducido de indicadores técnicos derivados:
`ema_60`, `roc_60`, `roc_30`, `roc_20`, `mom_10`, `mom_5`, y `atr_norm_14 / atr_norm_20` (según horizonte).

Esta selección es el resultado de un proceso sistemático que combinó:
- análisis de Information Coefficient (IS vs OOS),
- coherencia direccional,
- evaluación de colinealidad y redundancia,
- y validación por ventanas horarias y horizontes.

El set final preserva información complementaria:
- el **nivel de precio** (`close`) como variable de estado,
- la **estructura y magnitud del movimiento** (`ema_60`, `roc_*`),
- la **aceleración de corto plazo** (`mom_*`),
- y el **riesgo / volatilidad normalizada** (`atr_norm_*`),

# **8. Machine Learning for Algorithmic Trading**

## **8.1. Convertir indicadores en features estadísticamente estables**

### **8.1.1. Marco teórico**

Hasta este punto, los indicadores técnicos han sido utilizados como **series crudas**, es decir, en su escala original y sin normalización contextual.  
El siguiente nivel del pipeline consiste en **transformarlos en features estadísticamente estables**, incorporando el contexto intradía del mercado.

El objetivo **no es crear nuevos indicadores**, sino **reexpresar los existentes** de forma que su significado sea comparable entre jornadas y más robusto fuera de muestra.

---

**Normalización en contexto intradía**

Las transformaciones consideradas incluyen:

- **Z-score intradía (rolling por día)**  
  Normaliza cada indicador respecto a su media y desviación estándar diaria.

- **Percentil intradía**  
  Expresa la posición relativa del indicador dentro de su distribución diaria.

- **Distancia al régimen típico del día**  
  Mide cuán alejado está el valor actual del comportamiento intradía más frecuente.

---

**Intuición clave**

El valor informativo de un indicador **no reside en su magnitud absoluta**, sino en su **rareza relativa dentro del día**.

**Ejemplo conceptual**:  
`roc_30` no aporta señal por su valor bruto,  
aporta señal por **qué tan extremo es** respecto a su distribución intradía.

---

**Beneficios del enfoque**

Este esquema permite:

- Reducir la sensibilidad a escalas y cambios de volatilidad.
- Mejorar la estabilidad del *Information Coefficient* fuera de muestra.
- Mantener intacta la lógica económica de los indicadores originales.

### **8.1.2. Implementación**

#### **Listas de columnas a conservar para análisis**

In [221]:
base_cols = ['date',	'minute_of_day', 'split_fe']
price_features = ['close']
technical_indicators_features = tech_indicators_finals.copy()
targets = ['ret_60', 'ret_90', 'delta_60', 'delta_90']

print(f'base_cols: {base_cols}')
print(f'price_features: {price_features}')
print(f'technical_indicators_features: {technical_indicators_features}')
print(f'targets: {targets}')

base_cols: ['date', 'minute_of_day', 'split_fe']
price_features: ['close']
technical_indicators_features: ['atr_norm_14', 'atr_norm_20', 'ema_60', 'mom_10', 'mom_5', 'roc_20', 'roc_30', 'roc_60']
targets: ['ret_60', 'ret_90', 'delta_60', 'delta_90']


#### **Función: Z-score "point-in-time" por día (sin fuga)**

In [222]:
import numpy as np
import pandas as pd

EPS = 1e-12  # Para evitar división por cero en la std

# ============================================================
# Función: Z-score "point-in-time" por día (sin fuga)
# ============================================================
def add_expanding_zscore_by_day(
    df: pd.DataFrame,
    cols: list[str],
    date_col: str = "date",
    min_periods: int = 10,
) -> pd.DataFrame:
    """
    Crea features normalizadas por día usando estadísticos "hasta el momento".

    Para cada día (date_col), y para cada minuto t dentro de ese día:
        z(t) = (x(t) - mean(x[<=t])) / std(x[<=t])

    Ventajas:
    - No mezcla días (cada día se normaliza por separado)
    - No usa información del futuro (solo datos hasta t)
    - Suele mejorar estabilidad OOS cuando hay cambios de régimen/volatilidad

    Parámetros:
    - cols: columnas a normalizar
    - date_col: columna que identifica el día (por ejemplo, 'date')
    - min_periods: mínimos puntos del día para empezar a calcular mean/std
                  (antes de eso devuelve NaN en el z-score)
    """
    out = df.copy()

    # Validaciones mínimas
    if not isinstance(out.index, pd.DatetimeIndex):
        raise TypeError("El índice debe ser DatetimeIndex (datetime).")
    if date_col not in out.columns:
        raise KeyError(f"Falta la columna '{date_col}' para agrupar por día.")

    # Asegura orden temporal global (por seguridad)
    out = out.sort_index()

    # Agrupa por día (no reordena los grupos)
    g = out.groupby(date_col, sort=False)

    # Calcula z-score expanding por día para cada feature
    for c in cols:
        exp_mean = (
            g[c]
            .expanding(min_periods=min_periods)
            .mean()
            .reset_index(level=0, drop=True)
        )
        exp_std = (
            g[c]
            .expanding(min_periods=min_periods)
            .std(ddof=0)
            .reset_index(level=0, drop=True)
        )

        # Feature normalizada
        out[f"{c}_z_exp"] = (out[c] - exp_mean) / (exp_std + EPS)

    return out



#### **Función: Generación de dataset `mnq_features_targets` con columnas seleccionadas**

In [223]:
final_columns = (
    base_cols
    + price_features
    + technical_indicators_features
    + targets
)

# ============================================================
# Sanity check de columnas
# ============================================================
missing_cols = [c for c in final_columns if c not in mnq_intraday_with_indicators.columns]
if missing_cols:
    raise ValueError(f"Faltan columnas requeridas en el dataset original: {missing_cols}")

# ============================================================
# Construcción del dataset final
# ============================================================
mnq_features_targets = (
    mnq_intraday_with_indicators[final_columns]
    .copy()
)

# Opcional: orden explícito
mnq_features_targets = mnq_features_targets[final_columns]

#### **Función: Generación de raw + z_exp para tus indicadores finales**

In [224]:
# ============================================================
# Pipeline: crea raw + z_exp para tus indicadores finales
# ============================================================
df = mnq_intraday_with_indicators.copy()

# 1) Nos quedamos con las columnas necesarias (features + targets + date)
#    (esto evita arrastrar columnas que no usaremos)
keep_cols = base_cols + price_features + technical_indicators_features + targets
df = df.loc[:, keep_cols].copy()
features_cols = price_features + technical_indicators_features

# 2) Creamos columnas *_raw explícitas (para comparar raw vs normalizado)
for c in features_cols:
    df[f"{c}_raw"] = df[c]

# 3) Creamos columnas normalizadas *_z_exp (expanding z-score por día)
df = add_expanding_zscore_by_day(
    df,
    cols=features_cols,
    date_col="date",
    min_periods=10,  # Ajustable: 5–15 suele ser razonable intradía
)

# 4) Armamos dataset final (raw + z_exp + targets)
final_features = (
    [f"{c}_raw" for c in features_cols] +
    [f"{c}_z_exp" for c in features_cols]
)

final_cols = base_cols + final_features + [c for c in targets if c in df.columns]

# 5) Eliminamos filas donde aún no hay z-score (primeros min_periods-1 minutos de cada día)
mnq_features_targets_norm = df.loc[:, final_cols].dropna()

#print(mnq_features_targets_norm.head(3))


In [225]:
mnq_features_targets_norm.head(5)

,date,minute_of_day,split_fe,close_raw,atr_norm_14_raw,atr_norm_20_raw,ema_60_raw,mom_10_raw,mom_5_raw,roc_20_raw,...,ema_60_z_exp,mom_10_z_exp,mom_5_z_exp,roc_20_z_exp,roc_30_z_exp,roc_60_z_exp,ret_60,ret_90,delta_60,delta_90
datetime,,,,,,,,,,,,,,,,,,,,,
2019-12-23 08:08:00-05:00,2019-12-23,488,IS,8733.50,0.000095,0.000098,0.000009,-0.000057,-0.000057,0.008588,...,-0.851386,-0.023503,0.155150,0.218200,0.785683,-2.389790,0.000429,-0.000286,3.75,-2.50
2019-12-23 08:09:00-05:00,2019-12-23,489,IS,8734.00,0.000093,0.000096,0.000064,0.000000,0.000029,0.011451,...,0.806082,0.419749,1.168984,0.612305,1.205677,-2.505943,0.000343,-0.000258,3.00,-2.25
2019-12-23 08:10:00-05:00,2019-12-23,490,IS,8734.00,0.000086,0.000092,0.000062,0.000029,0.000086,0.014314,...,0.690034,0.623308,1.611348,0.971647,1.308934,-2.826190,0.000258,-0.000401,2.25,-3.50
2019-12-23 08:11:00-05:00,2019-12-23,491,IS,8734.25,0.000082,0.000088,0.000087,0.000029,0.000057,0.014314,...,1.346309,0.590095,1.147637,0.901316,1.581859,-1.535918,0.000258,-0.000258,2.25,-2.25
2019-12-23 08:12:00-05:00,2019-12-23,492,IS,8735.00,0.000095,0.000097,0.000167,0.000200,0.000143,0.017175,...,2.582572,1.780426,1.805709,1.225156,1.908451,-1.261039,-0.000057,-0.000744,-0.50,-6.50


#### **Función: `compute_ic_is_oos_by_split`**

In [226]:
import numpy as np
import pandas as pd

def compute_ic_is_oos_by_split(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    *,
    split_col: str = "split_fe",
    is_label: str = "is",
    oos_label: str = "oos",
    method: str = "spearman",
    min_obs: int = 200,
) -> pd.DataFrame:
    """
    Calcula IC (feature vs target) por separado en IS y OOS usando split_col.

    Retorna un DataFrame con:
      feature, n_is, ic_is, n_oos, ic_oos, ret_oos_minus_is
    """

    data = df.copy()

    # Checks básicos
    needed_cols = [split_col, target] + features
    missing = [c for c in needed_cols if c not in data.columns]
    if missing:
        raise KeyError(f"Faltan columnas requeridas: {missing}")

    # Normalizamos split a string lowercase para robustez
    split = data[split_col].astype(str).str.lower()

    mask_is = split == str(is_label).lower()
    mask_oos = split == str(oos_label).lower()

    df_is = data.loc[mask_is, :]
    df_oos = data.loc[mask_oos, :]

    rows = []
    for f in features:
        # --- IS ---
        tmp_is = df_is[[f, target]].dropna()
        n_is = len(tmp_is)
        ic_is = np.nan
        if n_is >= min_obs:
            ic_is = tmp_is[f].corr(tmp_is[target], method=method)

        # --- OOS ---
        tmp_oos = df_oos[[f, target]].dropna()
        n_oos = len(tmp_oos)
        ic_oos = np.nan
        if n_oos >= min_obs:
            ic_oos = tmp_oos[f].corr(tmp_oos[target], method=method)

        rows.append({
            "feature": f,
            "n_is": n_is,
            "ic_is": ic_is,
            "n_oos": n_oos,
            "ic_oos": ic_oos,
            "ret_oos_minus_is": (ic_oos - ic_is) if (pd.notna(ic_oos) and pd.notna(ic_is)) else np.nan,
        })

    out = (
        pd.DataFrame(rows)
        .sort_values(by="ic_oos", ascending=False)
        .reset_index(drop=True)
    )
    return out


In [227]:
features_raw = [
    "close_raw",
    "atr_norm_14_raw", "atr_norm_20_raw",
    "ema_60_raw",
    "mom_10_raw", "mom_5_raw",
    "roc_20_raw", "roc_30_raw", "roc_60_raw",
]

features_z = [
    "close_z_exp",
    "atr_norm_14_z_exp", "atr_norm_20_z_exp",
    "ema_60_z_exp",
    "mom_10_z_exp", "mom_5_z_exp",
    "roc_20_z_exp", "roc_30_z_exp", "roc_60_z_exp",
]

#### **Aplicación**

In [228]:
# ============================================================
# IC IS / OOS – TARGET: RETURNS
# ============================================================

# --- H = 60 ---
ic_ret_60_raw = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_raw,
    target="ret_60",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

ic_ret_60_z = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_z,
    target="ret_60",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

# --- H = 90 ---
ic_ret_90_raw = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_raw,
    target="ret_90",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

ic_ret_90_z = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_z,
    target="ret_90",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)


# ============================================================
# IC IS / OOS – TARGET: DELTA
# ============================================================

# --- H = 60 ---
ic_delta_60_raw = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_raw,
    target="delta_60",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

ic_delta_60_z = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_z,
    target="delta_60",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

# --- H = 90 ---
ic_delta_90_raw = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_raw,
    target="delta_90",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)

ic_delta_90_z = compute_ic_is_oos_by_split(
    df=mnq_features_targets_norm,
    features=features_z,
    target="delta_90",
    split_col="split_fe",
    is_label="is",
    oos_label="oos",
)


In [229]:
# ============================================================
# Visualización IC – TARGET: RETURNS
# ============================================================

print("\n=== IC | RET | H=60 | RAW ===\n")
display(ic_ret_60_raw)

print("\n=== IC | RET | H=60 | Z-SCORE ===\n")
display(ic_ret_60_z)

print("\n=== IC | RET | H=90 | RAW ===\n")
display(ic_ret_90_raw)

print("\n=== IC | RET | H=90 | Z-SCORE ===\n")
display(ic_ret_90_z)


# ============================================================
# Visualización IC – TARGET: DELTA
# ============================================================

print("\n=== IC | DELTA | H=60 | RAW ===\n")
display(ic_delta_60_raw)

print("\n=== IC | DELTA | H=60 | Z-SCORE ===\n")
display(ic_delta_60_z)

print("\n=== IC | DELTA | H=90 | RAW ===\n")
display(ic_delta_90_raw)

print("\n=== IC | DELTA | H=90 | Z-SCORE ===\n")
display(ic_delta_90_z)



=== IC | RET | H=60 | RAW ===



,feature,n_is,ic_is,n_oos,ic_oos,ret_oos_minus_is
0,atr_norm_20_raw,274611,0.021995,224438,0.036656,0.014661
1,atr_norm_14_raw,274611,0.022316,224438,0.034861,0.012544
2,roc_60_raw,274611,0.006143,224438,0.032794,0.026651
3,ema_60_raw,274611,0.011230,224438,0.030592,0.019362
4,roc_30_raw,274611,0.008698,224438,0.020319,0.011621
5,roc_20_raw,274611,0.010358,224438,0.016830,0.006472
6,mom_10_raw,274611,0.008395,224438,0.015515,0.007119
7,mom_5_raw,274611,0.005174,224438,0.011353,0.006178
8,close_raw,274611,-0.042849,224438,-0.044036,-0.001187



=== IC | RET | H=60 | Z-SCORE ===



,feature,n_is,ic_is,n_oos,ic_oos,ret_oos_minus_is
0,close_z_exp,274611,0.024172,224438,0.019992,-0.004180
1,roc_60_z_exp,274611,0.003252,224438,0.013661,0.010408
2,ema_60_z_exp,274611,0.005903,224438,0.010075,0.004172
3,mom_10_z_exp,274611,0.001725,224438,0.004816,0.003091
4,roc_30_z_exp,274611,0.004761,224438,0.004212,-0.000549
5,mom_5_z_exp,274611,0.000339,224438,0.003000,0.002661
6,roc_20_z_exp,274611,0.004045,224438,0.002263,-0.001782
7,atr_norm_20_z_exp,274611,0.011864,224438,-0.015128,-0.026992
8,atr_norm_14_z_exp,274611,0.011085,224438,-0.016110,-0.027195



=== IC | RET | H=90 | RAW ===



...

,feature,n_is,ic_is,n_oos,ic_oos,ret_oos_minus_is
0,close_z_exp,274611,0.023343,224438,0.017750,-0.005593
1,roc_60_z_exp,274611,0.004223,224438,0.010598,0.006376
2,ema_60_z_exp,274611,0.006620,224438,0.007485,0.000864
3,mom_10_z_exp,274611,0.001969,224438,0.003882,0.001914
4,mom_5_z_exp,274611,0.000457,224438,0.002432,0.001975
5,roc_30_z_exp,274611,0.005537,224438,0.002256,-0.003280
6,roc_20_z_exp,274611,0.004670,224438,0.000875,-0.003795
7,atr_norm_20_z_exp,274611,0.011465,224438,-0.011866,-0.023331
8,atr_norm_14_z_exp,274611,0.010765,224438,-0.012578,-0.023343



=== IC | DELTA | H=90 | RAW ===



,feature,n_is,ic_is,n_oos,ic_oos,ret_oos_minus_is
0,atr_norm_20_raw,274611,0.011066,224438,0.037782,0.026715
1,atr_norm_14_raw,274611,0.011626,224438,0.036291,0.024665
2,roc_60_raw,274611,0.009571,224438,0.028189,0.018618
3,ema_60_raw,274611,0.011329,224438,0.027584,0.016256
4,roc_30_raw,274611,0.009888,224438,0.021113,0.011225
5,roc_20_raw,274611,0.007366,224438,0.018024,0.010658
6,mom_10_raw,274611,0.006535,224438,0.013001,0.006466
7,mom_5_raw,274611,0.003775,224438,0.008365,0.004590
8,close_raw,274611,-0.034801,224438,-0.034216,0.000585



=== IC | DELTA | H=90 | Z-SCORE ===



,feature,n_is,ic_is,n_oos,ic_oos,ret_oos_minus_is
0,close_z_exp,274611,0.026452,224438,0.023508,-0.002944
1,roc_60_z_exp,274611,0.003743,224438,0.011629,0.007886
2,ema_60_z_exp,274611,0.003313,224438,0.006997,0.003685
3,roc_30_z_exp,274611,0.003565,224438,0.004431,0.000866
4,roc_20_z_exp,274611,0.001646,224438,0.002858,0.001212
5,mom_10_z_exp,274611,-0.000367,224438,0.002719,0.003086
6,mom_5_z_exp,274611,-0.001560,224438,-0.000175,0.001385
7,atr_norm_20_z_exp,274611,0.007493,224438,-0.013496,-0.020990
8,atr_norm_14_z_exp,274611,0.006830,224438,-0.014468,-0.021298


### **8.1.1. Conclusiones sobre IC: RAW vs Z-SCORE (RET y DELTA)**

**1. Comparación RAW vs Z-score (estabilidad OOS)**  
- Los *features RAW* presentan, en general, *valores de IC_OOS superiores* a sus versiones normalizadas, tanto para `ret` como para `delta`, y para ambos horizontes (H = 60 y H = 90).  
- Las versiones *Z-score expanding* muestran:
  - menor magnitud de IC_OOS,
  - mayor suavidad,
  - y menor dispersión entre features,  
  lo que indica *pérdida de señal direccional inmediata*, aunque con potencial ganancia en estabilidad estructural.

**2. Comportamiento de los indicadores técnicos (RAW)**  
- Indicadores como *ATR normalizado, ROC y EMA*:
  - exhiben una *mejora clara del IC desde IS hacia OOS* (`ret_oos_minus_is > 0`),
  - lo que sugiere ausencia de sobreajuste y captura de dinámica robusta.  
- Este patrón se mantiene de forma consistente para:
  - ambos targets (`ret` y `delta`),
  - ambos horizontes (H = 60 y H = 90).

**3. Diferencias entre horizontes temporales**  
- El horizonte *H = 90* presenta:
  - valores de IC_OOS sistemáticamente superiores a H = 60,
  - tanto en features RAW como normalizados,
  - especialmente en indicadores de volatilidad (ATR) y momentum (ROC).  
- Esto refuerza la hipótesis de que la *estructura intradía relevante se manifiesta con mayor claridad a horizontes más largos*.

**4. Comportamiento de la variable `close`**  
- En su versión *RAW*, `close` muestra:
  - IC negativo y estable,
  - coherencia entre IS y OOS,
  - confirmando su rol como *variable estructural de referencia*, más que como señal direccional directa.  
- En la versión *Z-score*, `close_z_exp`:
  - cambia de signo,
  - pero presenta IC_OOS bajos y decrecientes,
  - lo que indica que la normalización elimina gran parte de su contenido direccional.

**5. Impacto de la normalización Z-score expanding**  
- La normalización intradía:
  - reduce la sensibilidad a escalas y cambios de régimen,
  - pero *debilita la señal IC pura* en esta etapa del análisis.  
- Este efecto es especialmente marcado en:
  - los indicadores de volatilidad normalizados (`atr_norm_*_z_exp`),
  - que incluso muestran cambios de signo OOS, sugiriendo que la volatilidad absoluta contiene información relevante que se pierde al estandarizarla.

**6. Comparación entre RET y DELTA**  
- Los resultados son *altamente consistentes* entre ambos targets:
  - ranking de features prácticamente idéntico,
  - magnitudes de IC muy similares,
  - mismas conclusiones estructurales.  
- Esto confirma que la diferencia entre `ret` y `delta` *no altera la naturaleza informativa de los features*, sino únicamente su escala.


### **8.1.2. Conclusión operativa**

- Los *features RAW* conservan mayor poder predictivo inmediato (IC), especialmente útiles para exploración y ranking inicial.  

- Las transformaciones *Z-score expanding* deben interpretarse como una *herramienta de estabilización estadística*, no de maximización de IC, y su utilidad debe evaluarse en el contexto de modelos completos y desempeño out-of-sample.


## **8.2. Introducir interacciones mínimas**


Hasta ahora, los indicadores se utilizaron como **features individuales**.  
El siguiente paso metodológico consiste en introducir **interacciones simples y controladas**, cuyo objetivo es:

- capturar **relaciones estructurales** (tendencia + velocidad),
- **sin aumentar complejidad innecesaria**,
- manteniendo interpretabilidad y estabilidad OOS.

Estas interacciones **no crean nueva información**, sino que **reexpresan la existente en contexto**.

### **8.2.1. Interacciones recomendadas por jornada completa y por ventanas**

En esta sección se definen interacciones **mínimas, interpretables y económicamente justificadas**, ajustadas al **set final de indicadores técnicos** seleccionado para cada ventana horaria y horizonte.  

Las interacciones no buscan crear nuevos indicadores, sino **reexpresar relaciones relevantes entre tendencia, momentum y volatilidad**, respetando el contexto intradía.


**1. Jornada completa (Full Day)**

- Indicadores disponibles
  - **H = 60**: `ema_60`, `roc_60`
  - **H = 90**: `ema_60`, `roc_60`, `atr_norm_20`

- Interacciones propuestas

  - **Pendiente de tendencia**
    - `ema_slope_1 = ema_60 - ema_60_lag1`  
      *Justificación:* la EMA aporta información por su **dirección**, no por su nivel absoluto.

  - **Tendencia × Momentum**
    - `roc60_x_emaSlope = roc_60 * ema_slope_1`  
      *Justificación:* refuerza momentum alineado con la tendencia y penaliza señales contra-tendencia.

  - **Intensidad de movimiento**
    - `roc60_abs = |roc_60|`  
      *Justificación:* mide fuerza del movimiento independientemente del signo.

  - **Filtro de volatilidad (solo H = 90)**
    - `roc60_x_atr = roc_60 * atr_norm_20`  
      *Justificación:* pondera momentum por el régimen de volatilidad estructural.

---

**2. Ventana de gestación **

- Indicadores disponibles (H = 60 / 90)

  `ema_60`, `roc_60`, `roc_30`, `roc_20`, `mom_10`, `mom_5`

- Interacciones propuestas

  - **Tendencia × Momentum temprano**
    - `mom10_x_emaSlope = mom_10 * ema_slope_1`  
      *Justificación:* identifica impulso inicial alineado con la tendencia dominante.

  - **Aceleración multi-horizonte**
    - `roc20_minus_roc60 = roc_20 - roc_60`  
    - `roc30_minus_roc60 = roc_30 - roc_60`  
      *Justificación:* mide aceleración relativa frente al movimiento estructural.

  - **Diferenciales de momentum**
    - `mom5_minus_mom10 = mom_5 - mom_10`  
      *Justificación:* detecta cambios tempranos de velocidad entre escalas cortas y medias.

---

**3. Ventana de expansión / ejecución**

- Indicadores disponibles (H = 60 / 90)

  `ema_60`, `roc_60`, `roc_30`, `roc_20`, `atr_norm_14`

- Interacciones propuestas

  - **Pendiente de tendencia**
    - `ema_slope_1`  
      *Justificación:* sigue siendo el filtro direccional principal durante la ejecución.

  - **Momentum alineado con tendencia**
    - `roc20_x_emaSlope = roc_20 * ema_slope_1`  
      *Justificación:* refuerza continuidad operativa y filtra ruido direccional.

  - **Aceleración vs estructura**
    - `roc20_minus_roc60 = roc_20 - roc_60`  
      *Justificación:* identifica continuidad o agotamiento durante la ejecución.

  - **Volatilidad × Momentum**
    - `roc60_x_atr = roc_60 * atr_norm_14`  
      *Justificación:* ajusta la señal operativa al régimen de volatilidad intradía.

---

**4. Criterio metodológico clave**

- Las interacciones **pueden calcularse sobre todo el dataset**, pero su **evaluación (IC IS/OOS)** debe realizarse **exclusivamente dentro de la ventana horaria donde tienen sentido operativo**.
- Se mantiene:
  - mismo dataset base,
  - mismas features,
  - IC condicionado por ventana temporal.

Este enfoque:
- evita *leakage*,
- preserva consistencia del pipeline,
- permite reutilizar features sin duplicar lógica.

---

**Síntesis**

- Las interacciones propuestas son **pocas, interpretables y alineadas con la lógica económica** del mercado intradía.
- No se introducen combinaciones arbitrarias ni polinomios.
- Cada interacción responde a:
  - una ventana específica,
  - un rol claro (tendencia, momentum, volatilidad).
- El enfoque prioriza **estabilidad out-of-sample, control metodológico e interpretabilidad**.

### **8.2.2. Implementación**

In [230]:
import pandas as pd


def add_minimal_interactions(
    df: pd.DataFrame,
    *,
    date_col: str = "date",
    # Indicadores base (se asume que ya existen en df)
    ema_col: str = "ema_60",
    roc60_col: str = "roc_60",
    roc20_col: str = "roc_20",
    roc30_col: str = "roc_30",
    mom5_col: str = "mom_5",
    mom10_col: str = "mom_10",
    atr14_col: str = "atr_norm_14",
    atr20_col: str = "atr_norm_20",
    # Nombres de salida
    prefix: str = "",
    # Opcional: controlar si crear atr-interactions
    add_atr_interactions: bool = True,
) -> tuple[pd.DataFrame, list[str]]:
    """
    Crea interacciones mínimas, interpretables y compatibles con tu set final:
      - Full day: ema_60, roc_60 (+ atr_norm_20 en H=90)
      - Gestation: ema_60, roc_60, roc_20, roc_30, mom_10, mom_5
      - Execution: ema_60, roc_60, roc_20, roc_30, atr_norm_14

    Importante:
      - Calcula interacciones para TODO el dataset (toda la jornada).
      - Luego se evalúan IC/ventana filtrando por horario en tu pipeline.
      - Lags y slope se calculan por día (groupby(date).shift(1)) => sin leakage.

    Devuelve:
      - df_out: dataframe con nuevas columnas
      - created_cols: lista con los nombres creados
    """
    df_out = df.copy()

    # -------------------------
    # Helper para nombres
    # -------------------------
    def _col(name: str) -> str:
        return f"{prefix}{name}" if prefix else name

    # -------------------------
    # Validaciones mínimas (solo lo imprescindible)
    # -------------------------
    needed = [date_col, ema_col, roc60_col]
    missing = [c for c in needed if c not in df_out.columns]
    if missing:
        raise KeyError(f"Faltan columnas requeridas para interacciones: {missing}")

    created_cols: list[str] = []

    # -------------------------
    # 0) Lags / pendientes (por día) -> sin fuga
    # -------------------------
    ema_lag1 = _col(f"{ema_col}_lag1")
    df_out[ema_lag1] = df_out.groupby(date_col, sort=False)[ema_col].shift(1)
    created_cols.append(ema_lag1)

    ema_slope_1 = _col("ema_slope_1")
    df_out[ema_slope_1] = df_out[ema_col] - df_out[ema_lag1]
    created_cols.append(ema_slope_1)

    # -------------------------
    # 1) Full day (siempre)
    # -------------------------
    roc60_abs = _col("roc60_abs")
    df_out[roc60_abs] = df_out[roc60_col].abs()
    created_cols.append(roc60_abs)

    roc60_x_emaSlope = _col("roc60_x_emaSlope")
    df_out[roc60_x_emaSlope] = df_out[roc60_col] * df_out[ema_slope_1]
    created_cols.append(roc60_x_emaSlope)

    # ATR x ROC (contexto de volatilidad)
    # - atr_norm_20: más coherente para full_day H=90
    # - atr_norm_14: más coherente para execution
    if add_atr_interactions:
        if atr20_col in df_out.columns:
            roc60_x_atr20 = _col("roc60_x_atr20")
            df_out[roc60_x_atr20] = df_out[roc60_col] * df_out[atr20_col]
            created_cols.append(roc60_x_atr20)

        if atr14_col in df_out.columns:
            roc60_x_atr14 = _col("roc60_x_atr14")
            df_out[roc60_x_atr14] = df_out[roc60_col] * df_out[atr14_col]
            created_cols.append(roc60_x_atr14)

    # -------------------------
    # 2) Gestation (08:00-09:00): roc_20/roc_30 y mom_5/mom_10
    #    (se crean para todo el día; se evalúan en ventana)
    # -------------------------
    if roc20_col in df_out.columns:
        roc20_minus_roc60 = _col("roc20_minus_roc60")
        df_out[roc20_minus_roc60] = df_out[roc20_col] - df_out[roc60_col]
        created_cols.append(roc20_minus_roc60)

    if roc30_col in df_out.columns:
        roc30_minus_roc60 = _col("roc30_minus_roc60")
        df_out[roc30_minus_roc60] = df_out[roc30_col] - df_out[roc60_col]
        created_cols.append(roc30_minus_roc60)

    if (mom5_col in df_out.columns) and (mom10_col in df_out.columns):
        mom5_minus_mom10 = _col("mom5_minus_mom10")
        df_out[mom5_minus_mom10] = df_out[mom5_col] - df_out[mom10_col]
        created_cols.append(mom5_minus_mom10)

    if mom10_col in df_out.columns:
        mom10_x_emaSlope = _col("mom10_x_emaSlope")
        df_out[mom10_x_emaSlope] = df_out[mom10_col] * df_out[ema_slope_1]
        created_cols.append(mom10_x_emaSlope)

    # -------------------------
    # 3) Execution (09:00-10:00): tendencia/momentum operativo + aceleración
    # -------------------------
    if mom5_col in df_out.columns:
        mom5_x_emaSlope = _col("mom5_x_emaSlope")
        df_out[mom5_x_emaSlope] = df_out[mom5_col] * df_out[ema_slope_1]
        created_cols.append(mom5_x_emaSlope)

    # Nota: antes estaba mom3_minus_mom5, pero ya no usás mom_3 en tu set final.
    # Si en algún momento reintroducís mom_3, lo agregamos.

    return df_out, created_cols


In [231]:
mnq_with_interactions, interaction_cols = add_minimal_interactions(
    mnq_features_targets,
    date_col="date",
    ema_col="ema_60",
    roc60_col="roc_60",
    roc20_col="roc_20",
    roc30_col="roc_30",
    mom5_col="mom_5",
    mom10_col="mom_10",
    atr14_col="atr_norm_14",
    atr20_col="atr_norm_20",
)

print("Interacciones creadas:")
print(interaction_cols)


Interacciones creadas:
['ema_60_lag1', 'ema_slope_1', 'roc60_abs', 'roc60_x_emaSlope', 'roc60_x_atr20', 'roc60_x_atr14', 'roc20_minus_roc60', 'roc30_minus_roc60', 'mom5_minus_mom10', 'mom10_x_emaSlope', 'mom5_x_emaSlope']


In [232]:
features_interactions_to_test = technical_indicators_features + interaction_cols
#features_interactions_to_test

In [233]:
# ============================================================
# IC tables IS vs OOS para INTERACCIONES (con cache en Drive)
# - Se calcula por ventana (full_day / gestation / execution)
# - Para ambos targets: ret y delta
# - Para ambos horizontes: 60 y 90
# ==============================

# -------------------------
# TARGET: RET
# -------------------------
ic_table_full_day_interactions_ret = load_or_compute_ic_table(
    name="ic_table_full_day_interactions_ret",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_gestation_interactions_ret = load_or_compute_ic_table(
    name="ic_table_gestation_interactions_ret",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_execution_interactions_ret = load_or_compute_ic_table(
    name="ic_table_execution_interactions_ret",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    target_prefix="ret",
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

# -------------------------
# TARGET: DELTA
# -------------------------
ic_table_full_day_interactions_delta = load_or_compute_ic_table(
    name="ic_table_full_day_interactions_delta",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_full_day,
    window_end=final_time_full_day,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_gestation_interactions_delta = load_or_compute_ic_table(
    name="ic_table_gestation_interactions_delta",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_gestation_window,
    window_end=final_time_gestation_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

ic_table_execution_interactions_delta = load_or_compute_ic_table(
    name="ic_table_execution_interactions_delta",
    df=mnq_with_interactions,
    indicator_columns=features_interactions_to_test,
    target_prefix="delta",
    horizons=(60, 90),
    window_start=start_time_execution_window,
    window_end=final_time_execution_window,
    use_daily_ic=True,
    split_col="split_fe",
    date_col="date",
)

[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_full_day_interactions_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_gestation_interactions_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_return/ic_table_execution_interactions_ret.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_full_day_interactions_delta.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_gestation_interactions_delta.parquet
[cache] Loaded: /content/drive/MyDrive/neural_profit/data/targets/ic_tables_delta/ic_table_execution_interactions_delta.parquet


In [234]:
from IPython.display import display

# ============================================================
# Helper para mostrar IC tables por horizonte
# ============================================================
def show_ic_table(title: str, df: pd.DataFrame):
    print(f"\n=== {title} ===")
    display(df.sort_values(["horizon", "abs_IC_OOS"], ascending=[True, False]))


# ============================================================
# TARGET: RET
# ============================================================
show_ic_table(
    "FULL DAY | INTERACTIONS | RET",
    ic_table_full_day_interactions_ret
)

show_ic_table(
    "GESTATION | INTERACTIONS | RET",
    ic_table_gestation_interactions_ret
)

show_ic_table(
    "EXECUTION | INTERACTIONS | RET",
    ic_table_execution_interactions_ret
)


# ============================================================
# TARGET: DELTA
# ============================================================
show_ic_table(
    "FULL DAY | INTERACTIONS | DELTA",
    ic_table_full_day_interactions_delta
)

show_ic_table(
    "GESTATION | INTERACTIONS | DELTA",
    ic_table_gestation_interactions_delta
)

show_ic_table(
    "EXECUTION | INTERACTIONS | DELTA",
    ic_table_execution_interactions_delta
)



=== FULL DAY | INTERACTIONS | RET ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,ret,ret_60,-0.193578,-0.189173,0.004405,281064,229712,,0.189173
1,ema_60_lag1,60,ret,ret_60,-0.189416,-0.183959,0.005456,280347,229126,,0.183959
2,roc_60,60,ret,ret_60,-0.187156,-0.177858,0.009298,281064,229712,,0.177858
3,roc60_x_atr20,60,ret,ret_60,-0.180600,-0.170224,0.010376,281064,229712,,0.170224
4,roc60_x_atr14,60,ret,ret_60,-0.178861,-0.168693,0.010168,281064,229712,,0.168693
5,roc_30,60,ret,ret_60,-0.135632,-0.135081,0.000551,281064,229712,,0.135081
6,roc_20,60,ret,ret_60,-0.113311,-0.114449,-0.001138,281064,229712,,0.114449
7,roc20_minus_roc60,60,ret,ret_60,0.121940,0.102249,-0.019691,281064,229712,,0.102249
8,atr_norm_20,60,ret,ret_60,0.121862,0.093256,-0.028606,281064,229712,,0.093256
9,atr_norm_14,60,ret,ret_60,0.122841,0.091789,-0.031052,281064,229712,,0.091789



=== GESTATION | INTERACTIONS | RET ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,ret,ret_60,-0.247058,-0.293809,-0.046751,43020,35160,,0.293809
1,roc_60,60,ret,ret_60,-0.221354,-0.287290,-0.065936,43020,35160,,0.287290
2,roc60_x_atr20,60,ret,ret_60,-0.216144,-0.283131,-0.066987,43020,35160,,0.283131
3,roc60_x_atr14,60,ret,ret_60,-0.212908,-0.277420,-0.064512,43020,35160,,0.277420
4,ema_60_lag1,60,ret,ret_60,-0.219248,-0.266163,-0.046915,43020,35160,,0.266163
5,roc_30,60,ret,ret_60,-0.189454,-0.241835,-0.052381,43020,35160,,0.241835
6,roc_20,60,ret,ret_60,-0.178751,-0.210442,-0.031690,43020,35160,,0.210442
7,mom_10,60,ret,ret_60,-0.163331,-0.172891,-0.009561,43020,35160,,0.172891
8,mom_5,60,ret,ret_60,-0.130002,-0.130290,-0.000288,43020,35160,,0.130290
9,mom5_minus_mom10,60,ret,ret_60,0.088180,0.099037,0.010857,43020,35160,,0.099037



=== EXECUTION | INTERACTIONS | RET ===


...

,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,delta,delta_60,-0.193498,-0.189089,0.004409,281064,229712,,0.189089
1,ema_60_lag1,60,delta,delta_60,-0.189342,-0.183872,0.005469,280347,229126,,0.183872
2,roc_60,60,delta,delta_60,-0.187085,-0.177769,0.009315,281064,229712,,0.177769
3,roc60_x_atr20,60,delta,delta_60,-0.180545,-0.170150,0.010395,281064,229712,,0.170150
4,roc60_x_atr14,60,delta,delta_60,-0.178806,-0.168621,0.010184,281064,229712,,0.168621
5,roc_30,60,delta,delta_60,-0.135591,-0.135032,0.000559,281064,229712,,0.135032
6,roc_20,60,delta,delta_60,-0.113254,-0.114408,-0.001154,281064,229712,,0.114408
7,roc20_minus_roc60,60,delta,delta_60,0.121899,0.102185,-0.019714,281064,229712,,0.102185
8,atr_norm_20,60,delta,delta_60,0.121813,0.093347,-0.028467,281064,229712,,0.093347
9,atr_norm_14,60,delta,delta_60,0.122763,0.091854,-0.030909,281064,229712,,0.091854



=== GESTATION | INTERACTIONS | DELTA ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,ema_60,60,delta,delta_60,-0.246948,-0.293657,-0.046709,43020,35160,,0.293657
1,roc_60,60,delta,delta_60,-0.221223,-0.287196,-0.065973,43020,35160,,0.287196
2,roc60_x_atr20,60,delta,delta_60,-0.216021,-0.283099,-0.067078,43020,35160,,0.283099
3,roc60_x_atr14,60,delta,delta_60,-0.212789,-0.277419,-0.064630,43020,35160,,0.277419
4,ema_60_lag1,60,delta,delta_60,-0.219193,-0.265983,-0.046790,43020,35160,,0.265983
5,roc_30,60,delta,delta_60,-0.189397,-0.241679,-0.052281,43020,35160,,0.241679
6,roc_20,60,delta,delta_60,-0.178770,-0.210273,-0.031503,43020,35160,,0.210273
7,mom_10,60,delta,delta_60,-0.163290,-0.172891,-0.009601,43020,35160,,0.172891
8,mom_5,60,delta,delta_60,-0.130014,-0.130246,-0.000231,43020,35160,,0.130246
9,mom5_minus_mom10,60,delta,delta_60,0.088104,0.099053,0.010948,43020,35160,,0.099053



=== EXECUTION | INTERACTIONS | DELTA ===


,indicator,horizon,target_prefix,target_col,IC_IS,IC_OOS,target_oos_minus_is,n_pairs_IS,n_pairs_OOS,note,abs_IC_OOS
0,roc_60,60,delta,delta_60,-0.564986,-0.579559,-0.014573,64530,52740,,0.579559
1,ema_60,60,delta,delta_60,-0.538328,-0.574979,-0.036651,64530,52740,,0.574979
2,roc60_x_atr20,60,delta,delta_60,-0.553119,-0.572792,-0.019673,64530,52740,,0.572792
3,roc60_x_atr14,60,delta,delta_60,-0.538144,-0.561238,-0.023094,64530,52740,,0.561238
4,ema_60_lag1,60,delta,delta_60,-0.507946,-0.537306,-0.029360,64530,52740,,0.537306
5,roc_30,60,delta,delta_60,-0.416087,-0.457481,-0.041393,64530,52740,,0.457481
6,roc_20,60,delta,delta_60,-0.355251,-0.385666,-0.030415,64530,52740,,0.385666
7,mom_10,60,delta,delta_60,-0.280940,-0.302256,-0.021316,64530,52740,,0.302256
8,mom_5,60,delta,delta_60,-0.211075,-0.231262,-0.020188,64530,52740,,0.231262
9,roc20_minus_roc60,60,delta,delta_60,0.223637,0.224940,0.001303,64530,52740,,0.224940


### **8.2.3. Evaluación de interacciones por ventana temporal**

**1) Robustez general (RET vs DELTA)**

  - Los resultados de interacciones para RET y DELTA son prácticamente idénticos (magnitudes, signos y ranking), tanto en full_day como en gestation y execution.
  - Esto sugiere que las conclusiones sobre qué interacciones “aportan” no dependen del tipo de target, sino del régimen horario (ventana) y del horizonte (60 vs 90).

**2) Señal dominante por ventana (|IC_OOS|)**

  - execution >> gestation > full_day, para ambos horizontes:
    - execution: |IC_OOS| ~ 0.56-0.61 (muy alto)
    - gestation: |IC_OOS| ~ 0.26-0.29 (alto/moderado)
    - full_day:  |IC_OOS| ~ 0.17-0.24 (moderado)
  - Interpretación: la ventana de ejecución concentra el régimen más direccional y estable; full_day “diluye” señal al mezclar regímenes.

**3) Qué interacciones sí parecen útiles (y por qué)**

  - Interacciones del tipo “ROC x ATR”:
    - roc60_x_atr20 y roc60_x_atr14 quedan consistentemente arriba en gestation y execution, y también aportan en full_day.
    - Lectura: la interacción captura “dirección (ROC) ponderada por contexto de volatilidad/riesgo (ATR)”, y esa combinación es estable OOS.
  - Diferenciales de ROC:
    - roc20_minus_roc60 aparece con señal positiva estable en full_day y execution (y moderada en gestation).
    - Lectura: mide aceleración/cambio de ritmo (corto vs estructural). En execution destaca (|IC_OOS| ~ 0.225–0.230).
  - Diferencial de momentum:
    - mom5_minus_mom10 es consistente en las 3 ventanas y mejora bastante en execution (|IC_OOS| ~ 0.176–0.193 según H).
    - Lectura: útil como “proxy” de aceleración entre escalas (corto vs medio), especialmente en la fase operativa.

**4) Interacciones que NO aportan (o son débiles / inestables)**

  - Interacciones con la pendiente de la EMA:
    - ema_slope_1 tiene |IC_OOS| bajo en full_day (~0.01) y sube algo en gestation/execution (~0.04–0.07) pero sigue muy por debajo del resto.
    - roc60_x_emaSlope es esencialmente nulo (|IC_OOS| ~ 0.000–0.002) y con pequeños flips en algunos casos.
    - mom10_x_emaSlope y mom5_x_emaSlope son bajos (|IC_OOS| ~ 0.006–0.017) y, en gestation (H=60 y H=90), llegan a valores OOS cercanos a 0 o negativos.
  - Magnitud absoluta:
    - roc60_abs es débil en full_day (|IC_OOS| ~ 0.007–0.009), y en gestation incluso aparece con OOS negativo (signo contrario al esperado), lo que sugiere poca estabilidad.

**5) Efecto del horizonte (H=90 vs H=60)**

  - En general H=90 incrementa levemente |IC_OOS| para los “bloques base” (ema_60, roc_60, roc_30, roc_20) en full_day y execution.
  - En gestation, el patrón es más mixto (algunas variables e interacciones bajan al pasar a H=90), lo cual es coherente con que la dinámica “temprana” puede ser más útil para horizontes más cortos o intermedios.

**6) Implicación práctica para tu pipeline**

  - Mantener el core (ema_60, roc_60, roc_20/30 según ventana) sigue siendo coherente.
  - Si vas a priorizar pocas interacciones “con retorno marginal” y buena estabilidad OOS, las candidatas más sólidas son:
    - roc60_x_atr20 (y/o roc60_x_atr14)
    - roc20_minus_roc60 (y opcional roc30_minus_roc60, con menor fuerza relativa)
    - mom5_minus_mom10
  - Evitar (por baja señal / posible inestabilidad): roc60_x_emaSlope, roc60_abs y (con cautela) las interacciones momentum_x_emaSlope, especialmente evaluadas fuera de su ventana.

**7) Nota metodológica clave**

  - El fuerte desempeño de varias “interacciones” puede estar reflejando que combinan variables ya muy informativas en esas ventanas (especialmente execution).
  - Por tanto, la decisión final debería basarse en:
    - ganancia marginal vs indicadores base,
    - estabilidad OOS (que aquí luce buena en las top),
    - y control de redundancia (correlaciones entre interacciones y bases dentro de cada ventana).


### **8.2.4. Decisión metodológica sobre interacciones y features técnicos**

A partir del análisis de **IC (IS vs OOS)** y su evaluación por ventanas temporales (*Full Day, Gestation y Execution*), se adopta la siguiente decisión respecto al uso de **interacciones entre indicadores técnicos** y la **conservación de indicadores base**.

---

1) Indicadores técnicos base (se mantienen)

Features base (según consolidación final por ventana/horizonte)
- **Full Day (H=60):** `ema_60`, `roc_60`
- **Full Day (H=90):** `ema_60`, `roc_60`, `atr_norm_20`
- **Gestation (H=60/90):** `ema_60`, `roc_60`, `roc_20`, `roc_30`, `mom_10`, `mom_5`
- **Execution (H=60/90):** `ema_60`, `roc_60`, `roc_20`, `roc_30`, `atr_norm_14`

Justificación
- Presentan **coherencia direccional completa** (sin flips) y **estabilidad IS↔OOS** en todas las ventanas evaluadas.
- El patrón de fuerza es consistente con el régimen intradía: **Execution >> Gestation > Full Day** (en magnitud de |IC_OOS|).
- `ema_60` y `roc_60` se consolidan como **factores estructurales** (robustos en todo el día).
- `atr_norm_20` (H=90) y `atr_norm_14` (Execution) aportan **contexto de riesgo/volatilidad** relevante para horizontes/ventanas específicas.

Conclusión
- Los indicadores base constituyen el **núcleo del feature set** y no deben ser reemplazados.

---

2) Interacciones seleccionadas (candidatas con mejor aporte y estabilidad)

Interacciones con mejor evidencia empírica (prioridad)
- `roc60_x_atr20`
- `roc60_x_atr14`
- `roc20_minus_roc60`
- `mom5_minus_mom10`

Justificación
- Las interacciones **ROC × ATR** muestran **señal alta y estable OOS**, especialmente en **Execution** y también en **Gestation**, manteniendo coherencia entre RET y DELTA.
- `roc20_minus_roc60` captura **aceleración relativa** (corto vs movimiento estructural) y se vuelve particularmente informativa en **Execution**.
- `mom5_minus_mom10` captura **cambios de velocidad entre escalas** y presenta buen desempeño, con mejora clara en **Execution**.

Nota operativa
- Estas interacciones deben evaluarse y/o ponderarse **por ventana** (donde tienen sentido), aunque se calculen para todo el dataset.

---

3) Interacciones descartadas o de baja prioridad

Descartar (baja señal / aporte marginal)
- `roc60_x_emaSlope` (prácticamente nulo)
- `roc60_abs` (débil y con comportamiento OOS desfavorable fuera de Execution)
- `mom10_x_emaSlope`, `mom5_x_emaSlope` (señal baja y degradación OOS, especialmente en Gestation)
- `ema_slope_1` (señal baja frente a alternativas más fuertes)

Baja prioridad (puede quedar como opcional)
- `roc30_minus_roc60` (señal positiva, pero típicamente inferior a `roc20_minus_roc60` y más variable por ventana)

Motivo
- No aportan mejora clara respecto al set base + mejores interacciones, y agregan complejidad con retorno marginal.

---

4) Regla metodológica adoptada

- **Indicadores base**: describen el “estado” (tendencia, dirección, volatilidad/riesgo).
- **Interacciones**: describen “relaciones” (aceleración, dirección ponderada por riesgo, cambios de escala).

Por consistencia metodológica, las interacciones se incorporan como **refinamiento**, no como reemplazo de los indicadores base.

---

5) Implementación práctica por ventana

- Full Day
  - Base: ✅ (`ema_60`, `roc_60`, y `atr_norm_20` para H=90)
  - Interacciones: ✅ solo las más robustas y con aporte estable (preferentemente `roc60_x_atr20`, `roc60_x_atr14`, `roc20_minus_roc60`, `mom5_minus_mom10`), evitando “explosión” de features.

- Gestation
  - Base: ✅
  - Interacciones: ✅ priorizar `roc60_x_atr20`, `roc60_x_atr14` y `mom5_minus_mom10`; evaluar `roc20_minus_roc60` como complementaria.

- Execution
  - Base: ✅ (incluyendo `atr_norm_14`)
  - Interacciones: ✅ aquí es donde más sentido tienen; priorizar `roc60_x_atr*`, `roc20_minus_roc60` y `mom5_minus_mom10`.

---

Conclusión final

- El set base se mantiene como señal estructural.
- Las interacciones se incorporan de forma **selectiva y controlada**, privilegiando aquellas con **alta estabilidad OOS** y **coherencia RET/DELTA**, y concentrando su evaluación en las ventanas donde el mercado está en fase activa (especialmente Execution).


## **8.3. Verificar estabilidad por horizonte**


### **8.3.1. Introducción**


En predicción intradía, un mismo indicador **no necesariamente mantiene su poder informativo al variar el horizonte de predicción**.  
Un feature puede capturar dinámicas de corto plazo (impulsos, aceleraciones), pero perder relevancia cuando el horizonte se extiende, o viceversa.

Por este motivo, no basta con observar un **IC positivo en un único horizonte**: es necesario evaluar **la estabilidad de la relación feature–target al cambiar H**.

Este análisis permite distinguir entre:
- señales **estructurales** del mercado,
- señales **dependientes de la escala temporal**, y
- señales **inestables o espurias**.

El objetivo es determinar, para cada feature, si:

- **Generaliza entre horizontes** (H = 60 y H = 90),
- Es **específica de un horizonte**, o
- Es **inestable** y debe descartarse.

En esta etapa **no se maximiza el IC**, sino que se evalúa **consistencia out-of-sample**, coherencia direccional y estabilidad relativa de magnitud.

---

### Criterios de clasificación

Para cada feature *f*:

1. **Feature estable**
   - IC OOS distinto de cero en **H = 60 y H = 90**
   - Mantiene el **signo**
   - La variación de magnitud al cambiar H es **moderada y económicamente explicable**

   > Puede utilizarse en ambos horizontes.

2. **Feature especializada**
   - IC OOS relevante solo en **un horizonte**
   - IC OOS débil o cercano a cero en el otro
   - Sin cambios erráticos de signo

   > Se utiliza únicamente en el horizonte donde aporta señal.

3. **Feature inestable**
   - IC OOS cercano a cero o negativo
   - Cambios de signo sin patrón claro
   - Ruptura evidente al variar el horizonte

   > Se descarta.

---

### Intuición clave

- Un feature **estable** refleja una dinámica persistente del mercado.
- Un feature **especializado** captura efectos propios de una escala temporal concreta.
- Un feature **inestable** suele indicar ruido, sobreajuste o dependencia excesiva del contexto.

Este enfoque evita forzar indicadores fuera de su dominio natural de validez.

---

### Resultado esperado

- Clasificar los features en lugar de eliminarlos arbitrariamente.
- Definir un **set coherente por horizonte**.
- Preparar el terreno para modelos con mejor generalización out-of-sample.



### **8.3.2. Implementación**


#### Código

In [235]:
# ============================================================
# IC IS vs OOS usando split_fe (no fechas) - GENERICO
# ============================================================
def compute_ic_is_oos_by_split(
    df: pd.DataFrame,
    features: list[str],
    target: str,
    *,
    split_col: str = "split_fe",
    is_label: str = "IS",
    oos_label: str = "OOS",
    method: str = "spearman",
    min_obs: int = 200,
) -> pd.DataFrame:
    """
    Calcula IC (Spearman/Pearson) feature vs target por separado en IS y OOS
    usando la columna split_fe.

    Retorna columnas:
      feature, n_is, ic_is, n_oos, ic_oos, target_oos_minus_is
    """
    data = df.copy()

    if split_col not in data.columns:
        raise KeyError(f"Falta columna '{split_col}' en el DataFrame.")
    if target not in data.columns:
        raise KeyError(f"Target '{target}' no existe en el DataFrame.")

    df_is = data.loc[data[split_col] == is_label, :]
    df_oos = data.loc[data[split_col] == oos_label, :]

    rows = []
    for f in features:
        if f not in data.columns:
            raise KeyError(f"Feature '{f}' no existe en el DataFrame.")

        tmp_is = df_is[[f, target]].dropna()
        n_is = len(tmp_is)
        ic_is = np.nan
        if n_is >= min_obs:
            ic_is = tmp_is[f].corr(tmp_is[target], method=method)

        tmp_oos = df_oos[[f, target]].dropna()
        n_oos = len(tmp_oos)
        ic_oos = np.nan
        if n_oos >= min_obs:
            ic_oos = tmp_oos[f].corr(tmp_oos[target], method=method)

        rows.append({
            "feature": f,
            "n_is": n_is,
            "ic_is": ic_is,
            "n_oos": n_oos,
            "ic_oos": ic_oos,
            "target_oos_minus_is": (ic_oos - ic_is)
            if (pd.notna(ic_oos) and pd.notna(ic_is)) else np.nan
        })

    out = pd.DataFrame(rows)
    out["abs_ic_oos"] = out["ic_oos"].abs()
    out = out.sort_values("abs_ic_oos", ascending=False).reset_index(drop=True)
    return out

In [236]:
# ============================================================
# Estabilidad por horizonte (H=60 vs H=90) usando split_fe
# - FUNCIONA PARA RET O DELTA (o cualquier par de targets)
# ============================================================
def evaluate_feature_stability_by_horizon_split(
    df: pd.DataFrame,
    features: list[str],
    *,
    target_h60: str,
    target_h90: str,
    split_col: str = "split_fe",
    is_label: str = "IS",
    oos_label: str = "OOS",
    method: str = "spearman",
    min_obs: int = 200,
    min_abs_ic: float = 0.01,
    eps: float = 1e-12,
) -> pd.DataFrame:
    """
    Calcula IC IS/OOS para H=60 y H=90 y clasifica cada feature como:
      - stable: |IC_OOS|>=min_abs_ic en ambos horizontes + mismo signo
      - specialized_h60: señal en H60 (>=thr) y débil en H90 (<thr)
      - specialized_h90: señal en H90 (>=thr) y débil en H60 (<thr)
      - unstable: lo demás (incluye flips con señal relevante)
      - unknown: si hay NaNs por falta de datos

    Devuelve una tabla consolidada por feature.
    """
    ic60 = compute_ic_is_oos_by_split(
        df=df,
        features=features,
        target=target_h60,
        split_col=split_col,
        is_label=is_label,
        oos_label=oos_label,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_60",
        "ic_oos": "ic_oos_60",
        "target_oos_minus_is": "oos_minus_is_60",
        "n_is": "n_is_60",
        "n_oos": "n_oos_60",
    })

    ic90 = compute_ic_is_oos_by_split(
        df=df,
        features=features,
        target=target_h90,
        split_col=split_col,
        is_label=is_label,
        oos_label=oos_label,
        method=method,
        min_obs=min_obs,
    ).rename(columns={
        "ic_is": "ic_is_90",
        "ic_oos": "ic_oos_90",
        "target_oos_minus_is": "oos_minus_is_90",
        "n_is": "n_is_90",
        "n_oos": "n_oos_90",
    })

    out = ic60.merge(ic90, on="feature", how="inner")

    out["abs_oos_60"] = out["ic_oos_60"].abs()
    out["abs_oos_90"] = out["ic_oos_90"].abs()
    out["abs_min_oos"] = out[["abs_oos_60", "abs_oos_90"]].min(axis=1)

    out["ratio_oos_90_over_60"] = out["abs_oos_90"] / (out["abs_oos_60"] + eps)

    out["sign_oos_60"] = np.sign(out["ic_oos_60"])
    out["sign_oos_90"] = np.sign(out["ic_oos_90"])
    out["sign_flip_h60_h90"] = (
        (out["sign_oos_60"] != 0) & (out["sign_oos_90"] != 0) &
        (out["sign_oos_60"] != out["sign_oos_90"])
    )

    def _label(row) -> str:
        o60, o90 = row["ic_oos_60"], row["ic_oos_90"]
        if pd.isna(o60) or pd.isna(o90):
            return "unknown"

        strong60 = abs(o60) >= min_abs_ic
        strong90 = abs(o90) >= min_abs_ic
        flip = bool(row["sign_flip_h60_h90"])

        if strong60 and strong90 and (not flip):
            return "stable"
        if strong60 and (not strong90):
            return "specialized_h60"
        if strong90 and (not strong60):
            return "specialized_h90"
        if flip and strong60 and strong90:
            return "unstable"
        return "unstable"

    out["stability_label"] = out.apply(_label, axis=1)

    order = {"stable": 0, "specialized_h60": 1, "specialized_h90": 2, "unstable": 3, "unknown": 4}
    out["_ord"] = out["stability_label"].map(order).fillna(99).astype(int)
    out = out.sort_values(by=["_ord", "abs_min_oos"], ascending=[True, False]).drop(columns="_ord")

    cols = [
        "feature", "stability_label",
        "ic_is_60", "ic_oos_60", "oos_minus_is_60",
        "ic_is_90", "ic_oos_90", "oos_minus_is_90",
        "abs_oos_60", "abs_oos_90", "abs_min_oos",
        "ratio_oos_90_over_60",
        "sign_oos_60", "sign_oos_90", "sign_flip_h60_h90",
        "n_is_60", "n_oos_60", "n_is_90", "n_oos_90",
    ]
    return out[cols]

In [237]:
# ============================================================
# Helper: filtrar por ventana intradía (HH:MM -> minute_of_day)
# ============================================================
def filter_by_minute_window(
    df: pd.DataFrame,
    *,
    minute_col: str = "minute_of_day",
    start_time: str | None = None,   # "HH:MM"
    end_time: str | None = None,     # "HH:MM"
) -> pd.DataFrame:
    """
    Filtra df usando minute_of_day, pero recibiendo la ventana
    como strings "HH:MM".
    - start_time / end_time son inclusivos.
    """

    def hhmm_to_minute(hhmm: str) -> int:
        h, m = map(int, hhmm.split(":"))
        return h * 60 + m

    out = df

    if start_time is not None:
        start_minute = hhmm_to_minute(start_time)
        out = out[out[minute_col] >= start_minute]

    if end_time is not None:
        end_minute = hhmm_to_minute(end_time)
        out = out[out[minute_col] <= end_minute]

    return out

In [238]:
technical_indicators_features
base_features = ['close']
interactions_features = [
 'roc60_x_atr20',
 'roc60_x_atr14',
 'roc20_minus_roc60',
 'mom5_minus_mom10',
]

#### Aplicación Full Day

In [239]:
# 1) Define features
features_to_check = technical_indicators_features + interactions_features

In [240]:
# 2) Full day (sin filtrar)
stability_full_day_ret = evaluate_feature_stability_by_horizon_split(
    df=mnq_with_interactions,
    features=features_to_check,
    target_h60="ret_60",
    target_h90="ret_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)



stability_full_day_delta = evaluate_feature_stability_by_horizon_split(
    df=mnq_with_interactions,
    features=features_to_check,
    target_h60="delta_60",
    target_h90="delta_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)

#### Aplicación Gestation window

In [241]:
df_gestation = filter_by_minute_window(
    mnq_with_interactions,
    start_time=start_time_gestation_window,
    end_time=final_time_gestation_window,
)

In [242]:
stability_gestation_ret = evaluate_feature_stability_by_horizon_split(
    df=df_gestation,
    features=features_to_check,
    target_h60="ret_60",
    target_h90="ret_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)

stability_gestation_delta = evaluate_feature_stability_by_horizon_split(
    df=df_gestation,
    features=features_to_check,
    target_h60="delta_60",
    target_h90="delta_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)




#### Aplicación Execution window

In [243]:
df_execution = filter_by_minute_window(
    mnq_with_interactions,
    start_time=start_time_execution_window,
    end_time=final_time_execution_window,
)

In [244]:
stability_execution_ret = evaluate_feature_stability_by_horizon_split(
    df=df_execution,
    features=features_to_check,
    target_h60="ret_60",
    target_h90="ret_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)

stability_execution_delta = evaluate_feature_stability_by_horizon_split(
    df=df_execution,
    features=features_to_check,
    target_h60="delta_60",
    target_h90="delta_90",
    split_col="split_fe",
    is_label="IS",
    oos_label="OOS",
    method="spearman",
    min_obs=200,
    min_abs_ic=0.01,
)


### **8.3.3.Conclusiones rápidas - Estabilidad por horizonte y ventana**


#### **Observaciones - Full Day (Return y Delta)**


In [248]:
print("\nstability_full_day_return\n")
display(stability_full_day_ret)
print("\nstability_full_day_delta\n")
display(stability_full_day_delta)


stability_full_day_return



,feature,stability_label,ic_is_60,ic_oos_60,oos_minus_is_60,ic_is_90,ic_oos_90,oos_minus_is_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,atr_norm_20,stable,0.024497,0.036683,0.012187,0.017636,0.039877,0.022241,0.036683,0.039877,0.036683,1.087051,1.0,1.0,False,281064,229712,281064,229712
1,atr_norm_14,stable,0.024752,0.035008,0.010256,0.018018,0.038195,0.020176,0.035008,0.038195,0.035008,1.091030,1.0,1.0,False,281064,229712,281064,229712
2,roc60_x_atr14,stable,0.004762,0.033860,0.029098,0.008291,0.032460,0.024170,0.033860,0.032460,0.032460,0.958663,1.0,1.0,False,281064,229712,281064,229712
3,roc60_x_atr20,stable,0.004619,0.033640,0.029022,0.008220,0.032049,0.023829,0.033640,0.032049,0.032049,0.952689,1.0,1.0,False,281064,229712,281064,229712
4,roc_60,stable,0.005668,0.031913,0.026246,0.009021,0.030724,0.021703,0.031913,0.030724,0.030724,0.962728,1.0,1.0,False,281064,229712,281064,229712
5,ema_60,stable,0.010691,0.029803,0.019113,0.010930,0.029764,0.018834,0.029803,0.029764,0.029764,0.998678,1.0,1.0,False,281064,229712,281064,229712
6,roc20_minus_roc60,stable,-0.000973,-0.026092,-0.025119,-0.005576,-0.022569,-0.016993,0.026092,0.022569,0.022569,0.864997,-1.0,-1.0,False,281064,229712,281064,229712
7,roc_30,stable,0.007810,0.019952,0.012142,0.008917,0.023236,0.014319,0.019952,0.023236,0.019952,1.164625,1.0,1.0,False,281064,229712,281064,229712
8,roc_20,stable,0.009951,0.016372,0.006421,0.006671,0.019300,0.012628,0.016372,0.019300,0.016372,1.178798,1.0,1.0,False,281064,229712,281064,229712
9,mom_10,stable,0.008253,0.014951,0.006699,0.006128,0.013512,0.007384,0.014951,0.013512,0.013512,0.903746,1.0,1.0,False,281064,229712,281064,229712



stability_full_day_delta



,feature,stability_label,ic_is_60,ic_oos_60,oos_minus_is_60,ic_is_90,ic_oos_90,oos_minus_is_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,atr_norm_20,stable,0.021319,0.035851,0.014532,0.014038,0.038138,0.024100,0.035851,0.038138,0.035851,1.063799,1.0,1.0,False,281064,229712,281064,229712
1,atr_norm_14,stable,0.021645,0.034389,0.012744,0.014533,0.036695,0.022162,0.034389,0.036695,0.034389,1.067045,1.0,1.0,False,281064,229712,281064,229712
2,roc60_x_atr14,stable,0.003979,0.031059,0.027080,0.007603,0.029824,0.022221,0.031059,0.029824,0.029824,0.960227,1.0,1.0,False,281064,229712,281064,229712
3,roc60_x_atr20,stable,0.003838,0.030830,0.026992,0.007543,0.029412,0.021869,0.030830,0.029412,0.029412,0.954014,1.0,1.0,False,281064,229712,281064,229712
4,roc_60,stable,0.004955,0.028974,0.024018,0.008400,0.027976,0.019576,0.028974,0.027976,0.027976,0.965581,1.0,1.0,False,281064,229712,281064,229712
5,ema_60,stable,0.009924,0.027466,0.017542,0.010047,0.027145,0.017098,0.027466,0.027145,0.027145,0.988310,1.0,1.0,False,281064,229712,281064,229712
6,roc20_minus_roc60,stable,-0.000236,-0.023043,-0.022807,-0.005200,-0.020306,-0.015106,0.023043,0.020306,0.020306,0.881235,-1.0,-1.0,False,281064,229712,281064,229712
7,roc_30,stable,0.007472,0.018128,0.010655,0.008298,0.020837,0.012539,0.018128,0.020837,0.018128,1.149443,1.0,1.0,False,281064,229712,281064,229712
8,roc_20,stable,0.009732,0.014998,0.005266,0.006198,0.017395,0.011198,0.014998,0.017395,0.014998,1.159861,1.0,1.0,False,281064,229712,281064,229712
9,mom_10,stable,0.007908,0.013898,0.005990,0.005840,0.012212,0.006372,0.013898,0.012212,0.012212,0.878685,1.0,1.0,False,281064,229712,281064,229712


**Return**

1. **Predominio claro de señales estructurales**
   - `atr_norm_14`, `atr_norm_20`, `ema_60` y `roc_60` muestran:
     - IC OOS **positivo en H=60 y H=90**.
     - **Mismo signo** y ratios cercanos a 1.
   - Esto confirma que, en jornada completa, la **volatilidad normalizada**, la **tendencia** y el **momentum absoluto** capturan dinámicas persistentes del mercado.

2. **Robustez transversal por horizonte**
   - Para los features estables, `abs_oos_60 ≈ abs_oos_90`.
   - `ratio_oos_90_over_60` cercano a 1 indica **ausencia de degradación** al extender el horizonte.
   - Son candidatos naturales a **features compartidos entre H=60 y H=90**.

3. **Interacciones con ATR: señal real, no espuria**
   - `roc60_x_atr14` y `roc60_x_atr20` aparecen como *stable*.
   - El incremento fuerte de IC OOS vs IS sugiere que **el escalado del momentum por volatilidad** aporta información adicional en *full day*.
   - No hay flips ni colapso al pasar de H=60 a H=90.

4. **Diferenciales de ROC: señal estable pero contraria**
   - `roc20_minus_roc60` es estable con **signo negativo consistente**.
   - Interpretable como señal de **desaceleración relativa** en movimientos extendidos.
   - Puede usarse, pero su interpretación debe ser explícitamente *contrarian*.

5. **Momentum corto: utilidad limitada**
   - `mom_10` es estable pero con **magnitud menor**.
   - `mom_5` y `mom5_minus_mom10` quedan como *specialized_h60*:
     - Señal débil.
     - No generalizan bien a H=90.
   - En *full day*, el mercado diluye señales de muy corto plazo.

---

**Delta**

1. **Resultados altamente consistentes con Return**
   - El ranking y las etiquetas (*stable / specialized / unstable*) se mantienen casi idénticos.
   - Refuerza que las conclusiones **no dependen del target**, sino de la estructura temporal del mercado.

2. **ATR y EMA siguen liderando**
   - `atr_norm_14`, `atr_norm_20`, `ema_60` mantienen:
     - IC OOS sólido.
     - Incremento IS → OOS.
   - Confirma que el tamaño absoluto del movimiento (*delta*) está fuertemente ligado a la **volatilidad intradía agregada**.

3. **Interacciones con ATR siguen siendo válidas**
   - `roc60_x_atr*` vuelve a ser *stable*.
   - Aporta evidencia de que estas interacciones **no están sobreajustadas al target return**.

4. **Interacción momentum-diff se degrada**
   - `mom5_minus_mom10` pasa a *unstable*.
   - IC bajo, sin ventaja clara en ningún horizonte.
   - En *full day + delta*, **no justifica su inclusión**.

---

**Síntesis operativa — *Full Day***

- **Features a mantener (Return y Delta, H=60 y H=90):**
  - `atr_norm_14`, `atr_norm_20`
  - `ema_60`
  - `roc_60`, `roc_30`, `roc_20`
  - `roc60_x_atr14`, `roc60_x_atr20`
  - `roc20_minus_roc60` (con interpretación contraria)

- **Features secundarios / opcionales:**
  - `mom_10` (baja magnitud)

- **Features a excluir en full day:**
  - `mom_5`
  - `mom5_minus_mom10`

**Conclusión**  
En *Full Day*, el mercado premia **señales agregadas y estructurales**.  
Las interacciones solo funcionan cuando **refuerzan esa estructura** (momentum × volatilidad), no cuando intentan capturar micro-dinámica.


#### **Observaciones - Gestation (Return y Delta)**


In [247]:
print("\nstability_gestation_ret\n")
display(stability_gestation_ret)
print("\nstability_gestation_delta\n")
display(stability_gestation_delta)


stability_gestation_ret



,feature,stability_label,ic_is_60,ic_oos_60,oos_minus_is_60,ic_is_90,ic_oos_90,oos_minus_is_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,atr_norm_20,stable,0.011971,0.061300,0.049329,0.029529,0.086325,0.056797,0.061300,0.086325,0.061300,1.408243,1.0,1.0,False,43020,35160,43020,35160
1,atr_norm_14,stable,0.009354,0.058248,0.048894,0.028697,0.082117,0.053420,0.058248,0.082117,0.058248,1.409793,1.0,1.0,False,43020,35160,43020,35160
5,roc20_minus_roc60,stable,0.032386,-0.032117,-0.064503,0.028432,-0.024475,-0.052907,0.032117,0.024475,0.024475,0.762065,-1.0,-1.0,False,43020,35160,43020,35160
2,roc_20,specialized_h60,0.018859,-0.050759,-0.069617,0.014843,-0.008947,-0.023789,0.050759,0.008947,0.008947,0.176257,-1.0,-1.0,False,43020,35160,43020,35160
4,mom_10,specialized_h60,0.008405,-0.032800,-0.041205,0.013066,-0.005278,-0.018344,0.032800,0.005278,0.005278,0.160910,-1.0,-1.0,False,43020,35160,43020,35160
6,ema_60,specialized_h60,-0.001636,-0.030759,-0.029123,-0.006190,0.004516,0.010707,0.030759,0.004516,0.004516,0.146826,-1.0,1.0,True,43020,35160,43020,35160
7,mom5_minus_mom10,specialized_h60,-0.006587,0.018556,0.025144,-0.010439,0.001045,0.011485,0.018556,0.001045,0.001045,0.056334,1.0,1.0,False,43020,35160,43020,35160
3,roc_30,specialized_h60,0.012503,-0.047305,-0.059808,0.012039,-0.000952,-0.012991,0.047305,0.000952,0.000952,0.020118,-1.0,-1.0,False,43020,35160,43020,35160
8,mom_5,specialized_h60,0.005276,-0.013739,-0.019016,0.008136,0.000541,-0.007595,0.013739,0.000541,0.000541,0.039381,-1.0,1.0,True,43020,35160,43020,35160
9,roc60_x_atr14,specialized_h90,-0.014864,-0.008995,0.005869,-0.015989,0.011084,0.027073,0.008995,0.011084,0.008995,1.232200,-1.0,1.0,True,43020,35160,43020,35160



stability_gestation_delta



,feature,stability_label,ic_is_60,ic_oos_60,oos_minus_is_60,ic_is_90,ic_oos_90,oos_minus_is_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,atr_norm_20,stable,0.011162,0.056323,0.045161,0.026767,0.080064,0.053297,0.056323,0.080064,0.056323,1.421510,1.0,1.0,False,43020,35160,43020,35160
2,atr_norm_14,stable,0.008702,0.053126,0.044424,0.026314,0.075592,0.049277,0.053126,0.075592,0.053126,1.422883,1.0,1.0,False,43020,35160,43020,35160
6,roc20_minus_roc60,stable,0.034889,-0.031546,-0.066435,0.030130,-0.022850,-0.052981,0.031546,0.022850,0.022850,0.724348,-1.0,-1.0,False,43020,35160,43020,35160
1,roc_20,stable,0.018793,-0.054299,-0.073091,0.013338,-0.011513,-0.024851,0.054299,0.011513,0.011513,0.212029,-1.0,-1.0,False,43020,35160,43020,35160
9,roc60_x_atr14,specialized_h60,-0.016669,-0.012060,0.004610,-0.018287,0.008116,0.026403,0.012060,0.008116,0.008116,0.672989,-1.0,1.0,True,43020,35160,43020,35160
4,mom_10,specialized_h60,0.008767,-0.035690,-0.044457,0.012873,-0.007666,-0.020538,0.035690,0.007666,0.007666,0.214784,-1.0,-1.0,False,43020,35160,43020,35160
10,roc60_x_atr20,specialized_h60,-0.016863,-0.011958,0.004905,-0.018321,0.007639,0.025960,0.011958,0.007639,0.007639,0.638833,-1.0,1.0,True,43020,35160,43020,35160
3,roc_30,specialized_h60,0.012088,-0.051908,-0.063996,0.010061,-0.004681,-0.014742,0.051908,0.004681,0.004681,0.090173,-1.0,-1.0,False,43020,35160,43020,35160
7,mom5_minus_mom10,specialized_h60,-0.006437,0.021126,0.027563,-0.010288,0.002763,0.013051,0.021126,0.002763,0.002763,0.130777,1.0,1.0,False,43020,35160,43020,35160
8,mom_5,specialized_h60,0.005914,-0.015701,-0.021615,0.008418,-0.001390,-0.009808,0.015701,0.001390,0.001390,0.088541,-1.0,-1.0,False,43020,35160,43020,35160


**1. Dominio claro de la volatilidad normalizada (ATR)**

- `atr_norm_14` y `atr_norm_20` son **features claramente estables** tanto para *return* como para *delta*.
- Presentan:
  - IC OOS **muy elevado** en comparación con *full day*.
  - Incremento fuerte IS → OOS.
  - **Mejora al pasar de H=60 a H=90** (`ratio_oos_90_over_60 ≈ 1.4`).

**Interpretación**
- En la ventana de gestación, el mercado **define su régimen de volatilidad del día**.
- La volatilidad temprana es altamente informativa para movimientos posteriores, especialmente en horizontes más largos.
- Estos features deben considerarse **núcleo obligatorio** en esta ventana.

---

**2. Diferencial de ROC: señal estructural contraria**

- `roc20_minus_roc60` aparece como *stable* en ambos targets.
- IC OOS **negativo y consistente** en H=60 y H=90.
- Magnitud relevante, especialmente en H=60.

**Interpretación**
- Señala **desaceleración temprana** respecto a la tendencia estructural.
- Funciona como señal *contrarian* útil para identificar:
  - agotamiento temprano,
  - falsas rupturas iniciales.
- Su estabilidad lo hace **válido en ambos horizontes**, con lectura negativa explícita.

---

**3. Indicadores de momentum y ROC: altamente dependientes del horizonte**

- `roc_20`, `roc_30`, `mom_10`, `mom_5`, `ema_60`:
  - Clasificados como *specialized_h60*.
  - IC OOS **negativo en H=60** y casi nulo en H=90.
- No generalizan al extender el horizonte.

**Interpretación**
- En gestación, estas señales capturan **micro-dinámica temprana**, válida solo para horizontes cortos.
- Al aumentar H, el ruido inicial se diluye.
- Deben usarse **exclusivamente en H=60**, con cautela.

---

**4. Interacciones momentum-diff: útiles solo en corto plazo**

- `mom5_minus_mom10`:
  - *specialized_h60* tanto en *return* como en *delta*.
  - IC OOS positivo en H=60, colapsa en H=90.

**Interpretación**
- Captura **aceleración temprana** dentro de la ventana de gestación.
- Es una señal de *timing*, no estructural.
- Válida solo para **H=60**.

---

**5. Interacciones ROC × ATR: señales de escala temporal larga**

- `roc60_x_atr14`, `roc60_x_atr20`:
  - *specialized_h90* en *return*.
  - *specialized_h60* en *delta*, pero con flip de signo.
- Comportamiento **dependiente del target y del horizonte**.

**Interpretación**
- Estas interacciones parecen capturar **traslado del régimen de volatilidad temprana hacia horizontes más largos**.
- No son universales.
- Deben usarse **solo de forma condicionada** y separadas por target/horizonte.

---

**Síntesis operativa — *Gestation Window***

**Features robustos (Return y Delta, H=60 y H=90):**
- `atr_norm_14`
- `atr_norm_20`
- `roc20_minus_roc60` (interpretación contraria)

**Features válidos solo en H=60:**
- `roc_20`
- `roc_30`
- `mom_10`
- `mom_5`
- `ema_60`
- `mom5_minus_mom10`

**Features condicionados / dependientes del target:**
- `roc60_x_atr14`
- `roc60_x_atr20`

---

**Conclusión**

La ventana de gestación es el punto donde:
- **se fija el régimen de volatilidad del día**, y
- **las señales de corto plazo aún no son estructurales**.

El feature set debe:
- priorizar **ATR normalizado** como señal dominante,
- usar momentum e interacciones **solo para H=60**,
- evitar forzar señales tempranas en horizontes largos.


#### **Observaciones - Execution (Return y Delta)**


In [249]:
print("\nstability_execution_ret\n")
display(stability_execution_ret)
print("\nstability_execution_delta\n")
display(stability_execution_delta)


stability_execution_ret



,feature,stability_label,ic_is_60,ic_oos_60,oos_minus_is_60,ic_is_90,ic_oos_90,oos_minus_is_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,roc60_x_atr14,stable,0.024521,0.054280,0.029759,0.023998,0.055085,0.031087,0.054280,0.055085,0.054280,1.014826,1.0,1.0,False,64530,52740,64530,52740
1,roc60_x_atr20,stable,0.023948,0.053754,0.029806,0.023562,0.054354,0.030793,0.053754,0.054354,0.053754,1.011164,1.0,1.0,False,64530,52740,64530,52740
2,roc_60,stable,0.030569,0.050866,0.020297,0.030158,0.051634,0.021477,0.050866,0.051634,0.050866,1.015112,1.0,1.0,False,64530,52740,64530,52740
3,ema_60,stable,0.033298,0.045886,0.012588,0.028175,0.044038,0.015863,0.045886,0.044038,0.044038,0.959725,1.0,1.0,False,64530,52740,64530,52740
4,atr_norm_20,stable,0.032417,0.036300,0.003884,0.032724,0.042803,0.010079,0.036300,0.042803,0.036300,1.179129,1.0,1.0,False,64530,52740,64530,52740
5,roc_30,stable,0.029996,0.035489,0.005493,0.025458,0.033146,0.007688,0.035489,0.033146,0.033146,0.933997,1.0,1.0,False,64530,52740,64530,52740
7,roc20_minus_roc60,stable,-0.031639,-0.033140,-0.001502,-0.033215,-0.037731,-0.004516,0.033140,0.037731,0.033140,1.138509,-1.0,-1.0,False,64530,52740,64530,52740
8,atr_norm_14,stable,0.033585,0.030017,-0.003568,0.034265,0.038212,0.003947,0.030017,0.038212,0.030017,1.273031,1.0,1.0,False,64530,52740,64530,52740
6,roc_20,stable,0.030478,0.033350,0.002872,0.020640,0.029707,0.009067,0.033350,0.029707,0.029707,0.890776,1.0,1.0,False,64530,52740,64530,52740
9,mom_10,stable,0.025455,0.024698,-0.000756,0.017055,0.021487,0.004431,0.024698,0.021487,0.021487,0.869970,1.0,1.0,False,64530,52740,64530,52740



stability_execution_delta



,feature,stability_label,ic_is_60,ic_oos_60,oos_minus_is_60,ic_is_90,ic_oos_90,oos_minus_is_90,abs_oos_60,abs_oos_90,abs_min_oos,ratio_oos_90_over_60,sign_oos_60,sign_oos_90,sign_flip_h60_h90,n_is_60,n_oos_60,n_is_90,n_oos_90
0,roc60_x_atr14,stable,0.023495,0.053378,0.029883,0.023102,0.055275,0.032173,0.053378,0.055275,0.053378,1.035538,1.0,1.0,False,64530,52740,64530,52740
1,roc60_x_atr20,stable,0.022974,0.052811,0.029837,0.022704,0.054537,0.031833,0.052811,0.054537,0.052811,1.032676,1.0,1.0,False,64530,52740,64530,52740
2,roc_60,stable,0.030004,0.049868,0.019864,0.029582,0.051848,0.022266,0.049868,0.051848,0.049868,1.039708,1.0,1.0,False,64530,52740,64530,52740
3,ema_60,stable,0.032532,0.046001,0.013468,0.027488,0.043914,0.016426,0.046001,0.043914,0.043914,0.954631,1.0,1.0,False,64530,52740,64530,52740
6,atr_norm_20,stable,0.027538,0.033410,0.005872,0.027403,0.037763,0.010360,0.033410,0.037763,0.033410,1.130282,1.0,1.0,False,64530,52740,64530,52740
4,roc_30,stable,0.029510,0.035812,0.006302,0.024959,0.032241,0.007282,0.035812,0.032241,0.032241,0.900290,1.0,1.0,False,64530,52740,64530,52740
7,roc20_minus_roc60,stable,-0.030831,-0.029470,0.001360,-0.032595,-0.036848,-0.004252,0.029470,0.036848,0.029470,1.250326,-1.0,-1.0,False,64530,52740,64530,52740
5,roc_20,stable,0.030299,0.034205,0.003906,0.020707,0.029379,0.008672,0.034205,0.029379,0.029379,0.858922,1.0,1.0,False,64530,52740,64530,52740
8,atr_norm_14,stable,0.028680,0.027736,-0.000944,0.029000,0.034012,0.005012,0.027736,0.034012,0.027736,1.226285,1.0,1.0,False,64530,52740,64530,52740
9,mom_10,stable,0.025066,0.025097,0.000031,0.017444,0.021245,0.003800,0.025097,0.021245,0.021245,0.846494,1.0,1.0,False,64530,52740,64530,52740



**1. Ventana de ejecución: máxima coherencia y estabilidad**

A diferencia de *full day* y *gestation*, la ventana de **ejecución** presenta un resultado muy claro:

- **Prácticamente todos los features evaluados son clasificados como `stable`**.
- No se observan:
  - flips de signo,
  - colapsos al pasar de H=60 a H=90,
  - ni degradación severa OOS.

Esto confirma que la ventana de ejecución corresponde a un **régimen de mercado ya definido**, donde las relaciones precio–feature son más limpias y persistentes.

---

**2. Interacciones ROC × ATR: señal dominante y transversal**

- `roc60_x_atr14`  
- `roc60_x_atr20`

Ambas interacciones:
- Presentan **los IC OOS más altos de toda la tabla**.
- Son estables para:
  - *return* y *delta*,
  - H = 60 y H = 90.
- Ratio H90/H60 ≈ 1 → **escala muy bien con el horizonte**.

**Interpretación**
- En ejecución, la señal relevante no es solo dirección (`roc_60`), sino:
  - **dirección × régimen de volatilidad**.
- Estas interacciones capturan la *intensidad operativa efectiva* del movimiento.
- Son **features clave** para esta ventana.

---

**3. Indicadores base: plenamente estructurales en ejecución**

Los siguientes indicadores muestran:
- IC OOS alto,
- signo consistente,
- estabilidad perfecta entre horizontes y targets.

**Features base estables**
- `roc_60`
- `ema_60`
- `roc_20`, `roc_30`
- `mom_10`, `mom_5`
- `atr_norm_14`, `atr_norm_20`

**Interpretación**
- Durante la ejecución, el mercado ya está “en movimiento”.
- Las señales de:
  - tendencia,
  - momentum,
  - y volatilidad  
  pasan a ser **directamente predictivas**, sin depender de combinaciones complejas.

---

**4. Diferencial de ROC: señal contraria controlada**

- `roc20_minus_roc60`
  - IC OOS negativo, estable y consistente.
  - Magnitud moderada pero robusta.

**Interpretación**
- Identifica **desaceleración o agotamiento** durante la ejecución.
- Funciona como señal *contrarian* complementaria.
- Útil para:
  - gestión de salida,
  - reducción de exposición,
  - timing fino.

---

**5. Momentum-diff: estable pero secundario**

- `mom5_minus_mom10`
  - Estable en return y delta.
  - IC OOS menor que interacciones ROC × ATR.

**Interpretación**
- Aporta información marginal de aceleración/desaceleración.
- Puede mantenerse, pero **no es feature prioritario** en esta ventana.

---

**Síntesis operativa — *Execution Window***

  - **Features prioritarios (Return y Delta, H=60 y H=90):**
    - `roc60_x_atr14`
    - `roc60_x_atr20`
    - `roc_60`
    - `ema_60`
    - `atr_norm_14`
    - `atr_norm_20`

  - **Features complementarios (estables):**
    - `roc_20`
    - `roc_30`
    - `mom_10`
    - `mom_5`
    - `roc20_minus_roc60`
    - `mom5_minus_mom10`

---

**Conclusión**

La ventana de ejecución es donde:

- las **interacciones complejas sí valen la pena**,
- el mercado ya reveló su régimen,
- y la señal es **más fuerte, estable y escalable en horizonte**.

En este contexto:
- no hay conflicto entre indicadores base e interacciones,
- ambos se refuerzan,
- y el riesgo de sobreajuste es mínimo.

Es la ventana más “model-friendly” de todo el pipeline.



# **9. Consolidación de features**

**Target: **DELTA / RETURN****
> Nota: cuando no se indica diferencia explícita, la decisión aplica **tanto a return como a delta**.

| Ventana    | Horizonte | atr_norm_14 | atr_norm_20 | ema_60 | mom_10 | mom_5 | roc_20 | roc_30 | roc_60 | roc60_x_atr20 | roc60_x_atr14 | roc20_minus_roc60 | mom5_minus_mom10 |
|------------|-----------|-------------|-------------|--------|--------|-------|--------|--------|--------|---------------|---------------|-------------------|------------------|
| **Full Day** | H=60      |      x      |      x      |   x    |  (x)   |       |   x    |   x    |   x    |       x       |       x       |         x         |                  |
|            | H=90      |      x      |      x      |   x    |  (x)   |       |   x    |   x    |   x    |       x       |       x       |         x         |                  |
| **Gestation** | H=60      |      x      |      x      |   x    |   x    |   x   |   x    |   x    |        |               |               |         x         |        x         |
|            | H=90      |      x      |      x      |        |        |       |        |        |        |       (x)\*    |       (x)\*    |         x         |                  |
| **Execution** | H=60      |      x      |      x      |   x    |   x    |   x   |   x    |   x    |   x    |       x       |       x       |         x         |        x         |
|            | H=90      |      x      |      x      |   x    |   x    |   x   |   x    |   x    |   x    |       x       |       x       |         x         |        x         |

---

**Leyenda e interpretación**

- **x** → feature recomendado y consistente OOS.  
- **(x)** → feature secundario / opcional (baja magnitud o aporte marginal).  
- **vacío** → feature excluido para esa ventana–horizonte.  
- **(x)\*** → feature condicionado al target o con uso cauteloso (principalmente en *gestation H=90*).

---

**Lectura rápida por ventana**

- **Full Day**
  - Predominan señales **estructurales**.
  - Se mantienen ATR, EMA, ROC y sus interacciones con ATR.
  - Momentum corto y diferencias de momentum **no aportan estabilidad**.

- **Gestation**
  - Ventana **más frágil y dependiente del horizonte**.
  - En H=60 se permite mayor riqueza de features.
  - En H=90 solo sobreviven señales de **volatilidad** y **estructura relativa**.

- **Execution**
  - Ventana **más robusta del pipeline**.
  - Se justifica el uso conjunto de:
    - indicadores base,
    - interacciones ROC × ATR,
    - y diferencias de momentum.
    - Máxima coherencia entre return y delta, y entre horizontes.

---

**Conclusión final**

Esta tabla resume un **feature set condicional**, coherente con:

- estabilidad IS vs OOS,
- dependencia temporal del mercado,
- y separación clara entre:
  - señal estructural,
  - señal de aceleración,
  - y refinamiento operativo.

El resultado no es un único set global, sino un **mapa de uso racional de features por ventana y horizonte**, alineado con el comportamiento empírico observado.


## **9.1. Justificación del set final de features**


A partir del análisis estadístico, estructural y de estabilidad realizado a lo largo de este stage, se define un conjunto de features que equilibra **capacidad predictiva, estabilidad out-of-sample e interpretabilidad**, evitando redundancia y sobreajuste.

**1. Variable OHLC seleccionada**

  - **`close`**

    - Las variables OHLC presentan **correlación extremadamente alta entre sí** (> 0.99), por lo que aportan información prácticamente redundante.
    - `close` concentra la información relevante del precio:
      - Es el valor más utilizado por los indicadores técnicos.
      - Resume el consenso del mercado al cierre de cada minuto.
    - Su IC es **estable entre horizontes (H=60 y H=90)** y consistente entre IS y OOS.

  - **Decisión**: se utiliza únicamente `close` como variable de precio base.

**2. Indicadores técnicos seleccionados**

Los indicadores fueron elegidos por cumplir simultáneamente:

  - IC out-of-sample distinto de cero,
  - coherencia direccional IS vs OOS,
  - estabilidad entre horizontes o especialización explicable,
  - alineación con la lógica de mercado (tendencia + momentum).

  Los indicadores técnicos seleccionados son:

  - `ema_60`

    - Captura la **tendencia intradía dominante**.
    - Es estable en full day y relevante en ventanas operativas.
    - Funciona como **filtro estructural**, no como señal puntual.

  - `roc_20`, `roc_30`, `roc_60`

    - Representan **velocidad y magnitud del movimiento del precio** en distintas escalas.
    - `roc_60`: estructura de medio plazo (usable todo el día).
    - `roc_20` y `roc_30`: dinámicas más cortas, relevantes en ventanas específicas.
    - Muestran **buena generalización OOS** y coherencia entre horizontes.

  - `momentum_3`, `momentum_5`, `momentum_10`

    - Capturan impulsos de **muy corto plazo**, especialmente útiles en ventanas de gestación y ejecución.
    - Individualmente pueden ser ruidosos, pero:
      - mantienen señal consistente en contextos específicos,
      - son la base para interacciones más informativas.

**3. Interacciones seleccionadas**

Las interacciones no se introducen para aumentar complejidad, sino para **capturar relaciones relativas entre escalas temporales**, que mostraron mayor estabilidad que algunos indicadores crudos.

  - `mom3_minus_mom10`

    - Mide aceleración corta vs impulso base.
    - Detecta cambios tempranos de régimen.
    - Mostró buena estabilidad en execution.

  - `mom5_minus_mom10`

    - Captura divergencias entre momentum corto y medio.
    - Útil para distinguir continuidad vs agotamiento.

  - `mom3_minus_mom5`

    - Refina el análisis del impulso inmediato.
    - Aporta señal complementaria en ventanas operativas.

Las interacciones **no reemplazan** a los indicadores base, sino que los **contextualizan**.

**4. Cómo se utilizarán en el modelo**

- **Indicadores base (`ema`, `roc`, `momentum`)**:
  - Proveen información primaria de tendencia y velocidad.
  - Se utilizarán como features directos (eventualmente normalizados intradía).

- **Interacciones de momentum**:
  - Actúan como features de **contexto relativo**.
  - Ayudan al modelo a distinguir:
    - impulso genuino vs ruido,
    - aceleración vs desaceleración.

- **Separación por ventanas (full day / gestation / execution)**:
  - Los features se calculan para todo el día,
  - pero su **evaluación y peso** se interpretan según la ventana donde demostraron mayor valor.

## **9.2. Decisión final y set consolidado de features**


El set final de features:

- es **compacto** (sin redundancias),
- **estable out-of-sample**,
- **interpretable desde la lógica de mercado**,
- y está preparado para **modelos que generalicen**, no para optimizar IC in-sample.

Este conjunto constituye una base sólida para el entrenamiento de modelos intradía con control explícito de generalización y sin leakage.

| Tipo de feature                | Variable(s)                                      | Rol económico principal                                                                 |
|--------------------------------|--------------------------------------------------|------------------------------------------------------------------------------------------|
| Precio intradía                | `close`                                         | Estado representativo del precio; consenso del mercado en cada minuto                    |
| Tendencia intradía             | `ema_60`                                        | Dirección y nivel tendencial dominante del día; filtro estructural del mercado           |
| Momentum de muy corto plazo    | `momentum_3`, `momentum_5`                      | Impulso inmediato; captura aceleraciones y micro-movimientos del precio                 |
| Momentum de corto plazo        | `momentum_10`                                   | Intensidad del desplazamiento reciente; referencia base de impulso                       |
| Velocidad de corto plazo       | `roc_20`, `roc_30`                              | Rapidez y magnitud del movimiento en escalas cortas; útil en ventanas operativas         |
| Velocidad de medio plazo       | `roc_60`                                       | Dinámica estructural del movimiento; estabilidad direccional a lo largo del día          |
| Interacción de momentum        | `mom3_minus_mom10`                              | Aceleración relativa: impulso corto frente a impulso base                                |
| Interacción de momentum        | `mom5_minus_mom10`                              | Divergencia entre momentum corto y medio; detección de agotamiento o continuidad         |
| Interacción de momentum        | `mom3_minus_mom5`                               | Refinamiento del impulso inmediato; cambios tempranos en la dinámica intradía            |


## **9.3. Implicancia para el pipeline**


Este set consolidado constituye el núcleo de features técnicas del modelo y será utilizado en las etapas posteriores para:

- entrenamiento y validación del modelo predictivo,
- análisis de contribución por feature,
- y evaluación de desempeño económico.

La consolidación reduce la dimensionalidad, mejora la interpretabilidad y alinea el diseño del modelo con la estructura temporal y económica observada en los datos.

## **9.4. Implementación**


In [246]:
mnq_intraday_labeled = load_data()

# Asumimos que el índice es DatetimeIndex tz-aware
idx = mnq_intraday_labeled.index

mnq_intraday_labeled["minute_of_day"] = idx.hour * 60 + idx.minute

NameError: name 'load_data' is not defined

In [ ]:
def info_dataset_final(df):
    print("Información del dataset:\n")

    # -------------------------
    # Días y registros
    # -------------------------
    num_dias = df["date"].nunique()
    print(f"\tCantidad de días: {num_dias}")

    validos_por_dia = (
        df.dropna(subset=["close"])
          .groupby("date")
          .size()
    )
    promedio_por_fecha = validos_por_dia.mean()
    print(f"\tRegistros por día: {int(promedio_por_fecha)}")

    # -------------------------
    # Horarios (desde minute_of_day)
    # -------------------------
    if "minute_of_day" in df.columns:
        min_minute = int(df["minute_of_day"].min())
        max_minute = int(df["minute_of_day"].max())

        primer_hora = f"{min_minute // 60:02d}:{min_minute % 60:02d}"
        ultima_hora = f"{max_minute // 60:02d}:{max_minute % 60:02d}"
    else:
        primer_hora = None
        ultima_hora = None

    print(f"\tHora diaria de inicio: {primer_hora}")
    print(f"\tHora diaria de final: {ultima_hora}")

    # -------------------------
    # Columnas del dataset
    # -------------------------
    print("\n\tColumnas del dataset:")
    for c in df.columns:
        print(f"\t- {c}")



In [ ]:
info_dataset_final(mnq_intraday_labeled)

In [ ]:
import pandas as pd
from typing import List, Tuple
from ta.momentum import ROCIndicator


def build_mnq_features_targets_full(
    mnq_intraday_labeled: pd.DataFrame,
    *,
    date_col: str = "date",
    minute_col: str = "minute_of_day",
    close_col: str = "close",
    target_cols: Tuple[str, str] = ("ret_60", "ret_90"),
    tz: str | None = "America/New_York",   # se deja, pero NO se usa para el índice
    keep_datetime_col: bool = True,
) -> Tuple[pd.DataFrame, List[str], List[str]]:

    # ------------------------------------------------------------
    # Preservar el índice original (NO tocarlo)
    # ------------------------------------------------------------
    original_index = mnq_intraday_labeled.index

    # ------------------------------------------------------------
    # 0) Copiar dataset
    # ------------------------------------------------------------
    df = mnq_intraday_labeled.copy()

    # -------------------------
    # 0) Validación columnas mínimas
    # -------------------------
    required_base = [date_col, minute_col, close_col, *list(target_cols)]
    missing = [c for c in required_base if c not in df.columns]
    if missing:
        raise ValueError(f"Faltan columnas requeridas en mnq_intraday_labeled: {missing}")

    # ------------------------------------------------------------
    # 1) Normalización de nombres (si vinieran con delta_pts_*)
    # ------------------------------------------------------------
    #df = df.rename(columns={"ret_60": "ret60", "ret_90": "ret90"})

    # ------------------------------------------------------------
    # 2) Indicadores técnicos (por día)
    # ------------------------------------------------------------
    technical_indicators_features: List[str] = [
        "ema60",
        "mom3",
        "mom5",
        "mom10",
        "roc20",
        "roc30",
        "roc60",
    ]

    def _apply_per_day(g: pd.DataFrame) -> pd.DataFrame:
        g = g.copy()

        g["ema60"] = g[close_col] / g[close_col].ewm(span=60, adjust=False).mean() - 1.0

        g["mom3"] = g[close_col].pct_change(3)
        g["mom5"] = g[close_col].pct_change(5)
        g["mom10"] = g[close_col].pct_change(10)

        g["roc20"] = ROCIndicator(close=g[close_col], window=20).roc()
        g["roc30"] = ROCIndicator(close=g[close_col], window=30).roc()
        g["roc60"] = ROCIndicator(close=g[close_col], window=60).roc()

        return g

    # Groupby por date (columna), no por índice
    df = df.groupby(df[date_col], group_keys=False, sort=False).apply(_apply_per_day)

    # ✅ Garantía dura: restaurar el índice EXACTO del input
    # (si por cualquier motivo apply lo alteró)
    if not df.index.equals(original_index):
        df = df.copy()
        df.index = original_index

    # ------------------------------------------------------------
    # 3) Interacciones mínimas
    # ------------------------------------------------------------
    def calculate_interactions_features(
        df_in: pd.DataFrame,
        *,
        date_col: str = "date",
        minute_col: str = "minute_of_day",
        mom3_col: str = "mom3",
        mom5_col: str = "mom5",
        mom10_col: str = "mom10",
        prefix: str = "",
    ) -> Tuple[pd.DataFrame, List[str]]:

        df_out = df_in.copy()

        def _col(name: str) -> str:
            return f"{prefix}{name}" if prefix else name

        needed = [date_col, minute_col, mom3_col, mom5_col, mom10_col]
        miss = [c for c in needed if c not in df_out.columns]
        if miss:
            raise KeyError(f"Faltan columnas requeridas para interacciones: {miss}")

        created_cols: List[str] = []

        c = _col("mom3_mom10")
        df_out[c] = df_out[mom3_col] - df_out[mom10_col]
        created_cols.append(c)

        c = _col("mom5_mom10")
        df_out[c] = df_out[mom5_col] - df_out[mom10_col]
        created_cols.append(c)

        c = _col("mom3_mom5")
        df_out[c] = df_out[mom3_col] - df_out[mom5_col]
        created_cols.append(c)

        return df_out, created_cols

    df, interactions_features = calculate_interactions_features(
        df,
        date_col=date_col,
        minute_col=minute_col,
        mom3_col="mom3",
        mom5_col="mom5",
        mom10_col="mom10",
    )


    # ------------------------------------------------------------
    # 4) Dataset final (selección de columnas)
    # ------------------------------------------------------------
    final_cols: List[str] = [
        date_col,
        minute_col,
        close_col,
        *technical_indicators_features,
        *interactions_features,
        *list(target_cols),
    ]

    # Mantengo esto solo si la columna "datetime" EXISTE; si no, no la invento
    if keep_datetime_col and "datetime" in df.columns:
        final_cols = ["datetime"] + final_cols

    missing_final = [c for c in final_cols if c not in df.columns]
    if missing_final:
        raise ValueError(f"No se pudieron construir todas las columnas finales: {missing_final}")

    mnq_features_targets = df.loc[:, final_cols].copy()

    # Garantía final: el índice sale idéntico al que entró
    if not mnq_features_targets.index.equals(original_index):
        mnq_features_targets.index = original_index

    # ------------------------------------------------------------
    # 5) Renombrar targets al final: ret_* -> ret*
    # ------------------------------------------------------------
    rename_targets = {"ret_60": "ret60", "ret_90": "ret90"}
    mnq_features_targets = mnq_features_targets.rename(columns=rename_targets)

    targets = ['ret60', 'ret90']
    # Si desea que target_cols también salga coherente (opcional)
    # target_cols_out = tuple(rename_targets.get(c, c) for c in target_cols)

    return mnq_features_targets, technical_indicators_features, interactions_features, targets


In [ ]:
mnq_features_targets, tech_feats, inter_feats, targets = build_mnq_features_targets_full(
    mnq_intraday_labeled=mnq_intraday_labeled
)

In [ ]:
#mnq_features_targets.head(15)

In [ ]:
print(f"OHLVC feats: ['close'] \n")
print(f"Tech feats: {tech_feats} \n")
print(f"Interaction feats: {inter_feats} \n")
print(f"Targets: {targets}")

In [ ]:
info_dataset_final(mnq_features_targets)

# **10. Flags de activación por ventanas operativas**

## **10.1. Introducción conceptual**


**1. Motivación principal: valor predictivo dependiente del horario**

El dataset incluye features cuyo valor predictivo no es homogéneo a lo largo de la jornada.

En particular:

- Algunos indicadores (por ejemplo, ciertos momentum y ROC) muestran mayor capacidad predictiva durante las ventanas de:

  - gestation (08:00-09:00)
  - execution (09:00-10:00)

- Fuera de esas ventanas, esas mismas features:
  - pierden señal,
  - se vuelven ruidosas,
  - o directamente dejan de ser informativas para la toma de decisiones.

Sin embargo, el modelo se entrena usando ventanas deslizantes a lo largo de todo el full day.
Por lo tanto, es necesario informarle explícitamente al modelo en qué contextos temporales una feature es relevante.

**2. Por qué se utilizan flags y no filtros duros**

En lugar de:

- eliminar features fuera de ciertas horas, o
- entrenar modelos distintos por franja horaria,

se introducen flags de activación para permitir que el modelo:

- vea toda la jornada (maximizando datos),
- pero aprenda cuándo una feature es confiable.

Cada flag responde a la pregunta:

>“¿Esta feature tiene sentido predictivo en este momento del día?”

**3. Qué aprende el modelo con este esquema**

Cada muestra contiene pares del tipo:

$$  
𝑋_t = [roc20, roc20_{active}, mom3, mom3_{active}, ... ]
$$

Durante el entrenamiento, el modelo aprende patrones como:

- Cuando `roc20_active` = 1

  → `roc20` suele correlacionar mejor con el target.

- Cuando `roc20_active` = 0

  → `roc20` pierde poder explicativo y debe ser atenuado o ignorado.

De este modo, el modelo modula dinámicamente la importancia de cada feature en función del contexto horario, sin reglas hard-coded.

**4. Por qué no es suficiente “poner ceros”**

Asignar 0 a una feature fuera de su ventana introduce ambigüedad:

- `roc20` = 0 puede significar:

  - un valor legítimo del indicador, o
  - una feature fuera de su régimen predictivo.

Con flags explícitos:
  - `roc20` = 0
  - `roc20_active` = 0

el modelo puede distinguir claramente entre:
  - “valor bajo pero válido”
  - “valor no relevante en este horario”

**5. Interpretación conceptual**

El flag actúa como un contextualizador temporal, no como una regla.

- Feature → señal numérica
- Flag → indica si la señal está en su régimen de mayor valor predictivo

El modelo aprende implícitamente:

> “Esta feature solo merece atención cuando estoy dentro de la ventana operativa adecuada.”

**6. Condiciones necesarias para que el enfoque sea válido**

1. Cada feature dependiente del horario tiene su flag correspondiente.
2. Los flags se mantienen como variables binarias (0/1) y no se escalan.
3. El orden de columnas es estrictamente consistente en todo el pipeline.

**Conclusión**

El uso de flags permite entrenar un modelo con cobertura completa del día, sin perder la capacidad de:

- capturar regímenes horarios de mayor valor predictivo,
- diferenciar señal estructural de ruido intradía,
- y alinear el aprendizaje con las ventanas reales de operación y ejecución.

Este diseño es coherente con un enfoque full-day learning y window-aware decision making.

## **10.2. Implementación**


In [ ]:
execution_features = ['mom3', 'mom5', 'roc20', 'mom3_mom5']
gestation_features = ['mom10','roc30','mom3_mom10','mom5_mom10'] + execution_features

In [ ]:
#print(f'gestation_window: {start_time_gestation_window}-{final_time_gestation_window}')
#print(f'gestation_features: {gestation_features}')

#print(f'execution_window: {start_time_execution_window}-{final_time_execution_window}')
#print(f'execution_features: {execution_features}' )

In [ ]:
windows = {
    "gestation": (start_time_gestation_window, final_time_gestation_window),
    "execution": (start_time_execution_window, final_time_execution_window)
}

window_features = {
    "gestation": gestation_features,
    "execution": execution_features,
}

In [ ]:
import pandas as pd
from typing import Iterable, Dict, Tuple

def add_time_window_feature_flags(
    df: pd.DataFrame,
    *,
    windows: Dict[str, Tuple[str, str]],
    window_features: Dict[str, Iterable[str]],
    flag_suffix: str = "_active",
) -> pd.DataFrame:
    """
    Agrega flags binarios por feature indicando si el timestamp (índice datetime)
    cae dentro de la ventana temporal asociada a esa feature.

    - NO modifica el índice.
    - NO altera valores de las features.
    - Devuelve una copia del DataFrame con nuevas columnas *_active.

    Parámetros
    ----------
    df : pd.DataFrame
        Dataset con índice DatetimeIndex.
    windows : dict
        Mapeo nombre_ventana -> (start_time, end_time) en formato 'HH:MM'.
        Ej: {'gestation': ('08:00','09:00'), 'execution': ('09:00','10:00')}
    window_features : dict
        Mapeo nombre_ventana -> iterable de features que aplican a esa ventana.
    flag_suffix : str
        Sufijo para las columnas flag. Default '_active'.

    Retorna
    -------
    pd.DataFrame
        Copia del df con flags agregados.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise TypeError("El DataFrame debe tener un índice DatetimeIndex.")

    out = df.copy()

    # Hora del índice sin tocar timezone
    idx_time = out.index.strftime("%H:%M")

    for window_name, (start, end) in windows.items():
        if window_name not in window_features:
            continue

        is_in_window = (idx_time >= start) & (idx_time < end)

        for feat in window_features[window_name]:
            if feat not in out.columns:
                raise ValueError(f"La feature '{feat}' no existe en el DataFrame.")

            flag_col = f"{feat}{flag_suffix}"
            # Si la feature aparece en múltiples ventanas, OR lógico
            if flag_col in out.columns:
                out[flag_col] = out[flag_col] | is_in_window.astype("int8")
            else:
                out[flag_col] = is_in_window.astype("int8")

    return out



In [ ]:
mnq_features_targets = add_time_window_feature_flags(
    mnq_features_targets,
    windows=windows,
    window_features=window_features,
)

In [ ]:
info_dataset_final(mnq_features_targets)

In [ ]:
mnq_features_targets.columns

In [ ]:
base_order = [
    'date',
    'minute_of_day',
]

features_order = [
    'close',
    'ema60',
    'roc60',

    'roc30', 'roc30_active',
    'roc20', 'roc20_active',

    'mom10', 'mom10_active',
    'mom5',  'mom5_active',
    'mom3',  'mom3_active',

    'mom5_mom10', 'mom5_mom10_active',
    'mom3_mom10', 'mom3_mom10_active',
    'mom3_mom5',  'mom3_mom5_active',
]

target_order = [
    'ret60',
    'ret90',
]

In [ ]:
import pandas as pd
from typing import List

def reorder_mnq_columns(
    df: pd.DataFrame,
    *,
    base_order: List[str],
    features_order: List[str],
    target_order: List[str],
) -> pd.DataFrame:
    """
    Reordena las columnas del dataset mnq_features_targets según un orden explícito.

    - NO modifica el índice.
    - NO altera valores.
    - Valida que todas las columnas requeridas existan.
    - Devuelve una copia del DataFrame con el nuevo orden.

    Orden final:
        base_order + features_order + target_order
    """
    required_cols = base_order + features_order + target_order

    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"Faltan columnas requeridas en el DataFrame: {missing}"
        )

    return df[required_cols].copy()


In [ ]:
mnq_features_targets = reorder_mnq_columns(
    mnq_features_targets,
    base_order=[
        'date',
        'minute_of_day',
    ],
    features_order=[
        'close',
        'ema60',
        'roc60',

        'roc30', 'roc30_active',
        'roc20', 'roc20_active',

        'mom10', 'mom10_active',
        'mom5',  'mom5_active',
        'mom3',  'mom3_active',

        'mom5_mom10', 'mom5_mom10_active',
        'mom3_mom10', 'mom3_mom10_active',
        'mom3_mom5',  'mom3_mom5_active',
    ],
    target_order=[
        'ret60',
        'ret90',
    ],
)


In [ ]:
mnq_features_targets

In [ ]:
OUT_PARQUET

In [ ]:
OUT_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Guardar dataset
mnq_features_targets.to_parquet(OUT_PARQUET, index=True)